### AMML ranking competition 
## Система: Linux-6.6.159+-x86_64-with-glibc2.39 | Python: 3.13.16 | GPU: Tesla T4

это было тяжко

In [ ]:
#!pip uninstall -y torchvision
# без него были кое какие проблемы

In [ ]:
from pathlib import Path
PROJECT_DIR=Path(globals().get('PROJECT_DIR',Path.cwd())).resolve()
INPUT_PATH=PROJECT_DIR/'amml-ranking-competition-2026.zip'  # либо каталог с тремя исходными CSV
WORK_DIR=PROJECT_DIR/'amml_v19_final_work'  # выбери постоянный диск для длительного запуска
OUTPUT_PATH=PROJECT_DIR/'submission.csv'
AUTO_INSTALL=True


##### тут может потребовать перезапустить сеанс и выполнить код заново

In [ ]:
# Версии пакетов в requirements.txt и встроенном списке должны совпадать;
import os,sys,subprocess,importlib.metadata as im
REQUIREMENTS_TEXT='# Python 3.12; Linux x86_64 with an NVIDIA CUDA-capable GPU is recommended.\n# The public torch wheel installs its pinned CUDA runtime dependencies on Linux.\ntorch==2.8.0\nnumpy==2.3.5\npandas==2.2.3\nscipy==1.17.0\nscikit-learn==1.8.0\nlightgbm==4.6.0\ntransformers==4.55.4\naccelerate==1.10.1\nhuggingface-hub==0.36.2\ntokenizers==0.21.4\nsafetensors==0.8.0\nsentencepiece==0.2.1\nbitsandbytes==0.48.2\njoblib==1.5.3\nthreadpoolctl==3.6.0\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.30.1\njupyter-client==8.6.3\ncomm==0.2.3\ndebugpy==1.8.16\nnest-asyncio==1.6.0\npyzmq==27.0.2\ntornado==6.5.2\n'
req=PROJECT_DIR/'requirements.txt'
def requirement_pins(text):
    lines=(line.split('#',1)[0].strip() for line in text.splitlines())
    return sorted(line for line in lines if line)
if req.is_file() and requirement_pins(req.read_text())!=requirement_pins(REQUIREMENTS_TEXT):
    raise ValueError('Используй requirements.txt из той же финальной сборки')
WORK_DIR.mkdir(parents=True,exist_ok=True)
bootstrap=WORK_DIR/'requirements.runtime.txt';bootstrap.write_text(REQUIREMENTS_TEXT)
mods={'scikit-learn':'sklearn','huggingface-hub':'huggingface_hub'}
# Text ranking does not use these optional binary PyTorch extensions. Colab
# can leave builds for a newer torch installed after torch is pinned below.
optional=[]
for package in ('torchvision','torchaudio'):
    try:im.version(package);optional.append(package)
    except im.PackageNotFoundError:pass
restart=[]
if optional:
    if not AUTO_INSTALL:
        raise RuntimeError('Удали ненужные для этого текстового решения пакеты: '+', '.join(optional))
    for name in ('torchvision','torchaudio','transformers'):
        if any(k==name or k.startswith(name+'.') for k in sys.modules):restart.append(name)
    subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',*optional])
    import importlib
    importlib.invalidate_caches()
# Transformers caches optional-package availability on its first import.
availability=sys.modules.get('transformers.utils.import_utils')
if availability is not None:
    if (getattr(availability,'_torchvision_available',False) and 'torchvision' not in optional
            or getattr(availability,'_torchaudio_available',False) and 'torchaudio' not in optional):
        restart.append('transformers: устаревший кэш доступности torchvision/torchaudio')
missing=[];loaded=[]
for line in REQUIREMENTS_TEXT.splitlines():
    if not line or line.startswith('#'):continue
    package,expected=line.split('==')
    try:actual=im.version(package).split('+')[0]
    except im.PackageNotFoundError:actual=None
    if actual!=expected:
        missing.append(package)
        if mods.get(package,package.replace('-','_')) in sys.modules:loaded.append(package)
if missing:
    if not AUTO_INSTALL:raise RuntimeError('Сначала установи requirements.txt: '+', '.join(missing))
    subprocess.check_call([sys.executable,'-m','pip','install','-r',str(bootstrap)])
    restart.extend(loaded)
if restart:
    raise RuntimeError('Окружение исправлено. Перезапусти сеанс/ядро один раз и снова выполни все ячейки с тем же WORK_DIR: '+', '.join(sorted(set(restart))))
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['USE_TF']='0';os.environ['USE_FLAX']='0'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ.setdefault('HF_HOME',str(WORK_DIR/'huggingface'))
os.environ.setdefault('MPLCONFIGDIR',str(WORK_DIR/'mpl'))
import json,platform,hashlib,shutil,time
import numpy as np,pandas as pd,torch
if not torch.cuda.is_available():raise RuntimeError('Для полного обучения и 8B-инференса нужна NVIDIA CUDA GPU')
if torch.cuda.get_device_properties(0).total_memory<14*2**30:raise RuntimeError('Нужно не менее 16 ГБ VRAM')
# Import/forward check before downloading any neural weights or training BGE.
# The CPU RNG is restored, so this check does not change the model recipe.
try:
    from transformers import AutoModelForSequenceClassification,XLMRobertaConfig
    with torch.random.fork_rng(devices=[]),torch.device('cpu'):
        tiny_config=XLMRobertaConfig(vocab_size=32,hidden_size=16,num_hidden_layers=1,
            num_attention_heads=2,intermediate_size=32,max_position_embeddings=16,num_labels=1)
        tiny_model=AutoModelForSequenceClassification.from_config(tiny_config).eval()
        with torch.inference_mode():
            tiny_output=tiny_model(input_ids=torch.tensor([[0,4,2]],device='cpu'),
                attention_mask=torch.ones((1,3),dtype=torch.long,device='cpu')).logits
        assert tiny_output.shape==(1,1) and torch.isfinite(tiny_output).all()
        del tiny_model,tiny_output,tiny_config
except Exception as exc:
    raise RuntimeError('Предварительная проверка текстовой модели Transformers не прошла; исходная ошибка приведена ниже.') from exc
print('Проверка импорта и forward текстовой модели Transformers: OK')
RUNTIME_DIR=WORK_DIR/'runtime';RUNTIME_DIR.mkdir(exist_ok=True)
sys.path.insert(0,str(RUNTIME_DIR));os.chdir(RUNTIME_DIR)
print('Система:',platform.platform(),'| Python:',platform.python_version(),'| GPU:',torch.cuda.get_device_name(0))


In [ ]:
%%writefile cache_core.py
"""Data, exact ranking metric, and verified score-cache utilities.

Legacy CE scores are out-of-fold for train and a mean of five models for test.
They may be useful stack features, but ordinary second-level CV with these
features is an approximate evaluation rather than nested end-to-end CV.
Only numpy, pandas and scikit-learn are required.
"""
from __future__ import annotations

import hashlib
import html
import json
import os
from pathlib import Path
import re
import tempfile
import unicodedata
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold


PREPROCESS_VERSION = "observed-canonical-v2"
LEGACY_EXPECTED = {"bge": 0.9326, "bge2": 0.9336, "bge3": 0.9353}
LEGACY_FOLD_LENGTHS = (6307, 6306, 6305, 6306, 6306)
_SIGNED_COLUMNS = ("id", "query_id", "document_id", "article_id", "query", "title", "text",
                   "passage_index", "article_passage_count", "article_char_count", "ingest_source",
                   "relevance", "query_c", "title_c", "text_c", "was_moji", "has_stress", "q_stress")
_MOJI = re.compile(r"[РС][\u0080-\u00bf\u0400-\u04ff]")
_TAGS = re.compile(r"</?[A-Za-z][^>]*>")
_NAV = re.compile(r"Навигация:\s*Главная\s*\|\s*Статьи", re.I)


def fix_mojibake(value):
    """Repair the observed UTF-8-as-CP1251 corruption, otherwise retain text."""
    if not isinstance(value, str):
        return ""
    if len(_MOJI.findall(value)) <= 5 or re.search(r"[а-яё]{4}", value):
        return value
    for errors in ("strict", "ignore"):
        try:
            candidate = value.encode("cp1251", errors=errors).decode("utf-8", errors=errors)
            if candidate and re.search(r"[а-яё]{4}", candidate):
                return candidate
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return value


def clean_text(value):
    """Canonical Russian text: encoding/HTML/nav/Unicode repair; preserve case."""
    value = fix_mojibake(value)
    value = html.unescape(value)
    value = _TAGS.sub(" ", value)
    value = _NAV.sub(" ", value)
    value = re.sub(r"[\u0300\u0301\u00ad\u200b-\u200f\u2060\ufeff]", "", value)
    value = unicodedata.normalize("NFKC", value)
    value = re.sub(r"\(\s*[,;]?\s*\)", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def _best_observed(keys, values):
    # Prefer the most complete cleaned observation; deterministic lexical ties.
    candidates = pd.DataFrame({"key": np.asarray(keys), "value": np.asarray(values)})
    candidates = candidates.drop_duplicates()
    candidates["quality"] = candidates.value.str.len() - 8 * candidates.value.str.count("�")
    return (candidates.sort_values(["quality", "value"], ascending=[False, True], kind="stable")
            .drop_duplicates("key").set_index("key").value)


def restore_documents(frame):
    """Add query_c/title_c/text_c using only complete observed text variants.

    Raw columns and row order remain intact. Text is selected only within one
    document_id; titles only within article_id; queries only within query_id.
    No labels, external corpus, or invented article text are used. Calling this
    on train+test together makes restoration consistent for the full dataset.
    """
    result = frame.copy()
    for original, cleaned in (("query", "query_c"), ("title", "title_c"), ("text", "text_c")):
        result[cleaned] = result[original].map(clean_text)
    # Remove source wrappers/placeholders before selecting an observed title.
    result['title_c'] = result['title_c'].str.replace(r'\s*[-—–|]\s*Википедия\s*$', '', regex=True).str.strip()
    result['title_c'] = result['title_c'].mask(result['title_c'].str.lower().isin(
        ['', 'не указано', 'неизвестно', 'без названия', 'none', 'nan', 'null', 'n/a']), '')
    result["was_moji"] = [int(isinstance(x, str) and fix_mojibake(x) != x) for x in result.text]
    result["has_stress"] = result.text.fillna("").str.contains("\u0301", regex=False).astype("int8")
    result["q_stress"] = result["query"].fillna("").str.contains("\u0301", regex=False).astype("int8")
    for key, column in (("query_id", "query_c"), ("article_id", "title_c"), ("document_id", "text_c")):
        best = _best_observed(result[key], result[column])
        result[column] = result[key].map(best).fillna("")
    return result


def legacy_groups(raw_train):
    """Reconstruct the user's original unnamespaced union-find before sorting.

    Do not change the traversal order, root representation, or GroupKFold
    implementation while importing the original .npy files.
    """
    parent = {}

    def find(value):
        while parent.setdefault(value, value) != value:
            parent[value] = parent[parent[value]]
            value = parent[value]
        return value

    for query, document in zip(raw_train.query_id, raw_train.document_id):
        root_query, root_document = find(query), find(document)
        if root_query != root_document:
            parent[root_query] = root_document
    return raw_train.query_id.map(lambda query: find(query) if query in parent else query).to_numpy()


def load_competition(data_dir):
    """Return (train, test, legacy_folds) in old cache order, retaining metadata.

    New cleaned columns use canonical observed-document restoration. Legacy
    model predictions can still be imported: their alignment depends on raw
    IDs and old fold membership, not on the new cleaned text.
    """
    directory = Path(data_dir)
    train = pd.read_csv(directory / "train.csv")
    test = pd.read_csv(directory / "test.csv")
    for name, frame in (("train", train), ("test", test)):
        if frame.id.isna().any() or not frame.id.is_unique:
            raise ValueError(f"{name}: id must be present and unique")
        if frame.query_id.isna().any() or frame.document_id.isna().any():
            raise ValueError(f"{name}: missing query_id/document_id")
    if not train.relevance.isin([0, 1]).all():
        raise ValueError("Only binary relevance is supported")
    if not set(train.query_id).isdisjoint(test.query_id):
        raise ValueError("Train/test queries overlap")
    if not set(train.document_id).isdisjoint(test.document_id):
        raise ValueError("Train/test documents overlap")
    train["grp"] = legacy_groups(train)
    train["is_train"], test["is_train"] = 1, 0
    test["relevance"] = -1
    test["grp"] = test.query_id
    combined = restore_documents(pd.concat([train, test], ignore_index=True))
    train = combined.loc[combined.is_train.eq(1)].sort_values(["query_id", "id"]).reset_index(drop=True)
    test = combined.loc[combined.is_train.eq(0)].sort_values(["query_id", "id"]).reset_index(drop=True)
    folds = list(GroupKFold(5).split(train, groups=train.grp))
    return train, test, folds


class ExactNDCG:
    """Fast exact mean NDCG@10, including string-ID ties and zero-positive queries."""

    def __init__(self, frame, label="relevance", k=10):
        self.n = len(frame)
        if not self.n:
            raise ValueError("NDCG needs at least one row")
        self.k = int(k)
        if self.k < 1:
            raise ValueError("k must be positive")
        self.ids = frame.id.astype(str).to_numpy()
        self.query_codes, self.query_ids = pd.factorize(frame.query_id, sort=True)
        self.n_queries = len(self.query_ids)
        self.labels = frame[label].to_numpy(dtype=float)
        if not np.isfinite(self.labels).all() or not np.isin(self.labels, [0, 1]).all():
            raise ValueError("Expected finite binary relevance")
        self.discounts = 1 / np.log2(np.arange(2, self.k + 2))
        self.cumulative_discounts = np.r_[0.0, self.discounts.cumsum()]
        n_positive = np.bincount(self.query_codes, weights=self.labels, minlength=self.n_queries).astype(int)
        self.idcg = self.cumulative_discounts[np.minimum(n_positive, self.k)]

    def per_query(self, prediction):
        """Return scores in self.query_ids order, with all-zero queries equal to zero."""
        prediction = np.asarray(prediction, dtype=float)
        if prediction.shape != (self.n,) or not np.isfinite(prediction).all():
            raise ValueError(f"Expected {self.n} finite predictions")
        order = np.lexsort((self.ids, -prediction, self.query_codes))
        groups = self.query_codes[order]
        start = np.maximum.accumulate(np.where(np.r_[True, groups[1:] != groups[:-1]], np.arange(self.n), 0))
        rank = np.arange(self.n) - start
        chosen = rank < self.k
        gains = self.labels[order[chosen]] * self.discounts[rank[chosen]]
        dcg = np.bincount(groups[chosen], weights=gains, minlength=self.n_queries)
        return np.divide(dcg, self.idcg, out=np.zeros_like(dcg), where=self.idcg > 0)

    def score(self, prediction):
        """Return competition mean NDCG@k."""
        return float(self.per_query(prediction).mean())

    def bootstrap_gain(self, baseline, candidate, n_boot=2000, seed=42):
        """Paired query-bootstrap 95% interval for candidate minus baseline.

        This quantifies query-sampling uncertainty conditional on these scores;
        it does not correct model-selection bias or stacked-CV dependencies.
        """
        delta = self.per_query(candidate) - self.per_query(baseline)
        rng = np.random.default_rng(seed)
        means = []
        for offset in range(0, int(n_boot), 100):
            size = min(100, int(n_boot) - offset)
            indices = rng.integers(0, self.n_queries, (size, self.n_queries))
            means.extend(delta[indices].mean(axis=1))
        if not means:
            raise ValueError("n_boot must be positive")
        low, high = np.quantile(means, [.025, .975])
        return {"gain": float(delta.mean()), "ci95_low": float(low), "ci95_high": float(high),
                "queries": self.n_queries, "bootstrap_samples": int(n_boot)}


def import_legacy_scores(train, test, folds, legacy_root, include_q3emb=True, tolerance=.00025):
    """Import only complete old caches with exact length and known OOF-score checks.

    Mismatches are declined with a warning, never silently trained or repaired.
    Returns dict[name, (OOF_train_array, averaged_test_array)]. Because the old
    files do not store IDs, expected OOF is an important additional sanity check.
    """
    scores = {}
    root = Path(legacy_root)
    if len(folds) != 5 or tuple(len(b) for _, b in folds) != LEGACY_FOLD_LENGTHS:
        warnings.warn("Legacy cache declined: fold lengths differ from the original notebook")
        return scores
    expected_order_train = train.sort_values(["query_id", "id"]).id.to_numpy()
    expected_order_test = test.sort_values(["query_id", "id"]).id.to_numpy()
    if not np.array_equal(train.id.to_numpy(), expected_order_train) or not np.array_equal(test.id.to_numpy(), expected_order_test):
        warnings.warn("Legacy cache declined: rows are not in the original (query_id,id) order")
        return scores
    metric = ExactNDCG(train)
    for name, expected in LEGACY_EXPECTED.items():
        directory = root / name
        paths = [(directory / f"oof_{k}.npy", directory / f"te_{k}.npy") for k in range(5)]
        if not all(a.is_file() and b.is_file() for a, b in paths):
            warnings.warn(f"Legacy {name}: incomplete or absent cache; skipped")
            continue
        try:
            oof = np.empty(len(train), dtype=np.float32)
            test_mean = np.zeros(len(test), dtype=np.float32)
            for (_, validation), (oof_path, test_path) in zip(folds, paths):
                val = np.load(oof_path, allow_pickle=False)
                pred = np.load(test_path, allow_pickle=False)
                if val.shape != (len(validation),) or pred.shape != (len(test),):
                    raise ValueError(f"wrong shape in {oof_path.name}/{test_path.name}")
                if not np.isfinite(val).all() or not np.isfinite(pred).all():
                    raise ValueError("nonfinite scores")
                oof[validation] = val
                # Preserve the float32 accumulation used by the user's notebook.
                test_mean += pred.astype(np.float32) / 5
            actual = metric.score(oof)
            if abs(actual - expected) > tolerance:
                raise ValueError(f"OOF {actual:.6f}, expected approximately {expected:.4f}; alignment is unsafe")
            scores[name] = (oof, test_mean.astype(np.float32))
            print(f"Imported legacy {name}: OOF NDCG@10={actual:.6f}")
        except (OSError, ValueError, TypeError) as error:
            warnings.warn(f"Legacy {name} declined: {error}")
    if include_q3emb and (root / "q3emb.npz").is_file():
        try:
            with np.load(root / "q3emb.npz", allow_pickle=False) as cache:
                a, b = cache["tr"].copy(), cache["te"].copy()
            if a.shape != (len(train),) or b.shape != (len(test),) or not np.isfinite(a).all() or not np.isfinite(b).all():
                raise ValueError("wrong shape or nonfinite scores")
            actual = metric.score(a)
            if abs(actual - .7327) > tolerance:
                raise ValueError(f"OOF/zero-shot sanity {actual:.6f}, expected approximately .7327")
            scores["q3emb_legacy"] = (a.astype(np.float32), b.astype(np.float32))
            print(f"Imported legacy q3emb: train NDCG@10={actual:.6f}")
        except (OSError, ValueError, TypeError, KeyError) as error:
            warnings.warn(f"Legacy q3emb declined: {error}")
    if scores:
        warnings.warn("Imported train CE scores are OOF; test CE scores average five models. Ordinary stack CV is approximate, not nested end-to-end CV.")
    return scores


def frame_signature(frame):
    """Order-independent SHA256 of raw IDs/text/metadata/labels and clean texts."""
    columns = sorted(c for c in _SIGNED_COLUMNS if c in frame.columns)
    stable = frame.sort_values("id")[columns]
    return hashlib.sha256(stable.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest()


def dataset_signature(train, test):
    """Signature covering the full data and the preprocessing implementation version."""
    value = PREPROCESS_VERSION + ":" + frame_signature(train) + ":" + frame_signature(test)
    return hashlib.sha256(value.encode()).hexdigest()


def save_score_cache(path, train, test, train_scores, test_scores, metadata=None):
    """Atomically save scores with explicit row IDs, input digest and user metadata."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    a, b = np.asarray(train_scores, dtype=np.float32), np.asarray(test_scores, dtype=np.float32)
    if a.shape != (len(train),) or b.shape != (len(test),) or not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError("Cache predictions have invalid shape or nonfinite values")
    meta = {"format_version": 1, "preprocess_version": PREPROCESS_VERSION,
            "dataset_signature": dataset_signature(train, test), "user": metadata or {}}
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, prefix=path.name + ".", suffix=".tmp", delete=False) as handle:
            temporary = Path(handle.name)
            np.savez_compressed(handle, train_ids=train.id.astype(str).to_numpy(dtype=str),
                                test_ids=test.id.astype(str).to_numpy(dtype=str), train_scores=a, test_scores=b,
                                metadata=np.asarray(json.dumps(meta, ensure_ascii=False, sort_keys=True, allow_nan=False)))
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    return meta


def load_score_cache(path, train, test, expected_metadata=None):
    """Load, verify and align a new cache by IDs; return (train_scores,test_scores).

    Raises ValueError on an input digest, metadata, ID, shape, or finite-value
    mismatch. A caller may catch it and recompute the whole named experiment.
    """
    with np.load(path, allow_pickle=False) as cache:
        metadata = json.loads(str(cache["metadata"].item()))
        a_ids, b_ids = cache["train_ids"].astype(str), cache["test_ids"].astype(str)
        a, b = cache["train_scores"].copy(), cache["test_scores"].copy()
    if metadata.get("format_version") != 1 or metadata.get("preprocess_version") != PREPROCESS_VERSION:
        raise ValueError("Cache format/preprocessing version mismatch")
    if metadata.get("dataset_signature") != dataset_signature(train, test):
        raise ValueError("Cache input dataset signature mismatch")
    user = metadata.get("user", {})
    if expected_metadata is not None and any(user.get(k) != value for k, value in expected_metadata.items()):
        raise ValueError("Cache experiment metadata mismatch")
    result = []
    for ids, values, frame in ((a_ids, a, train), (b_ids, b, test)):
        if ids.shape != (len(frame),) or values.shape != (len(frame),) or len(set(ids)) != len(ids):
            raise ValueError("Cache IDs/prediction shape mismatch")
        target_ids = frame.id.astype(str).to_numpy()
        if set(ids) != set(target_ids) or not np.isfinite(values).all():
            raise ValueError("Cache IDs differ or predictions are nonfinite")
        result.append(pd.Series(values, index=ids).reindex(target_ids).to_numpy(dtype=np.float32))
    return tuple(result)


In [ ]:
%%writefile tabular_features.py
"""Fast, label-free ranking features; inputs must contain query/title/text_c.

The output preserves the input row order and index. Corpus statistics use only
text, never relevance. Score-derived statistics stay inside each query.
"""
from collections import Counter
import re
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

TOK = re.compile(r'[а-яёa-z0-9]+', re.I)
NUM = re.compile(r'\d+(?:[.,]\d+)?')
STOP = set('а без был была были было быть в во вот все всего вы где да для до его ее если есть еще за зачем и из или им их к как какой какая какое какие когда кто ли либо мне на над не него нее нет ни но о об он она они оно от по под при про с со так там те тем то того тоже тут ты у уже что чтобы это этот эта эти я году года год сколько'.split())
# A deliberately conservative Russian suffix stemmer: no external morphology.
SUFFIX = re.compile(r'(иями|ями|ами|иях|ах|ях|ого|ему|ому|ыми|ими|ией|иям|ием|ия|ие|ий|ая|яя|ое|ее|ые|ие|ую|юю|ов|ев|ей|ам|ям|ом|ем|ах|ях|ы|а|я|о|е|у|ю|и|ь)$')

def _tokens(text):
    out=[]
    for w in TOK.findall(str(text).lower().replace('ё','е')):
        if w in STOP or len(w)<2:
            continue
        if len(w)>4 and not w.isdigit():
            w = SUFFIX.sub('', w)
        out.append(w)
    return out


def _num(series):
    return pd.to_numeric(series.astype(str).str.replace(r'\s+', '', regex=True).str.replace(',', '.', regex=False), errors='coerce')


def _add_query_stats(out, df, columns):
    q = df['query_id'].astype(str).to_numpy()
    for c in columns:
        s=pd.Series(np.asarray(out[c]), index=np.arange(len(df)))
        g=s.groupby(q,sort=False)
        out[c+'_qrk'] = g.rank(ascending=False,pct=True).to_numpy()
        out[c+'_qz'] = ((s-g.transform('mean')) / (g.transform('std').fillna(0)+1e-5)).to_numpy()
        out[c+'_qmaxgap'] = (s-g.transform('max')).to_numpy()


def build_features(df, verbose=True):
    """Return a finite float32 frame, aligned with df, without reading labels."""
    start=time.monotonic()
    n=len(df)
    qtexts=df['query_c'].fillna('').astype(str).tolist()
    titles=df['title_c'].fillna('').astype(str).tolist()
    texts=df['text_c'].fillna('').astype(str).tolist()
    # Cache repeated strings: queries and duplicated passages are common.
    cache={}
    def tok(s):
        if s not in cache:
            cache[s]=_tokens(s)
        return cache[s]
    qt=[tok(s) for s in qtexts]; tt=[tok(s) for s in texts]; ht=[tok(s) for s in titles]
    unique_pos=np.flatnonzero(~pd.Series(texts).duplicated().to_numpy())
    N=max(1,len(unique_pos)); freq=Counter()
    for i in unique_pos:
        freq.update(set(tt[i]))
    idf={w:np.log1p((N-c+0.5)/(c+0.5)) for w,c in freq.items()}
    unknown=np.log1p((N+0.5)/0.5)
    avgdl=max(1.,np.mean([len(tt[i]) for i in unique_pos]))
    keys=['bm25','coverage','idf_coverage','title_coverage','title_idf_coverage',
          'first_coverage','best_sentence_coverage','bigram_coverage',
          'title_precision','q_words','text_words','title_words','number_overlap',
          'number_recall','q_number_count','text_number_count','text_year_count',
          'q_title_substring','title_q_substring','max_matched_span']
    out={k:np.zeros(n,dtype=np.float32) for k in keys}
    for i,(q,t,h) in enumerate(zip(qt,tt,ht)):
        qs=set(q); ts=set(t); hs=set(h); count=Counter(t); qlen=max(1,len(qs))
        weights=sum(idf.get(w,unknown) for w in qs) or 1.
        match=qs & ts
        out['bm25'][i]=sum(idf.get(w,unknown)*count[w]*2.2/(count[w]+1.2*(.25+.75*len(t)/avgdl)) for w in match)
        out['coverage'][i]=len(match)/qlen
        out['idf_coverage'][i]=sum(idf.get(w,unknown) for w in match)/weights
        out['title_coverage'][i]=len(qs & hs)/qlen
        out['title_idf_coverage'][i]=sum(idf.get(w,unknown) for w in qs & hs)/weights
        out['title_precision'][i]=len(qs & hs)/max(1,len(hs))
        out['first_coverage'][i]=len(qs & set(t[:35]))/qlen
        # Fixed windows avoid excessively expensive sentence tokenization.
        out['best_sentence_coverage'][i]=max((len(qs & set(t[j:j+40]))/qlen for j in range(0,max(1,len(t)),25)),default=0.)
        qb=set(zip(q,q[1:])); tb=set(zip(t,t[1:])); out['bigram_coverage'][i]=len(qb&tb)/max(1,len(qb))
        out['q_words'][i]=len(q);out['text_words'][i]=len(t);out['title_words'][i]=len(h)
        qn=set(NUM.findall(qtexts[i]));tn=set(NUM.findall(texts[i]));out['number_overlap'][i]=len(qn&tn)
        out['number_recall'][i]=len(qn&tn)/max(1,len(qn));out['q_number_count'][i]=len(qn);out['text_number_count'][i]=len(tn)
        out['text_year_count'][i]=sum(len(x)==4 and x.isdigit() and 1000<=int(x)<=2100 for x in tn)
        qjoin=' '.join(q);hjoin=' '.join(h)
        out['q_title_substring'][i]=int(bool(qjoin) and qjoin in hjoin)
        out['title_q_substring'][i]=int(bool(hjoin) and hjoin in qjoin)
        positions=[j for j,w in enumerate(t) if w in qs]
        out['max_matched_span'][i]=(positions[-1]-positions[0]+1)/max(1,len(positions)) if positions else len(t)
    # TF-IDF fitted on unique documents; all pair similarities are row-aligned.
    doc_strings=[' '.join(tt[i]) for i in unique_pos]
    word=TfidfVectorizer(tokenizer=str.split,token_pattern=None,lowercase=False,
                         max_features=120000,ngram_range=(1,2),dtype=np.float32,min_df=min(2,N))
    word.fit(doc_strings)
    qmat=word.transform([' '.join(x) for x in qt]);tmat=word.transform([' '.join(x) for x in tt]);hmat=word.transform([' '.join(x) for x in ht])
    out['tfidf_word_text']=np.asarray(qmat.multiply(tmat).sum(axis=1)).ravel()
    out['tfidf_word_title']=np.asarray(qmat.multiply(hmat).sum(axis=1)).ravel()
    del qmat,tmat,hmat,word
    # Character features handle inflection, misspellings and proper nouns.
    char=TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),max_features=80000,
                         min_df=min(3,N),sublinear_tf=True,dtype=np.float32)
    char.fit([texts[i][:4000].lower().replace('ё','е') for i in unique_pos])
    qmat=char.transform([x.lower().replace('ё','е') for x in qtexts]);tmat=char.transform([x[:4000].lower().replace('ё','е') for x in texts]);hmat=char.transform([x.lower().replace('ё','е') for x in titles])
    out['tfidf_char_text']=np.asarray(qmat.multiply(tmat).sum(axis=1)).ravel()
    out['tfidf_char_title']=np.asarray(qmat.multiply(hmat).sum(axis=1)).ravel()
    del qmat,tmat,hmat,char
    src=df['ingest_source'].fillna('').astype(str)
    raw_idx=pd.to_numeric(df['passage_index'],errors='coerce')
    idx=raw_idx-(src=='S3').astype(float)
    cnt=pd.to_numeric(df['article_passage_count'],errors='coerce')
    chars_raw=_num(df['article_char_count'])
    chars=chars_raw.where(chars_raw<1e6,chars_raw/1000)
    # S2 has ~1.8x higher chars/passage at matching passage text lengths;
    # keep raw features too, since this is a source scale heuristic.
    chars=chars/(np.where(src.eq('S2'),1.8,1.))
    out['passage_index_raw']=raw_idx.to_numpy()
    out['passage_index']=idx.to_numpy()
    out['passage_count_raw']=cnt.to_numpy()
    out['chars_raw_log']=np.log1p(chars_raw.clip(lower=0)).to_numpy()
    out['chars_fixed_log']=np.log1p(chars.clip(lower=0)).to_numpy()
    out['chars_x1000_flag']=(chars_raw>=1e6).astype(float).to_numpy()
    out['count_invalid']=((cnt<=0)|cnt.isna()).astype(float).to_numpy()
    out['index_invalid']=((idx<0)|(idx>=cnt)|idx.isna()).astype(float).to_numpy()
    out['first_passage']=(idx==0).astype(float).to_numpy()
    cnt=cnt.where(cnt>0)
    out['relative_passage_index']=(idx/cnt).to_numpy()
    out['chars_per_passage']=(chars/cnt).to_numpy()
    out['passage_count']=cnt.to_numpy()
    for c in ['title','passage_index','article_passage_count','article_char_count']:
        out[c+'_missing']=df[c].isna().astype(float).to_numpy()
    for s in ['S0','S1','S2','S3']:
        out['source_'+s]=src.eq(s).astype(float).to_numpy()
    out['query_chars']=np.array([len(x) for x in qtexts],dtype=np.float32)
    out['title_chars']=np.array([len(x) for x in titles],dtype=np.float32)
    out['text_chars']=np.array([len(x) for x in texts],dtype=np.float32)
    out['text_digit_frac']=np.array([sum(c.isdigit() for c in x)/max(1,len(x)) for x in texts],dtype=np.float32)
    out['text_colon_count']=np.array([x.count(':') for x in texts],dtype=np.float32)
    out['text_question_count']=np.array([x.count('?') for x in texts],dtype=np.float32)
    for w in ['кто','что','где','когда','почему','зачем','как','сколько','какой','какая','каком','чем','ли']:
        pat=re.compile(r'\b'+w+r'\b')
        out['question_'+w]=np.array([bool(pat.search(x.lower())) for x in qtexts],dtype=np.float32)
    for c in ['was_moji','has_stress','q_stress']:
        if c in df:
            out[c]=pd.to_numeric(df[c],errors='coerce').to_numpy()
    qgroup=df.groupby('query_id',sort=False)
    out['query_candidates']=qgroup['id'].transform('size').to_numpy()
    agroup=df.groupby(['query_id','article_id'],sort=False)
    out['article_candidates']=agroup['id'].transform('size').to_numpy()
    out['article_fraction']=out['article_candidates']/np.maximum(1,out['query_candidates'])
    out['query_article_count']=qgroup['article_id'].transform('nunique').to_numpy()
    _add_query_stats(out,df,['bm25','coverage','idf_coverage','title_coverage',
                           'tfidf_word_text','tfidf_word_title','tfidf_char_text','tfidf_char_title','text_chars','passage_index'])
    group_keys=[df['query_id'].to_numpy(),df['article_id'].to_numpy()]
    for c in ['bm25','tfidf_word_text','coverage']:
        s=pd.Series(out[c])
        g=s.groupby(group_keys,sort=False)
        out[c+'_article_max']=g.transform('max').to_numpy()
        out[c+'_article_mean']=g.transform('mean').to_numpy()
    result=pd.DataFrame(out,index=df.index,dtype=np.float32)
    result=result.replace([np.inf,-np.inf],np.nan).fillna(-1.)
    if verbose:
        print(f'Feature frame {result.shape}; {time.monotonic()-start:.1f}s; label-free corpus statistics')
    return result


def score_features(df, scores):
    """scores: mapping of feature name to row-aligned 1D scores.

    Query/article aggregates cannot transfer information across held-out queries.
    """
    n=len(df);out={}; q=df['query_id'].astype(str).to_numpy()
    group_keys=[q,df['article_id'].astype(str).to_numpy()]
    for name, values in scores.items():
        values=np.asarray(values,dtype=np.float32).reshape(-1)
        if len(values)!=n or not np.isfinite(values).all():
            raise ValueError(f'{name}: expected {n} finite row-aligned scores')
        name='score_'+str(name)
        out[name]=values
        s=pd.Series(values)
        g=s.groupby(q,sort=False)
        std=g.transform('std').fillna(0)
        out[name+'_qz']=((s-g.transform('mean'))/(std+1e-5)).to_numpy()
        out[name+'_qrk']=g.rank(ascending=False,pct=True).to_numpy()
        out[name+'_qgap']=(s-g.transform('max')).to_numpy()
        out[name+'_qstd']=std.to_numpy()
        out[name+'_qrange']=(g.transform('max')-g.transform('min')).to_numpy()
        ag=s.groupby(group_keys,sort=False)
        out[name+'_article_max']=ag.transform('max').to_numpy()
        out[name+'_article_mean']=ag.transform('mean').to_numpy()
        out[name+'_article_gap']=(s-ag.transform('max')).to_numpy()
        # Peer article signal excludes the current passage itself.
        count=ag.transform('size')
        peer=(ag.transform('sum')-s)/np.maximum(1,count-1)
        out[name+'_article_peer_mean']=peer.where(count>1,s).to_numpy()
    keys=list(scores)
    if len(keys)>1:
        z=np.column_stack([out['score_'+str(k)+'_qz'] for k in keys])
        out['score_mean_qz']=z.mean(axis=1)
        out['score_std_qz']=z.std(axis=1)
        out['score_min_qz']=z.min(axis=1)
        out['score_max_qz']=z.max(axis=1)
    return pd.DataFrame(out,index=df.index,dtype=np.float32).replace([np.inf,-np.inf],np.nan).fillna(-1.)


In [ ]:
%%writefile legacy_original.py
"""Optional original BGE training for reproduction from scratch.
Not executed on the 5-hour cached route. This follows the uploaded solution.
Its scores are source-OOF features, not nested stacking validation.
"""
import re, html, unicodedata, math
import numpy as np, pandas as pd
from collections import Counter
from functools import lru_cache

MOJI_RE = re.compile(r'[РС][\u0080-\u00bf\u0400-\u04ff]')
def fix_moji(s):
    """UTF-8 текст, ошибочно прочитанный как cp1251 -> чиним обратно."""
    if not isinstance(s, str): return s
    if len(MOJI_RE.findall(s)) > 5 and not re.search(r'[а-яё]{4}', s):
        try: return s.encode('cp1251', errors='strict').decode('utf-8', errors='strict')
        except Exception:
            try: return s.encode('cp1251', errors='ignore').decode('utf-8', errors='ignore')
            except Exception: return s
    return s

TAG_RE = re.compile(r'</?[a-zA-Z][^>]*>')
NAV_RE = re.compile(r'Навигация:\s*Главная\s*\|\s*Статьи')
def clean_text(s):
    if not isinstance(s, str): return ''
    s = fix_moji(s)
    s = html.unescape(TAG_RE.sub(' ', s))
    s = NAV_RE.sub(' ', s)
    s = s.replace('\u0301', '').replace('\u00ad', '')
    s = unicodedata.normalize('NFKC', s)       # \xa0, \u202f -> пробел
    s = re.sub(r'\(\s*[,;]?\s*\)', ' ', s)      # пустые скобки от вырезанных дат
    return re.sub(r'\s+', ' ', s).strip()

def parse_num(x):
    if not isinstance(x, str): return np.nan
    x = unicodedata.normalize('NFKC', x).replace(' ', '').replace(',', '.')
    try: return float(x)
    except: return np.nan

def load(data_dir='/mnt/user-data/uploads'):
    tr = pd.read_csv(f'{data_dir}/train.csv'); te = pd.read_csv(f'{data_dir}/test.csv')
    tr['is_train'] = 1; te['is_train'] = 0; te['relevance'] = -1
    df = pd.concat([tr, te], ignore_index=True)
    df['was_moji'] = df.text.map(lambda s: fix_moji(s) != s).astype(int)
    df['has_stress'] = df.text.str.contains('\u0301').astype(int)
    df['q_stress'] = df['query'].str.contains('\u0301').astype(int)
    df['query_c'] = df['query'].map(clean_text)
    df['title_c'] = df['title'].map(clean_text)
    df['text_c'] = df['text'].map(clean_text)
    df['acc'] = df.article_char_count.map(parse_num)
    # восстановление пропусков метаданных по статье / документу
    df['apc'] = df.article_passage_count
    for c, src in [('apc', 'article_id'), ('acc', 'article_id')]:
        df[c] = df[c].fillna(df.groupby(src)[c].transform('median'))
    df['pidx'] = df.passage_index.fillna(df.groupby('document_id').passage_index.transform('first'))
    df['title_c'] = df['title_c'].where(df['title_c'] != '', np.nan)
    df['title_c'] = df['title_c'].fillna(df.groupby('article_id').title_c.transform('first')).fillna('')
    return df

def make_groups(df):
    """Связные компоненты запрос<->документ: в тесте документы не пересекаются с train."""
    tr = df[df.is_train == 1]
    parent = {}
    def f(x):
        while parent.setdefault(x, x) != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for q, d in zip(tr.query_id, tr.document_id):
        ra, rb = f(q), f(d)
        if ra != rb: parent[ra] = rb
    return df.query_id.map(lambda q: f(q) if q in parent else q)

def ndcg10(df, col, label='relevance'):
    res = []
    for q, d in df.groupby('query_id'):
        d = d.assign(_id=d['id'].astype(str)).sort_values([col, '_id'], ascending=[False, True])
        rel = d[label].values; disc = 1 / np.log2(np.arange(2, len(rel) + 2))
        idcg = (np.sort(rel)[::-1][:10] * disc[:10]).sum()
        res.append((rel[:10] * disc[:10]).sum() / idcg if idcg > 0 else 0)
    return float(np.mean(res))
import os, time, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import GroupKFold

dev = 'cuda'
OUT_ROOT = './unused_legacy_default'

# ---- конфиги моделей ----
CFG = {
  'bge': dict(name='BAAI/bge-reranker-v2-m3', prefix=('', ''), lr=2e-5, epochs=1, qbs=1, accum=4, maxlen=256, freeze_emb=True),
  'bge2': dict(name='BAAI/bge-reranker-v2-m3', prefix=('', ''), lr=2e-5, epochs=1, qbs=1, accum=2, maxlen=256, freeze_emb=True, seed=7),
  'bge3': dict(name='BAAI/bge-reranker-v2-m3', prefix=('', ''), lr=2e-5, epochs=1, qbs=1, accum=4, maxlen=320, freeze_emb=True, seed=11),
  'e5':   dict(name='intfloat/multilingual-e5-base', prefix=('query: ', 'passage: '), lr=3e-5, epochs=2, qbs=4, accum=1, maxlen=320, freeze_emb=True),
}

def fmt(df, prefix):
    title = df.title_c.values; text = df.text_c.values
    p = [prefix[1] + (t + '. ' + x if t else x) for t, x in zip(title, text)]
    q = [prefix[0] + x for x in df.query_c.values]
    return q, p

def query_groups(df):
    """df отсортирован по query_id -> строки запроса идут подряд."""
    qid = df.query_id.values; cuts = np.flatnonzero(qid[1:] != qid[:-1]) + 1
    return np.split(np.arange(len(df)), cuts)

def train_batches(groups, qbs, rng):
    """Батч = несколько запросов целиком (все их кандидаты) -> listwise-лосс."""
    perm = rng.permutation(len(groups)); out = []
    for i in range(0, len(perm), qbs):
        gs = [groups[j] for j in perm[i:i + qbs]]
        out.append((np.concatenate(gs).tolist(), [len(g) for g in gs]))
    return out

def infer_batches(lens, bs):
    order = np.argsort(lens, kind='stable')
    return order, [(order[i:i + bs].tolist(), None) for i in range(0, len(order), bs)]

def make_loader(tok, q, p, y, batches, maxlen):
    def coll(item):
        idx, sizes = item
        enc = tok([q[i] for i in idx], [p[i] for i in idx], truncation='only_second', max_length=maxlen, padding=True, return_tensors='pt')
        yy = torch.tensor([y[i] for i in idx], dtype=torch.float) if y is not None else None
        return dict(enc), yy, sizes
    return DataLoader(batches, batch_size=None, shuffle=False, collate_fn=coll, num_workers=2)

def listwise_loss(z, y, sizes):
    out = []; s = 0
    for n in sizes:
        zz, yy = z[s:s + n], y[s:s + n]; s += n
        if yy.sum() > 0: out.append(-(yy * torch.log_softmax(zz, 0)).sum() / yy.sum())
    return torch.stack(out).mean() if out else z.sum() * 0

@torch.no_grad()
def predict(model, tok, q, p, maxlen, bs=64):
    lens = [len(a) + len(b) for a, b in zip(q, p)]
    order, batches = infer_batches(lens, bs); model.eval(); out = []
    for enc, _, _ in make_loader(tok, q, p, None, batches, maxlen):
        enc = {k: v.to(dev) for k, v in enc.items()}
        with torch.autocast('cuda', dtype=torch.float16): out.append(model(**enc).logits.squeeze(-1).float().cpu())
    res = np.empty(len(q), dtype=np.float32); res[order] = torch.cat(out).numpy(); return res

def train_fold(cfg, out_dir, k, A, B, T, tok, seed=None):
    seed = cfg.get('seed', 42) if seed is None else seed
    f_oof, f_te = f'{out_dir}/oof_{k}.npy', f'{out_dir}/te_{k}.npy'
    if os.path.exists(f_oof) and os.path.exists(f_te):
        print(f'fold{k}: загружен с диска'); return np.load(f_oof), np.load(f_te)
    torch.manual_seed(seed + k); rng = np.random.RandomState(seed + k)
    model = AutoModelForSequenceClassification.from_pretrained(cfg['name'], revision=cfg['revision'], num_labels=1).to(dev)
    if cfg['freeze_emb']:   # эмбеддинги слов = половина параметров: экономим память и время
        for prm in model.base_model.embeddings.word_embeddings.parameters(): prm.requires_grad = False
    qa, pa = fmt(A, cfg['prefix']); ya = A.relevance.tolist(); ga = query_groups(A)
    qb, pb = fmt(B, cfg['prefix']); qt, pt = fmt(T, cfg['prefix'])
    n_b = len(train_batches(ga, cfg['qbs'], rng)); acc = cfg.get('accum', 1)
    steps = cfg['epochs'] * ((n_b + acc - 1) // acc)
    params = [p_ for p_ in model.parameters() if p_.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg['lr'], weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler('cuda'); bce = torch.nn.BCEWithLogitsLoss(); t0 = time.time()
    for ep in range(cfg['epochs']):
        model.train(); loader = make_loader(tok, qa, pa, ya, train_batches(ga, cfg['qbs'], rng), cfg['maxlen'])
        opt.zero_grad(set_to_none=True)
        for i, (enc, y, sizes) in enumerate(loader):
            enc = {kk: v.to(dev) for kk, v in enc.items()}; y = y.to(dev)
            with torch.autocast('cuda', dtype=torch.float16): z = model(**enc).logits.squeeze(-1)
            z = z.float(); loss = 0.5 * bce(z, y) + listwise_loss(z, y, sizes)
            scaler.scale(loss / acc).backward()
            if (i + 1) % acc == 0 or i == n_b - 1:     # накопление градиента: эффективный батч = qbs*accum запросов
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
            if i % 400 == 0: print(f'fold{k} ep{ep} it{i}/{n_b} loss {loss.item():.4f} {time.time()-t0:.0f}s | GPU {torch.cuda.max_memory_allocated()/2**30:.1f} GB', flush=True)
    o = predict(model, tok, qb, pb, cfg['maxlen'])
    print(f'fold{k} val NDCG@10 = %.4f  ({time.time()-t0:.0f}s)' % ndcg10(B.assign(s=o), 's'), flush=True)
    te = predict(model, tok, qt, pt, cfg['maxlen']); np.save(f_oof, o); np.save(f_te, te)
    del model, opt; torch.cuda.empty_cache(); return o, te

def run_ce(key, tr, te, smoke=False, n_folds=5):
    """tr/te отсортированы по query_id. Возвращает OOF для train и усреднённый скор для test."""
    cfg = CFG[key]; out_dir = f'{OUT_ROOT}/{key}' + ('_smoke' if smoke else ''); os.makedirs(out_dir, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(cfg['name'])
    oof = np.zeros(len(tr), dtype=np.float32); tp = np.zeros(len(te), dtype=np.float32); done = 0
    for k, (a, b) in enumerate(GroupKFold(n_folds).split(tr, groups=tr.grp)):
        A, B, T = tr.iloc[a].reset_index(drop=True), tr.iloc[b].reset_index(drop=True), te
        if smoke: A, B, T = A.head(500), B.head(300), te.head(300)
        o, t = train_fold(cfg, out_dir, k, A, B, T, tok)
        if smoke: print('SMOKE ok:', o[:3], t[:3]); return None, None
        oof[b] = o; tp += t / n_folds; done += 1
    print(f'[{key}] OOF NDCG@10 = %.4f' % ndcg10(tr.assign(s=oof), 's')); return oof, tp

def rebuild_legacy(data_dir, legacy_root):
    global OUT_ROOT
    OUT_ROOT=str(legacy_root)
    df=load(str(data_dir)); df['grp']=make_groups(df)
    train=df[df.is_train==1].sort_values(['query_id','id']).reset_index(drop=True)
    test=df[df.is_train==0].sort_values(['query_id','id']).reset_index(drop=True)
    pairs={}
    for key in ['bge','bge2','bge3']:
        pairs[key]=run_ce(key,train,test,smoke=False,n_folds=5)
    return pairs


In [ ]:
%%writefile neural_features.py
"""GPU inference only; supervised ranking is in meta_stack.py.
Qwen pooling/prompt follow the official model cards. No external training sets.
"""
import gc
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np

TASK = 'Given a web search query, retrieve relevant passages that answer the query'
RR_PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. '
             'Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
RR_SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

class StageBudget(Exception):
    pass

class TimeBudget:
    def __init__(self, total_seconds=4.65*3600, started_at=None, reserve_seconds=900):
        self.started_at = time.monotonic() if started_at is None else started_at
        self.deadline = self.started_at + total_seconds
        self.reserve = reserve_seconds
    def remaining(self):
        return max(0., self.deadline - time.monotonic())
    def for_neural(self):
        return max(0., self.remaining() - self.reserve)
    def check(self):
        if self.for_neural() < 30:
            raise StageBudget('Оставляем время для обучения ранкера и записи submission.')


def _atomic_npz(path, **arrays):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name+'.tmp')
    with open(tmp, 'wb') as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)


def resolve_revision(model_name, cache_dir):
    """Pin the first downloaded revision for subsequent resumes."""
    from huggingface_hub import model_info
    path = Path(cache_dir)/'model_revisions.json'
    manifest = json.loads(path.read_text()) if path.exists() else {}
    if model_name not in manifest:
        manifest[model_name] = model_info(model_name).sha
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix('.tmp'); tmp.write_text(json.dumps(manifest, indent=2)); os.replace(tmp,path)
    return manifest[model_name]


def _signature(ids, texts, config):
    h = hashlib.sha256(json.dumps(config, sort_keys=True).encode())
    for i,t in zip(ids,texts):
        for s in (str(i),str(t)):
            b=s.encode(); h.update(len(b).to_bytes(8,'little')); h.update(b)
    return h.hexdigest()


def _cached_vectors(path, ids, signature, width=None):
    if Path(path).exists():
        with np.load(path, allow_pickle=False) as z:
            if str(z['signature'].item()) != signature or not np.array_equal(z['ids'].astype(str),np.asarray(ids,dtype=str)):
                raise ValueError(f'Несовместимый новый кэш {path}; изменились тексты/модель/порядок. Укажите новую RUN_NAME.')
            values=z['values'].copy()
            expected=(len(ids),) if width is None else (len(ids),width)
            if values.shape != expected or not np.issubdtype(values.dtype,np.number):
                raise ValueError(f'Повреждён новый кэш {path}: shape={values.shape}, expected={expected}')
            # NaN marks unfinished rows; infinity is never a valid partial result.
            if np.isinf(values).any():
                raise ValueError(f'Повреждён новый кэш {path}: infinite predictions')
            return values.astype(np.float32,copy=False)
    return np.full((len(ids),) if width is None else (len(ids),width), np.nan, dtype=np.float32)


def _save_vectors(path, ids, signature, values):
    _atomic_npz(path, ids=np.asarray(ids,dtype=str), signature=np.asarray(signature), values=values)


def _run_cached(ids, texts, signature, path, lengths, predict_batch, batch_size, budget,
                width=None, benchmark=False, description='model'):
    """Resume ID-bound partial arrays. Never return unfinished scores to the stack."""
    import torch
    values = _cached_vectors(path, ids, signature, width)
    done = np.isfinite(values) if values.ndim==1 else np.isfinite(values).all(1)
    missing = np.flatnonzero(~done)
    if not len(missing):
        print(description, '— полный кэш:',len(ids)); return values
    order = missing[np.argsort(np.asarray(lengths)[missing],kind='stable')]
    current_bs = batch_size
    def evaluate(selected):
        nonlocal current_bs
        pos=0
        while pos<len(selected):
            budget.check()
            ix=selected[pos:pos+current_bs]
            try:
                result=np.asarray(predict_batch(ix),dtype=np.float32)
                expected=(len(ix),) if width is None else (len(ix),width)
                if result.shape != expected:
                    raise ValueError(f'{description}: shape={result.shape}, expected={expected}')
                if not np.isfinite(result).all():
                    raise FloatingPointError(f'{description}: nonfinite outputs; incomplete cache will not be used.')
                values[ix]=result; pos+=len(ix)
            except torch.cuda.OutOfMemoryError:
                gc.collect(); torch.cuda.empty_cache()
                if current_bs==1: raise
                current_bs=max(1,current_bs//2)
                print(description,'OOM -> batch',current_bs,flush=True)
    try:
        if benchmark:
            budget.check()
            # Warmup is cached but excluded from the measured throughput.
            evaluate(order[len(order)//2:len(order)//2+min(current_bs,len(order))])
            if torch.cuda.is_available(): torch.cuda.synchronize()
            probe=order[np.unique(np.linspace(0,len(order)-1,min(96,len(order))).astype(int))]
            start=time.monotonic(); evaluate(probe)
            if torch.cuda.is_available(): torch.cuda.synchronize()
            rate=len(probe)/max(time.monotonic()-start,1e-6)
            left=(~(np.isfinite(values) if width is None else np.isfinite(values).all(1))).sum()
            estimate=1.35*left/rate
            _save_vectors(path,ids,signature,values)
            print(f'{description}: {rate:.2f} пар/с, оценка остатка с запасом {estimate/60:.1f} мин; '
                  f'доступно {budget.for_neural()/60:.1f} мин',flush=True)
            if estimate>budget.for_neural():
                print(description,'— полный прогон не помещается в бюджет; сохранён частичный кэш.')
                return None
        last_save=time.monotonic(); last_checkpoint=time.monotonic(); start=time.monotonic(); processed=0
        since_checkpoint=0
        # Chunks checkpoint often enough for Colab disconnect; sorted lengths reduce padding.
        for offset in range(0,len(order),256):
            block=order[offset:offset+256]
            active=block[~(np.isfinite(values[block]) if width is None else np.isfinite(values[block]).all(1))]
            if len(active): evaluate(active)
            processed+=len(block)
            since_checkpoint+=len(active)
            # Scalar scores are tiny. Large vector arrays should not rewrite
            # hundreds of MB on Drive after every 256 documents.
            if width is None or time.monotonic()-last_checkpoint>=60 or since_checkpoint>=2048 or offset+256>=len(order):
                _save_vectors(path,ids,signature,values)
                last_checkpoint=time.monotonic(); since_checkpoint=0
            if time.monotonic()-last_save>45 or offset==0:
                print(f'{description}: {np.isfinite(values).all(1).sum() if values.ndim>1 else np.isfinite(values).sum()}/{len(ids)}, '
                      f'{(time.monotonic()-start)/60:.1f} мин',flush=True); last_save=time.monotonic()
    except StageBudget:
        _save_vectors(path,ids,signature,values)
        print(description,'— остановлен по бюджету, частичный кэш сохранён.'); return None
    except BaseException:
        # Keep all already validated batches on OOM/nonfinite output or manual
        # interruption, then let the caller decide whether to use a fallback.
        _save_vectors(path,ids,signature,values)
        raise
    assert np.isfinite(values).all()
    return values


def _load_transformer(model_name, model_cls, revision):
    import torch
    kwargs=dict(revision=revision,torch_dtype=torch.float16,low_cpu_mem_usage=True,
                device_map={'':'cuda:0'},attn_implementation='sdpa')
    return model_cls.from_pretrained(model_name,**kwargs).eval()


def _free_gpu():
    import torch
    gc.collect(); torch.cuda.empty_cache()


def qwen_reranker(frame, cache_dir, budget, model_name='Qwen/Qwen3-Reranker-4B', max_length=768,
                  batch_size=4):
    """Raw yes-minus-no logits, last real token; no generation or full vocab output."""
    import torch
    from transformers import AutoModelForCausalLM,AutoTokenizer
    revision=resolve_revision(model_name,cache_dir)
    tok=AutoTokenizer.from_pretrained(model_name,revision=revision,padding_side='right')
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    prefix=tok.encode(RR_PREFIX,add_special_tokens=False); suffix=tok.encode(RR_SUFFIX,add_special_tokens=False)
    inputs=[f'<Instruct>: {TASK}\n<Query>: {q}\n<Document>: '+(t+'. ' if t else '')+x
            for q,t,x in zip(frame.query_c,frame.title_c,frame.text_c)]
    body=tok(inputs,add_special_tokens=False,truncation=True,
             max_length=max_length-len(prefix)-len(suffix),return_attention_mask=False)['input_ids']
    seqs=[prefix+x+suffix for x in body]
    config=dict(model=model_name,revision=revision,max_length=max_length,prompt='official_v1',pool='last_real_yes_no_v1')
    sig=_signature(frame.id,inputs,config)
    path=Path(cache_dir)/(model_name.split('/')[-1]+f'_{max_length}.npz')
    # A complete cache needs no model in VRAM.
    existing=_cached_vectors(path,frame.id,sig)
    if np.isfinite(existing).all(): return existing
    budget.check(); model=_load_transformer(model_name,AutoModelForCausalLM,revision)
    yn=[tok.convert_tokens_to_ids(t) for t in ['no','yes']]
    for t,i in zip(['no','yes'],yn):
        assert tok.encode(t,add_special_tokens=False)==[i],f'{t} is not a single token'
    weights=model.get_output_embeddings().weight[yn].detach().float()
    @torch.inference_mode()
    def predict_batch(ix):
        batch=tok.pad({'input_ids':[seqs[i] for i in ix]},padding=True,return_tensors='pt').to(model.device)
        h=model.model(**batch,use_cache=False,return_dict=True).last_hidden_state
        last=h[torch.arange(len(ix),device=h.device),batch['attention_mask'].sum(1)-1]
        logits=last.float()@weights.T
        return (logits[:,1]-logits[:,0]).cpu().numpy()
    try:
        return _run_cached(frame.id,inputs,sig,path,[len(x) for x in seqs],predict_batch,batch_size,budget,
                           benchmark=True,description=model_name.split('/')[-1])
    finally:
        del predict_batch,weights,model; _free_gpu()


def bge_zeroshot(frame,cache_dir,budget,max_length=512,batch_size=16):
    import torch
    from transformers import AutoModelForSequenceClassification,AutoTokenizer
    name='BAAI/bge-reranker-v2-m3'; revision=resolve_revision(name,cache_dir)
    tok=AutoTokenizer.from_pretrained(name,revision=revision)
    queries=frame.query_c.tolist(); docs=[(t+'. ' if t else '')+x for t,x in zip(frame.title_c,frame.text_c)]
    inputs=[q+'\n'+d for q,d in zip(queries,docs)]
    sig=_signature(frame.id,inputs,dict(model=name,revision=revision,max_length=max_length,pool='ce_raw_v1'))
    path=Path(cache_dir)/f'bge_pretrained_{max_length}.npz'
    old=_cached_vectors(path,frame.id,sig)
    if np.isfinite(old).all(): return old
    budget.check(); model=_load_transformer(name,AutoModelForSequenceClassification,revision)
    enc=tok(queries,docs,truncation='only_second',max_length=max_length,return_attention_mask=False)
    @torch.inference_mode()
    def predict_batch(ix):
        batch=tok.pad({k:[v[i] for i in ix] for k,v in enc.items()},padding=True,return_tensors='pt').to(model.device)
        return model(**batch).logits[:,0].float().cpu().numpy()
    try:
        return _run_cached(frame.id,inputs,sig,path,[len(x) for x in enc['input_ids']],predict_batch,batch_size,
                           budget,benchmark=True,description='BGE pretrained')
    finally:
        del predict_batch,model; _free_gpu()


def qwen_embeddings(frame,cache_dir,budget,max_length=768,batch_size=16):
    """Correct last-token pooling; retain vectors for learned similarity experiments."""
    import pandas as pd
    import torch
    from transformers import AutoModel,AutoTokenizer
    name='Qwen/Qwen3-Embedding-0.6B'; revision=resolve_revision(name,cache_dir)
    tok=AutoTokenizer.from_pretrained(name,revision=revision,padding_side='right')
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    # Deduplicate exact cleaned text, not observed document_id, to preserve distinct variants.
    queries=['Instruct: '+TASK+'\nQuery: '+q for q in frame.query_c]
    docs=[(t+'. ' if t else '')+x for t,x in zip(frame.title_c,frame.text_c)]
    unique=list(dict.fromkeys(queries+docs)); lookup={x:i for i,x in enumerate(unique)}
    ids=[hashlib.sha256(x.encode()).hexdigest() for x in unique]
    config=dict(model=name,revision=revision,max_length=max_length,pool='last_real_no_manual_eos_v1')
    sig=_signature(ids,unique,config); path=Path(cache_dir)/f'qwen_embeddings_correct_{max_length}.npz'
    width=1024
    old=_cached_vectors(path,ids,sig,width)
    values=None
    if np.isfinite(old).all(): values=old
    else:
        budget.check(); model=_load_transformer(name,AutoModel,revision)
        enc=tok(unique,truncation=True,max_length=max_length,return_attention_mask=False)
        @torch.inference_mode()
        def predict_batch(ix):
            batch=tok.pad({k:[v[i] for i in ix] for k,v in enc.items()},padding=True,return_tensors='pt').to(model.device)
            h=model(**batch,use_cache=False,return_dict=True).last_hidden_state
            last=h[torch.arange(len(ix),device=h.device),batch['attention_mask'].sum(1)-1].float()
            return torch.nn.functional.normalize(last,dim=-1).cpu().numpy()
        try:
            values=_run_cached(ids,unique,sig,path,[len(x) for x in enc['input_ids']],predict_batch,batch_size,budget,
                               width=width,benchmark=True,description='Qwen embeddings correct')
        finally:
            del predict_batch,model; _free_gpu()
    if values is None: return None
    scores=np.empty(len(frame),np.float32)
    for start in range(0,len(frame),1024):
        sl=slice(start,start+1024)
        q=values[[lookup[x] for x in queries[sl]]]; d=values[[lookup[x] for x in docs[sl]]]
        scores[sl]=(q*d).sum(1)
    return scores


def add_neural_scores(tr,te,scores,cache_dir,budget,run_rr4=True,run_bge=True,run_emb=True):
    """4B first. Fall back to 0.6B if throughput makes a full 4B run infeasible."""
    import pandas as pd
    import traceback
    import torch
    if not torch.cuda.is_available():
        print('GPU отсутствует: используются готовые кэши и обучаемый табличный ранкер.'); return scores
    full=pd.concat([tr,te],ignore_index=True)
    jobs=[]
    if run_rr4: jobs.append(('rr4',lambda:qwen_reranker(full,cache_dir,budget)))
    else: jobs.append(('rr06',lambda:qwen_reranker(full,cache_dir,budget,model_name='Qwen/Qwen3-Reranker-0.6B',batch_size=16)))
    if run_bge: jobs.append(('bge_pretrained',lambda:bge_zeroshot(full,cache_dir,budget)))
    if run_emb: jobs.append(('qwen_embed_correct',lambda:qwen_embeddings(full,cache_dir,budget)))
    for name,fn in jobs:
        if budget.for_neural()<120:
            print('Бюджет GPU завершён.'); break
        try:
            values=fn()
        except (StageBudget,RuntimeError,ValueError,FloatingPointError,OSError) as exc:
            print(f'Этап {name} не завершён: {type(exc).__name__}: {exc}')
            traceback.print_exc(limit=2); values=None; _free_gpu()
        if values is not None:
            assert len(values)==len(full) and np.isfinite(values).all()
            scores[name]=(values[:len(tr)].copy(),values[len(tr):].copy())
            print(name,'добавлен, осталось',round(budget.remaining()/60),'мин',flush=True)
        elif name=='rr4' and budget.for_neural()>900:
            try:
                values=qwen_reranker(full,cache_dir,budget,model_name='Qwen/Qwen3-Reranker-0.6B',batch_size=16)
                if values is not None: scores['rr06']=(values[:len(tr)].copy(),values[len(tr):].copy())
            except (StageBudget,RuntimeError,ValueError,FloatingPointError,OSError) as exc:
                print('rr06 тоже не завершён:',exc); _free_gpu()
    return scores


In [ ]:
%%writefile jina_joint.py
"""Pinned Jina3.5 joint reranking with resumable complete-query checkpoints.

No labels are read. All candidates of every query remain in the model context.
Only complete orders are returned; original v4 model caches are never touched.
"""
from __future__ import annotations

import gc
import hashlib
import importlib
import json
import os
from pathlib import Path
import tempfile
import time
import uuid

import numpy as np

MODEL_NAME = "jinaai/jina-reranker-v3.5"
MODEL_REVISION = "e8a93f33f0b22108f8c2364f8484ce3422552fbc"
VECTOR_WIDTH = 512
FORMAT_VERSION = 1


class JointBudget(Exception):
    """Stop before spending the reserve intended for the supervised stage."""


def _available(budget):
    if hasattr(budget, "for_neural"):
        return float(budget.for_neural())
    if hasattr(budget, "remaining"):
        return max(0., float(budget.remaining()) - 900.)
    raise TypeError("Budget must expose for_neural() or remaining()")


def _check_budget(budget):
    if hasattr(budget, "check"):
        try:
            budget.check()
        except Exception as error:
            if type(error).__name__ in {"StageBudget", "JointBudget"}:
                raise JointBudget(str(error)) from error
            raise
    if _available(budget) <= 10:
        raise JointBudget("Reserved time for final supervised fitting and submission")


def _atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent,
                                         prefix=path.name + ".", delete=False) as handle:
            temporary = Path(handle.name)
            json.dump(value, handle, ensure_ascii=False, indent=2, allow_nan=False)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


def _atomic_npz(path, **arrays):
    path = Path(path)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile("wb", dir=path.parent, prefix=path.name + ".", delete=False) as handle:
            temporary = Path(handle.name)
            np.savez_compressed(handle, **arrays)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


def _input_digest(frame):
    h = hashlib.sha256()
    required = ["id", "query_id", "query_c", "title_c", "text_c"]
    for row in frame.sort_values("id")[required].itertuples(index=False, name=None):
        for value in row:
            b = str(value).encode("utf-8")
            h.update(len(b).to_bytes(8, "little"))
            h.update(b)
    return h.hexdigest()


def _make_state(frame, groups, order, cache_dir, config, data_digest, store_vectors):
    config = {**config, "order": order, "store_vectors": bool(store_vectors)}
    signature = hashlib.sha256((data_digest + json.dumps(config, sort_keys=True)).encode()).hexdigest()
    directory = Path(cache_dir) / ("jina35_" + order)
    directory.mkdir(parents=True, exist_ok=True)
    manifest_path = directory / "manifest.json"
    query_keys = list(groups)
    query_index = {q: i for i, q in enumerate(query_keys)}
    id_index = {str(value): i for i, value in enumerate(frame.id)}
    row_query = np.empty(len(frame), dtype=np.int64)
    for q, rows in groups.items():
        row_query[rows] = query_index[q]
    ids = frame.id.astype(str).to_numpy(dtype=str)
    row_order = {}
    for q, rows in groups.items():
        ordered = rows[np.argsort(ids[rows], kind="stable")]
        row_order[q] = ordered if order == "id" else ordered[::-1]
    state = dict(directory=directory, manifest_path=manifest_path, signature=signature,
                 config=config, groups=groups, query_index=query_index, row_query=row_query,
                 row_order=row_order, ids=ids, done=set(), pending=[], chunks=[], caps={},
                 scores=np.full(len(frame), np.nan, dtype=np.float32), store_vectors=store_vectors)
    if store_vectors:
        state["document"] = np.full((len(frame), VECTOR_WIDTH), np.nan, dtype=np.float32)
        state["query"] = np.full((len(groups), VECTOR_WIDTH), np.nan, dtype=np.float32)
    if not manifest_path.is_file():
        return state
    manifest = json.loads(manifest_path.read_text())
    if (manifest.get("format_version") != FORMAT_VERSION or manifest.get("signature") != signature
            or manifest.get("config") != config or manifest.get("data_digest") != data_digest):
        raise ValueError(f"Incompatible Jina cache at {directory}; choose a new cache directory")
    for filename in manifest["chunks"]:
        if Path(filename).name != filename:
            raise ValueError("Invalid chunk path in Jina manifest")
        with np.load(directory / filename, allow_pickle=False) as cached:
            if str(cached["signature"].item()) != signature:
                raise ValueError("Jina chunk signature mismatch")
            row_ids = cached["row_ids"].astype(str)
            group_ids = cached["query_ids"].astype(str)
            offsets = cached["offsets"].astype(np.int64)
            caps = cached["caps"].astype(np.int64)
            values = cached["scores"].astype(np.float32)
            if (offsets.shape != (len(group_ids) + 1,) or offsets[0] != 0 or offsets[-1] != len(row_ids)
                    or np.any(np.diff(offsets) <= 0) or values.shape != (len(row_ids),)
                    or caps.shape != (len(group_ids),) or not np.isfinite(values).all()):
                raise ValueError("Malformed or incomplete Jina query chunk")
            if store_vectors:
                docs = cached["document"].astype(np.float32)
                queries = cached["query"].astype(np.float32)
                if (docs.shape != (len(row_ids), VECTOR_WIDTH) or queries.shape != (len(group_ids), VECTOR_WIDTH)
                        or not np.isfinite(docs).all() or not np.isfinite(queries).all()):
                    raise ValueError("Malformed Jina vector chunk")
            for j, q in enumerate(group_ids):
                if q not in groups or q in state["done"]:
                    raise ValueError("Unknown or duplicate query in Jina cache")
                selected_ids = row_ids[offsets[j]:offsets[j + 1]]
                expected = set(ids[groups[q]])
                if len(selected_ids) != len(expected) or set(selected_ids) != expected:
                    raise ValueError("Jina chunk must contain every candidate of each query exactly once")
                rows = np.asarray([id_index[value] for value in selected_ids])
                state["scores"][rows] = values[offsets[j]:offsets[j + 1]]
                if store_vectors:
                    state["document"][rows] = docs[offsets[j]:offsets[j + 1]]
                    state["query"][query_index[q]] = queries[j]
                state["done"].add(q)
                state["caps"][q] = int(caps[j])
        state["chunks"].append(filename)
    if int(manifest.get("complete_queries", -1)) != len(state["done"]):
        raise ValueError("Jina manifest query count does not match committed chunks")
    print(f"Jina {order}: resumed {len(state['done'])}/{len(groups)} complete queries", flush=True)
    return state


def _checkpoint(state, data_digest):
    if not state["pending"]:
        return
    pending = state["pending"].copy()
    rows = np.concatenate([state["row_order"][q] for q in pending])
    offsets = np.r_[0, np.cumsum([len(state["row_order"][q]) for q in pending])]
    filename = f"chunk_{uuid.uuid4().hex}.npz"
    arrays = dict(signature=np.asarray(state["signature"]), row_ids=state["ids"][rows],
                  query_ids=np.asarray(pending, dtype=str), offsets=offsets.astype(np.int64),
                  caps=np.asarray([state["caps"][q] for q in pending], dtype=np.int32),
                  scores=state["scores"][rows])
    if state["store_vectors"]:
        arrays["document"] = state["document"][rows]
        arrays["query"] = state["query"][[state["query_index"][q] for q in pending]]
    _atomic_npz(state["directory"] / filename, **arrays)
    chunks = state["chunks"] + [filename]
    manifest = dict(format_version=FORMAT_VERSION, signature=state["signature"],
                    data_digest=data_digest, config=state["config"], chunks=chunks,
                    complete_queries=len(state["done"]), total_queries=len(state["groups"]),
                    cap_min=int(min(state["caps"].values())), cap_max=int(max(state["caps"].values())))
    # A crash before this atomic commit leaves an ignored orphan, not half a query.
    _atomic_json(state["manifest_path"], manifest)
    state["chunks"] = chunks
    state["pending"].clear()


def _load_components(model_name, revision, device):
    import torch
    from transformers import AutoModel, AutoTokenizer
    dtype = torch.float32 if str(device).startswith("cpu") else torch.float16
    tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision, padding_side="left")
    model = AutoModel.from_pretrained(model_name, revision=revision, code_revision=revision,
                                      trust_remote_code=True, torch_dtype=dtype,
                                      low_cpu_mem_usage=True, device_map={"": str(device)},
                                      attn_implementation="sdpa").eval()
    module = importlib.import_module(model.__class__.__module__)
    return model, tokenizer, module.format_docs_prompts_func


def _run_query(model, tokenizer, formatter, query, documents, max_doc_tokens=1024,
               max_group_tokens=8192, min_doc_tokens=64, max_query_tokens=256):
    """Project only final hidden states; return normalized vectors and effective cap."""
    import torch
    tokens = model.special_tokens
    # Sanitize before truncation so inserted markers are the only marker tokens.
    for marker in tokens.values():
        query = query.replace(marker, "")
        documents = [document.replace(marker, "") for document in documents]
    query_ids = tokenizer.encode(query, add_special_tokens=False)[:max_query_tokens]
    query = tokenizer.decode(query_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    document_ids = [tokenizer.encode(document, add_special_tokens=False) for document in documents]
    cap = int(max_doc_tokens)
    device = next(model.parameters()).device
    while True:
        docs = [tokenizer.decode(value[:cap], skip_special_tokens=True, clean_up_tokenization_spaces=False)
                for value in document_ids]
        prompt = formatter(query, docs, instruction=None, special_tokens=tokens, no_thinking=True)
        encoded = tokenizer(prompt, return_tensors="pt", return_attention_mask=True)
        length = int(encoded["input_ids"].shape[1])
        if length > max_group_tokens:
            proposed = max(min_doc_tokens, min(cap - 1, int(cap * max_group_tokens / length * .95)))
            if proposed >= cap or cap <= min_doc_tokens:
                raise ValueError("All candidate markers cannot fit the configured group context")
            cap = proposed
            continue
        ids = encoded["input_ids"]
        if (int(ids.eq(model.doc_embed_token_id).sum()) != len(documents)
                or int(ids.eq(model.query_embed_token_id).sum()) != 1):
            raise ValueError("Joint prompt lost or duplicated a document/query marker")
        try:
            batch = {key: value.to(device) for key, value in encoded.items()}
            with torch.inference_mode():
                hidden = model.model(**batch, use_cache=False, output_hidden_states=False,
                                     return_dict=True).last_hidden_state
                doc_hidden = hidden[batch["input_ids"].eq(model.doc_embed_token_id)]
                query_hidden = hidden[batch["input_ids"].eq(model.query_embed_token_id)]
                docs_projected = model.projector(doc_hidden).float()
                query_projected = model.projector(query_hidden).float()
                docs_projected = torch.nn.functional.normalize(docs_projected, dim=-1)
                query_projected = torch.nn.functional.normalize(query_projected, dim=-1)
                scores = (docs_projected * query_projected).sum(-1)
                result = (scores.cpu().numpy().astype(np.float32),
                          docs_projected.cpu().numpy().astype(np.float32),
                          query_projected[0].cpu().numpy().astype(np.float32), cap)
            if (result[0].shape != (len(documents),) or result[1].shape != (len(documents), VECTOR_WIDTH)
                    or result[2].shape != (VECTOR_WIDTH,) or not all(np.isfinite(x).all() for x in result[:3])):
                raise RuntimeError("Jina produced nonfinite or malformed scores/vectors (FP16 on T4 may be unstable). "
                                   "This query is not committed; completed queries remain resumable. "
                                   "Retain previous model scores and run the supervised stage without this incomplete order.")
            return result
        except torch.cuda.OutOfMemoryError:
            # Release current forward activations before retrying every document
            # at a shorter cap. No candidate is dropped or separately encoded.
            hidden = doc_hidden = query_hidden = docs_projected = query_projected = scores = batch = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if cap <= min_doc_tokens:
                raise
            cap = max(min_doc_tokens, int(cap * .75))
            print(f"Jina query OOM: retry all {len(documents)} candidates, doc cap={cap}", flush=True)


def _synchronize(device):
    import torch
    if str(device).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


def run_jina_joint(frame, cache_dir, budget, orders=("id", "reverse"), return_embeddings=True,
                   revision=MODEL_REVISION, max_doc_tokens=1024, max_group_tokens=8192,
                   min_doc_tokens=64, max_query_tokens=256, device="cuda"):
    """Return (scores_dict, vectors_dict), using no relevance labels.

    scores_dict contains full row-aligned arrays ``jina35_id`` and/or
    ``jina35_reverse`` only when the whole corresponding order has completed.
    vectors_dict is empty unless order id has completed and return_embeddings
    is true. Then ``document`` and ``query`` are normalized float32[n,512]
    arrays in input row order (a query vector is repeated for its candidates).

    Budget must expose for_neural(), or remaining() with a 900-second reserve.
    A complete cache is reusable without a GPU, tokenizer, or model download.
    """
    import pandas as pd
    required = {"id", "query_id", "query_c", "title_c", "text_c"}
    if not required.issubset(frame.columns) or frame.id.isna().any() or not frame.id.is_unique:
        raise ValueError("Jina needs unique IDs and canonical query/title/text columns")
    if frame.query_id.isna().any() or not len(frame):
        raise ValueError("Jina needs at least one complete query")
    if not set(orders).issubset({"id", "reverse"}) or len(orders) != len(set(orders)):
        raise ValueError("Orders must be distinct members of ('id','reverse')")
    if not (0 < min_doc_tokens <= max_doc_tokens and max_group_tokens > 0 and max_query_tokens > 0):
        raise ValueError("Invalid context limits")
    frame = frame.reset_index(drop=True).copy()
    for column in ("query_c", "title_c", "text_c"):
        frame[column] = frame[column].fillna("").astype(str)
    frame["query_id"] = frame.query_id.astype(str)
    groups = {str(q): np.asarray(rows, dtype=np.int64)
              for q, rows in frame.groupby("query_id", sort=True).indices.items()}
    for rows in groups.values():
        if frame.iloc[rows].query_c.nunique() != 1:
            raise ValueError("One query_id has inconsistent canonical query text")
    data_digest = _input_digest(frame)
    config = dict(model=MODEL_NAME, revision=revision, max_doc_tokens=int(max_doc_tokens),
                  min_doc_tokens=int(min_doc_tokens), max_group_tokens=int(max_group_tokens),
                  max_query_tokens=int(max_query_tokens), pooling="projected-f32-normalized-v1",
                  prompt="official-format_docs_prompts_func", attn_implementation="sdpa",
                  dtype="float32" if str(device).startswith("cpu") else "float16")
    states = {order: _make_state(frame, groups, order, cache_dir, config, data_digest,
                                 bool(return_embeddings and order == "id")) for order in orders}
    scores, vectors = {}, {}

    def collect_complete():
        for order, state in states.items():
            if len(state["done"]) == len(groups):
                if not np.isfinite(state["scores"]).all():
                    raise ValueError("Completed Jina cache has missing rows")
                scores["jina35_" + order] = state["scores"].copy()
                if state["store_vectors"]:
                    vectors["document"] = state["document"]
                    vectors["query"] = state["query"][state["row_query"]]

    collect_complete()
    if len(scores) == len(orders):
        return scores, vectors
    model = tokenizer = formatter = None
    try:
        _check_budget(budget)
        model, tokenizer, formatter = _load_components(MODEL_NAME, revision, device)
        for order in orders:
            state = states[order]
            if len(state["done"]) == len(groups):
                continue
            _check_budget(budget)
            pending_since = time.monotonic()
            processed_since = 0

            def run_one(q):
                nonlocal pending_since, processed_since
                if q in state["done"]:
                    return
                _check_budget(budget)
                rows = state["row_order"][q]
                selected = frame.iloc[rows]
                documents = [(title + ". " if title else "") + text
                             for title, text in zip(selected.title_c, selected.text_c)]
                values, docs, query, cap = _run_query(model, tokenizer, formatter,
                    selected.query_c.iloc[0], documents, max_doc_tokens=max_doc_tokens,
                    max_group_tokens=max_group_tokens, min_doc_tokens=min_doc_tokens,
                    max_query_tokens=max_query_tokens)
                state["scores"][rows] = values
                if state["store_vectors"]:
                    state["document"][rows] = docs
                    state["query"][state["query_index"][q]] = query
                state["caps"][q] = int(cap)
                state["done"].add(q)
                state["pending"].append(q)
                processed_since += 1
                if processed_since >= 64 or time.monotonic() - pending_since >= 30:
                    _checkpoint(state, data_digest)
                    print(f"Jina {order}: {len(state['done'])}/{len(groups)} queries", flush=True)
                    pending_since = time.monotonic()
                    processed_since = 0

            remaining = [q for q in groups if q not in state["done"]]
            # Cover the observed distribution of group/context sizes, including
            # the longest queries, rather than timing only short easy batches.
            lengths = [sum(min(len(frame.text_c.iloc[i]) + len(frame.title_c.iloc[i]), max_doc_tokens * 5)
                           for i in groups[q]) + len(frame.query_c.iloc[groups[q][0]]) for q in remaining]
            ordered = [remaining[i] for i in np.argsort(lengths, kind="stable")]
            warmup = ordered[len(ordered) // 2]
            run_one(warmup)
            probe = [ordered[i] for i in np.unique(np.linspace(0, len(ordered) - 1, min(24, len(ordered))).astype(int))
                     if ordered[i] not in state["done"]]
            _synchronize(device)
            started = time.monotonic()
            for q in probe:
                run_one(q)
            _synchronize(device)
            elapsed = time.monotonic() - started
            _checkpoint(state, data_digest)
            missing_count = len(groups) - len(state["done"])
            rate = len(probe) / max(elapsed, 1e-6) if probe else 1.
            estimate = 1.35 * missing_count / max(rate, 1e-9)
            print(f"Jina {order}: {rate:.3f} queries/s; remaining estimate {estimate / 60:.1f} min, "
                  f"available {_available(budget) / 60:.1f} min", flush=True)
            if estimate > _available(budget):
                print(f"Jina {order}: stage does not fit; partial complete-query cache saved", flush=True)
                break
            for q in groups:
                if q not in state["done"]:
                    run_one(q)
            _checkpoint(state, data_digest)
            print(f"Jina {order}: complete; effective doc caps "
                  f"{min(state['caps'].values())}..{max(state['caps'].values())}", flush=True)
            collect_complete()
    except JointBudget:
        for state in states.values():
            _checkpoint(state, data_digest)
        print("Jina: stopped at budget; only fully complete orders are returned", flush=True)
    except BaseException:
        for state in states.values():
            _checkpoint(state, data_digest)
        raise
    finally:
        del model, tokenizer, formatter
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except ImportError:
            pass
    collect_complete()
    return scores, vectors


In [ ]:
%%writefile jina_probe.py
"""A fixed linear relevance metric over frozen Jina query/document states.

This trains a linear probe, not the neural Jina backbone. No external data or
hyperparameter search is used. The returned training scores are group-held-out
OOF scores. Downstream ordinary stack CV with this learned feature is still
approximate: a probe used for another row's OOF feature can have seen the
stack-validation labels. Declare this score's kind as ``trained``.
"""
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


_CONFIG = dict(schema="jina-linear-probe-v1", dimension=512,
               input_normalization="row_l2", features=["q_times_d", "abs_q_minus_d", "cosine"],
               standardization="fit_on_training_fold_only", C=.05, solver="lbfgs",
               max_iter=500, tol=1e-4, class_weight=None, neural_backbone_finetuned=False)


def _json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def _array_digest(values):
    a = np.ascontiguousarray(values)
    h = hashlib.sha256()
    h.update(_json({"shape": list(a.shape), "dtype": a.dtype.str}).encode())
    view = memoryview(a).cast("B")
    for start in range(0, len(view), 4 * 1024 * 1024):
        h.update(view[start:start + 4 * 1024 * 1024])
    return h.hexdigest()


def _strings_digest(values):
    h = hashlib.sha256()
    for v in values:
        b = str(v).encode("utf-8")
        h.update(len(b).to_bytes(8, "little"))
        h.update(b)
    return h.hexdigest()


def _atomic_json(path, value):
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)


def _atomic_npz(path, **arrays):
    tmp = Path(str(path) + ".tmp")
    with tmp.open("wb") as f:
        np.savez_compressed(f, **arrays)
    os.replace(tmp, path)


def _embeddings(value, nrows, name):
    a = np.asarray(value)
    if a.shape != (nrows, _CONFIG["dimension"]) or not np.issubdtype(a.dtype, np.floating):
        raise ValueError(f"{name}: expected a floating ({nrows},512) array; got {a.shape}, {a.dtype}")
    if not np.isfinite(a).all():
        raise ValueError(f"{name}: embeddings contain NaN/infinity or incomplete cached rows")
    lengths = np.sqrt(np.einsum("ij,ij->i", a, a, dtype=np.float64))
    if np.any(lengths < 1e-10):
        raise ValueError(f"{name}: zero embedding vector cannot be normalized")
    # Hash exactly the supplied array's shape, dtype and contents before casting.
    digest = _array_digest(a)
    normalized = (a / lengths[:, None]).astype(np.float32)
    return normalized, digest


def _features(q, d):
    product = q * d
    f = np.concatenate((product, np.abs(q - d), product.sum(axis=1, keepdims=True)), axis=1)
    if not np.isfinite(f).all():
        raise FloatingPointError("Nonfinite linear-probe features")
    return f.astype(np.float32, copy=False)


def _ndcg(frame, prediction):
    p = np.asarray(prediction, dtype=float)
    if p.shape != (len(frame),) or not np.isfinite(p).all():
        raise ValueError("Metric requires one finite prediction for every pair")
    ids = frame.id.astype(str).to_numpy()
    gain = np.exp2(frame.relevance.to_numpy(dtype=float)) - 1
    discounts = 1 / np.log2(np.arange(2, 12))
    scores = []
    for ix in frame.groupby("query_id", sort=False).indices.values():
        ranked = np.lexsort((ids[ix], -p[ix]))[:10]
        ideal = np.sort(gain[ix])[::-1][:10]
        idcg = float(ideal @ discounts[:len(ideal)])
        dcg = float(gain[ix][ranked] @ discounts[:len(ranked)])
        scores.append(dcg / idcg if idcg else 0.)
    return float(np.mean(scores))


def _fit(X, y, label):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(C=_CONFIG["C"], solver=_CONFIG["solver"],
             max_iter=_CONFIG["max_iter"], tol=_CONFIG["tol"], class_weight=None,
             random_state=42)),
    ])
    # Thread limits prevent five CPU fits from spawning a large BLAS thread pool.
    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)
    convergence = [str(w.message) for w in caught if issubclass(w.category, ConvergenceWarning)]
    # Re-emit unrelated warnings instead of silently swallowing them.
    for w in caught:
        if not issubclass(w.category, ConvergenceWarning):
            warnings.warn(str(w.message), w.category, stacklevel=2)
    if convergence:
        raise RuntimeError(f"{label}: fixed max_iter=500 probe did not converge. "
                           "No predictions from this fit were cached. " + "; ".join(convergence))
    lr = model.named_steps["logistic"]
    if not np.isfinite(lr.coef_).all() or not np.isfinite(lr.intercept_).all():
        raise FloatingPointError(f"{label}: nonfinite learned parameters")
    return model, int(np.max(lr.n_iter_))


def _decision(model, X):
    with threadpool_limits(limits=2):
        result = np.asarray(model.decision_function(X), dtype=np.float64)
    if result.shape != (len(X),) or not np.isfinite(result).all():
        raise FloatingPointError("Linear probe returned malformed/nonfinite scores")
    return result


def _cached_prediction(path, model_path, signature, ids):
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as z:
        if str(z["signature"].item()) != signature or not np.array_equal(z["ids"].astype(str), ids):
            raise ValueError(f"Incompatible probe cache {path}; use a new output directory")
        p = z["prediction"].copy()
        n_iter = int(z["n_iter"].item())
    if p.shape != (len(ids),) or not np.isfinite(p).all():
        raise ValueError(f"Malformed probe prediction cache {path}")
    # Prediction caching never unpickles a model. Refit if its reviewable artifact
    # was removed; otherwise source features and predictions identify this fit.
    return (p, n_iter) if model_path.is_file() else None


def fit_jina_probe(tr, te, folds, qtr, dtr, qte, dte, outdir):
    """Return (OOF logits, test logits, diagnostics), preserving input row order.

    Each state array must contain one finite, nonzero 512D row per pair. Query
    states may be repeated within a query but must come from the same joint
    inference as that pair's document state. The caller supplies grouped folds.
    C=.05 is fixed in advance; no audit labels select C or other settings.
    """
    started = time.monotonic()
    tr, te = tr.reset_index(drop=True).copy(), te.reset_index(drop=True).copy()
    for label, frame in (("train", tr), ("test", te)):
        if not {"id", "query_id"} <= set(frame) or frame.id.isna().any() or not frame.id.is_unique:
            raise ValueError(f"{label}: present unique ids and query_id are required")
        if frame.query_id.isna().any():
            raise ValueError(f"{label}: missing query_id")
    if set(tr.query_id) & set(te.query_id):
        raise ValueError("Train and test queries overlap")
    if "relevance" not in tr or not tr.relevance.isin([0, 1]).all() or tr.relevance.nunique() != 2:
        raise ValueError("Training relevance must contain both binary classes")
    y = tr.relevance.to_numpy(dtype=np.int8)
    folds = [(np.asarray(a, dtype=np.int64), np.asarray(b, dtype=np.int64)) for a, b in folds]
    if len(folds) < 2:
        raise ValueError("At least two grouped folds are required")
    val = np.concatenate([b for _, b in folds])
    if len(val) != len(tr) or not np.array_equal(np.sort(val), np.arange(len(tr))):
        raise ValueError("Validation fold indices must partition all training pairs once")
    for k, (a, b) in enumerate(folds):
        if len(a) == 0 or len(b) == 0 or np.any(a < 0) or np.any(a >= len(tr)) or len(np.unique(a)) != len(a):
            raise ValueError(f"Invalid training indices in fold {k}")
        if set(tr.iloc[a].query_id) & set(tr.iloc[b].query_id):
            raise ValueError(f"Queries cross training/validation in fold {k}")
        if len(np.unique(y[a])) != 2:
            raise ValueError(f"Fold {k} training labels have only one class")
    arrays, digests = [], {}
    for name, array, size in (("qtr", qtr, len(tr)), ("dtr", dtr, len(tr)),
                              ("qte", qte, len(te)), ("dte", dte, len(te))):
        normalized, digest = _embeddings(array, size, name)
        arrays.append(normalized)
        digests[name] = digest
    qtr_n, dtr_n, qte_n, dte_n = arrays
    X, Xt = _features(qtr_n, dtr_n), _features(qte_n, dte_n)
    config = dict(_CONFIG)
    config["versions"] = {name: importlib.metadata.version(name) for name in ("numpy", "scipy", "scikit-learn", "joblib")}
    binding = dict(config=config, embedding_digests=digests,
                   train_ids=_strings_digest(tr.id), test_ids=_strings_digest(te.id),
                   train_queries=_strings_digest(tr.query_id), test_queries=_strings_digest(te.query_id),
                   labels=_array_digest(y),
                   folds=[dict(train=_array_digest(a), validation=_array_digest(b)) for a, b in folds])
    signature = hashlib.sha256(_json(binding).encode()).hexdigest()
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    manifest_path = out / "probe_manifest.json"
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text(encoding="utf-8"))
        if previous.get("signature") != signature:
            raise ValueError("Probe inputs/config/versions changed; use a new output directory")
    _atomic_json(manifest_path, dict(signature=signature, **binding))
    warnings.warn("Jina linear probe is supervised on frozen embeddings. Declare kind='trained'; "
                  "ordinary downstream stacking CV with its OOF features is approximate, not nested end-to-end CV.",
                  UserWarning, stacklevel=2)
    oof = np.full(len(tr), np.nan, dtype=np.float64)
    details = []
    for k, (a, b) in enumerate(folds):
        step_start = time.monotonic()
        ids = tr.iloc[b].id.to_numpy(dtype=str)
        path, model_path = out / f"probe_fold_{k}.npz", out / f"probe_fold_{k}.joblib"
        cached = _cached_prediction(path, model_path, signature, ids)
        reused = cached is not None
        if reused:
            pred, n_iter = cached
        else:
            model, n_iter = _fit(X[a], y[a], f"probe fold {k}")
            pred = _decision(model, X[b])
            tmp = Path(str(model_path) + ".tmp")
            joblib.dump(model, tmp, compress=3)
            os.replace(tmp, model_path)
            _atomic_npz(path, ids=ids, prediction=pred, signature=np.asarray(signature), n_iter=np.asarray(n_iter))
            del model
        oof[b] = pred
        detail = dict(fold=k, n_train=len(a), n_validation=len(b), n_iter=n_iter,
                      cached=reused, ndcg=_ndcg(tr.iloc[b].reset_index(drop=True), pred),
                      seconds=time.monotonic() - step_start)
        details.append(detail)
        print(f"Jina probe fold{k}: NDCG@10={detail['ndcg']:.6f}; "
              f"iterations={n_iter}; {'cache' if reused else 'fitted'}", flush=True)
    ids = te.id.to_numpy(dtype=str)
    final_cache, final_model_path = out / "probe_final_test.npz", out / "probe_final.joblib"
    cached = _cached_prediction(final_cache, final_model_path, signature, ids)
    if cached is None:
        model, n_iter = _fit(X, y, "probe full train")
        test = _decision(model, Xt)
        tmp = Path(str(final_model_path) + ".tmp")
        joblib.dump(model, tmp, compress=3)
        os.replace(tmp, final_model_path)
        _atomic_npz(final_cache, ids=ids, prediction=test, signature=np.asarray(signature), n_iter=np.asarray(n_iter))
        del model
    else:
        test, n_iter = cached
    if not np.isfinite(oof).all():
        raise RuntimeError("Incomplete OOF probe predictions")
    diag = dict(kind="trained", scope="linear relevance probe over frozen Jina states; neural backbone unchanged",
                config=config, signature=signature, embedding_digests=digests,
                n_features=int(X.shape[1]), folds=details, final_n_iter=n_iter,
                oof_ndcg=_ndcg(tr, oof), frozen_cosine_ndcg=_ndcg(tr, (qtr_n * dtr_n).sum(axis=1)),
                seconds=time.monotonic() - started,
                validation_caveat="Probe OOF excludes its validation query groups. A downstream stack using these scores needs nested CV for unbiased end-to-end evaluation.")
    _atomic_npz(out / "probe_scores.npz", train_ids=tr.id.to_numpy(dtype=str), test_ids=ids,
                oof=oof, test=test, signature=np.asarray(signature))
    _atomic_json(out / "probe_metrics.json", diag)
    return oof, test, diag


In [ ]:
%%writefile jina_rank_adapter.py
"""Fixed small shared residual metric over frozen 512D Jina states.

The Jina neural backbone is unchanged. This optional adapter learns exclusively
from competition labels, with group-held-out scores for every training row.
Ordinary downstream stacking CV with these learned OOF features is approximate;
use kind='trained'. An interrupted/time-limited run never returns partial OOF.
"""
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from threadpoolctl import threadpool_limits

from jina_probe import _array_digest, _atomic_json, _atomic_npz, _embeddings, _ndcg, _strings_digest


class AdapterBudget(RuntimeError):
    """Optional adapter exceeded its explicit time allowance; reuse other scores."""


CONFIG = dict(schema="jina-residual-rank-adapter-v1", dimension=512, hidden=64,
              dropout=.1, epochs=4, query_batch_size=32, learning_rate=3e-4,
              weight_decay=.05, scale=20., positive_prior=.26,
              bce_weight=.1, residual_l2=.01, seed=42, dtype="float32",
              loss="query-mean RankNet weighted by normalized absolute delta NDCG@10",
              backbone_finetuned=False)


def _check_time(deadline):
    if time.monotonic() >= deadline:
        raise AdapterBudget("Optional Jina rank adapter reached max_seconds; "
                            "completed folds remain cached, incomplete OOF must not enter the final stack.")


class ResidualMetric(nn.Module):
    def __init__(self):
        super().__init__()
        self.project = nn.Sequential(nn.Linear(512, 64, bias=False), nn.GELU(),
                                     nn.Dropout(.1), nn.Linear(64, 512, bias=False))
        nn.init.zeros_(self.project[-1].weight)

    def adapt(self, vectors):
        residual = self.project(vectors)
        return F.normalize(vectors + residual, p=2, dim=-1, eps=1e-8), residual

    def forward(self, q, d):
        aq, rq = self.adapt(q)
        ad, rd = self.adapt(d)
        return (aq * ad).sum(-1), .5 * (rq.square().mean() + rd.square().mean())


def query_objective(cosines, labels, sizes):
    """All positive/negative pairs per complete query; all-one/zero groups valid."""
    offset, rank_terms, bce_terms = 0, [], []
    prior_logit = math.log(CONFIG["positive_prior"] / (1 - CONFIG["positive_prior"]))
    for n in sizes:
        z = CONFIG["scale"] * cosines[offset:offset + n].float()
        y = labels[offset:offset + n].float()
        offset += n
        if n == 0:
            continue
        centered = z - z.mean() + prior_logit
        bce_terms.append(F.binary_cross_entropy_with_logits(centered, y))
        positive, negative = torch.nonzero(y > .5).flatten(), torch.nonzero(y <= .5).flatten()
        if len(positive) == 0 or len(negative) == 0:
            rank_terms.append(z.sum() * 0.)
            continue
        with torch.no_grad():
            order = torch.argsort(z.detach(), descending=True, stable=True)
            ranks = torch.empty_like(order)
            ranks[order] = torch.arange(n, device=z.device)
            positions = torch.arange(n, device=z.device)
            discount = (positions < 10).float() / torch.log2(positions.float() + 2.)
            idcg = discount[:min(len(positive), 10)].sum().clamp_min(1e-8)
            delta = (discount[ranks[positive]][:, None] -
                     discount[ranks[negative]][None, :]).abs() / idcg
        pair_loss = F.softplus(-(z[positive, None] - z[negative][None, :]))
        rank_terms.append((pair_loss * delta).sum() / delta.sum().clamp_min(1e-8))
    if offset != len(cosines) or len(labels) != len(cosines):
        raise ValueError("Query sizes do not cover all adapter examples")
    zero = cosines.sum() * 0.
    rank = torch.stack(rank_terms).mean() if rank_terms else zero
    bce = torch.stack(bce_terms).mean() if bce_terms else zero
    return rank + CONFIG["bce_weight"] * bce


def _query_groups(frame, indices):
    indices = np.asarray(indices, dtype=np.int64)
    sub = frame.iloc[indices].reset_index(drop=True)
    return [indices[ix] for ix in sub.groupby("query_id", sort=False).indices.values()]


def _fit(q, d, labels, groups, seed, deadline, device):
    _check_time(deadline)
    torch.random.default_generator.manual_seed(seed)
    if device.type == "cuda":
        with torch.cuda.device(device):
            torch.cuda.manual_seed(seed)
    model = ResidualMetric().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"],
                                  weight_decay=CONFIG["weight_decay"])
    rng = np.random.default_rng(seed)
    history = []
    for epoch in range(CONFIG["epochs"]):
        model.train()
        order = rng.permutation(len(groups))
        total, count = 0., 0
        for start in range(0, len(order), CONFIG["query_batch_size"]):
            _check_time(deadline)
            chosen = [groups[int(j)] for j in order[start:start + CONFIG["query_batch_size"]]]
            ix = np.concatenate(chosen)
            sizes = [len(g) for g in chosen]
            qq = torch.as_tensor(q[ix], dtype=torch.float32, device=device)
            dd = torch.as_tensor(d[ix], dtype=torch.float32, device=device)
            yy = torch.as_tensor(labels[ix], dtype=torch.float32, device=device)
            optimizer.zero_grad(set_to_none=True)
            cosines, residual_square = model(qq, dd)
            loss = query_objective(cosines, yy, sizes) + CONFIG["residual_l2"] * residual_square
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite Jina adapter objective; this fold was not cached")
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
            if not torch.isfinite(grad_norm):
                raise FloatingPointError("Nonfinite Jina adapter gradient; this fold was not cached")
            optimizer.step()
            total += float(loss.detach()) * len(chosen)
            count += len(chosen)
        history.append(total / max(1, count))
    if not all(torch.isfinite(p).all() for p in model.parameters()):
        raise FloatingPointError("Nonfinite Jina adapter parameters")
    return model, history


@torch.no_grad()
def _predict(model, q, d, deadline, device):
    model.eval()
    out = np.empty(len(q), dtype=np.float32)
    for start in range(0, len(q), 2048):
        _check_time(deadline)
        stop = min(start + 2048, len(q))
        qq = torch.as_tensor(q[start:stop], dtype=torch.float32, device=device)
        dd = torch.as_tensor(d[start:stop], dtype=torch.float32, device=device)
        score, _ = model(qq, dd)
        out[start:stop] = score.cpu().numpy()
    if not np.isfinite(out).all():
        raise FloatingPointError("Adapter returned nonfinite/incomplete predictions")
    return out


def _save_checkpoint(path, model, signature, history):
    temp = Path(str(path) + ".tmp")
    torch.save(dict(state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
                    signature=signature, history=history, config=dict(CONFIG)), temp)
    os.replace(temp, path)
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _cached(path, model_path, signature, ids):
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as z:
        if str(z["signature"].item()) != signature or not np.array_equal(z["ids"].astype(str), ids):
            raise ValueError(f"Incompatible adapter cache {path}; use a new output directory")
        p = z["prediction"].copy()
        history = json.loads(str(z["history"].item()))
        expected_hash = str(z["weights_sha256"].item())
    if p.shape != (len(ids),) or not np.isfinite(p).all():
        raise ValueError(f"Malformed adapter cache {path}")
    if not model_path.is_file():
        return None
    if hashlib.sha256(model_path.read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"Adapter weights changed or are corrupt: {model_path}")
    # Only our fixed checkpoint paths inside outdir are opened. weights_only
    # accepts tensors/basic metadata and does not enable arbitrary pickle code.
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=True)
    if checkpoint.get("signature") != signature or checkpoint.get("config") != CONFIG:
        raise ValueError(f"Adapter checkpoint signature/config mismatch: {model_path}")
    for value in checkpoint["state_dict"].values():
        if not isinstance(value, torch.Tensor) or not torch.isfinite(value).all():
            raise ValueError(f"Malformed adapter checkpoint parameters: {model_path}")
    return p, history


def fit_rank_adapter(tr, te, folds, qtr, dtr, qte, dte, outdir, max_seconds=600, device="cpu"):
    """Return complete (OOF cosine scores, test scores, diagnostics), or raise.

    This trains only a 65,536-parameter shared residual projection on frozen
    states. All settings are fixed; no validation labels select epochs or loss.
    Cache identity includes raw states, rows, labels, folds, package versions,
    configuration and device. Complete folds can resume within a fresh budget.
    """
    started = time.monotonic()
    deadline = started + float(max_seconds)
    _check_time(deadline)
    device = torch.device(device)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested for adapter but unavailable; select device='cpu'")
    tr, te = tr.reset_index(drop=True).copy(), te.reset_index(drop=True).copy()
    for name, frame in (("train", tr), ("test", te)):
        if not {"id", "query_id"} <= set(frame) or frame.id.isna().any() or not frame.id.is_unique or frame.query_id.isna().any():
            raise ValueError(f"{name}: unique present pair ids and query ids are required")
    if set(tr.query_id) & set(te.query_id):
        raise ValueError("Adapter train and test queries overlap")
    if "relevance" not in tr or not tr.relevance.isin([0, 1]).all():
        raise ValueError("Adapter labels must be binary")
    y = tr.relevance.to_numpy(dtype=np.float32)
    folds = [(np.asarray(a, dtype=np.int64), np.asarray(b, dtype=np.int64)) for a, b in folds]
    if len(folds) < 2:
        raise ValueError("At least two grouped folds are required")
    validation = np.concatenate([b for _, b in folds])
    if len(validation) != len(tr) or not np.array_equal(np.sort(validation), np.arange(len(tr))):
        raise ValueError("Adapter validation folds must partition all training pairs once")
    for k, (a, b) in enumerate(folds):
        if not len(a) or not len(b) or np.any(a < 0) or np.any(a >= len(tr)) or len(np.unique(a)) != len(a):
            raise ValueError(f"Invalid adapter train indices, fold {k}")
        if set(tr.iloc[a].query_id) & set(tr.iloc[b].query_id):
            raise ValueError(f"Adapter query leaks between train/validation, fold {k}")
    values, digests = [], {}
    for name, a, n in (("qtr", qtr, len(tr)), ("dtr", dtr, len(tr)),
                        ("qte", qte, len(te)), ("dte", dte, len(te))):
        normalized, digest = _embeddings(a, n, name)
        values.append(normalized)
        digests[name] = digest
        _check_time(deadline)
    q, d, qt, dt = values
    versions = {p: importlib.metadata.version(p) for p in ("numpy", "pandas", "torch")}
    binding = dict(config=CONFIG, device=str(device), versions=versions, embedding_digests=digests,
                   train_ids=_strings_digest(tr.id), test_ids=_strings_digest(te.id),
                   train_queries=_strings_digest(tr.query_id), test_queries=_strings_digest(te.query_id),
                   labels=_array_digest(y),
                   folds=[dict(train=_array_digest(a), validation=_array_digest(b)) for a, b in folds])
    signature = hashlib.sha256(json.dumps(binding, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    out = Path(outdir)
    out.mkdir(parents=True, exist_ok=True)
    manifest = out / "rank_adapter_manifest.json"
    if manifest.exists() and json.loads(manifest.read_text()).get("signature") != signature:
        raise ValueError("Adapter inputs/config/folds/versions/device changed; use a new output directory")
    _atomic_json(manifest, dict(signature=signature, **binding))
    oof = np.full(len(tr), np.nan, dtype=np.float32)
    details = []
    prior_threads = torch.get_num_threads()
    cuda_devices = [device.index if device.index is not None else torch.cuda.current_device()] if device.type == "cuda" else []
    try:
        torch.set_num_threads(2)
        with threadpool_limits(limits=2), torch.random.fork_rng(devices=cuda_devices):
            for k, (a, b) in enumerate(folds):
                _check_time(deadline)
                before = time.monotonic()
                ids = tr.iloc[b].id.to_numpy(dtype=str)
                score_path, weight_path = out / f"rank_fold_{k}.npz", out / f"rank_fold_{k}.pt"
                cached = _cached(score_path, weight_path, signature, ids)
                if cached is None:
                    model, history = _fit(q, d, y, _query_groups(tr, a), CONFIG["seed"] + k, deadline, device)
                    pred = _predict(model, q[b], d[b], deadline, device)
                    weights_hash = _save_checkpoint(weight_path, model, signature, history)
                    _atomic_npz(score_path, ids=ids, prediction=pred, signature=np.asarray(signature),
                                history=np.asarray(json.dumps(history)), weights_sha256=np.asarray(weights_hash))
                    del model
                else:
                    pred, history = cached
                oof[b] = pred
                record = dict(fold=k, cached=cached is not None, ndcg=_ndcg(tr.iloc[b].reset_index(drop=True), pred),
                              history=history, seconds=time.monotonic() - before, n_train=len(a), n_validation=len(b))
                details.append(record)
                print(f"Jina rank adapter fold{k}: NDCG@10={record['ndcg']:.6f}; "
                      f"{'cache' if cached is not None else 'fitted'}; {record['seconds']:.1f}s", flush=True)
            _check_time(deadline)
            ids = te.id.to_numpy(dtype=str)
            score_path, weight_path = out / "rank_final_test.npz", out / "rank_final.pt"
            cached = _cached(score_path, weight_path, signature, ids)
            if cached is None:
                groups = _query_groups(tr, np.arange(len(tr)))
                model, final_history = _fit(q, d, y, groups, CONFIG["seed"] + len(folds), deadline, device)
                test = _predict(model, qt, dt, deadline, device)
                weights_hash = _save_checkpoint(weight_path, model, signature, final_history)
                _atomic_npz(score_path, ids=ids, prediction=test, signature=np.asarray(signature),
                            history=np.asarray(json.dumps(final_history)), weights_sha256=np.asarray(weights_hash))
                del model
            else:
                test, final_history = cached
    finally:
        torch.set_num_threads(prior_threads)
    if not np.isfinite(oof).all() or not np.isfinite(test).all():
        raise RuntimeError("Adapter OOF/test incomplete; do not use these scores")
    diag = dict(kind="trained", scope="shared residual metric on frozen Jina states; backbone unchanged",
                config=dict(CONFIG), signature=signature, device=str(device), versions=versions,
                oof_ndcg=_ndcg(tr, oof), frozen_cosine_ndcg=_ndcg(tr, (q * d).sum(-1)),
                folds=details, final_history=final_history, seconds=time.monotonic() - started,
                validation_caveat="Adapter OOF excludes validation query groups. Ordinary downstream stack CV using this learned OOF feature is approximate, not nested end-to-end CV.")
    _atomic_npz(out / "rank_adapter_scores.npz", train_ids=tr.id.to_numpy(dtype=str), test_ids=ids,
                oof=oof, test=test, signature=np.asarray(signature))
    _atomic_json(out / "rank_adapter_metrics.json", diag)
    return oof, test, diag


In [ ]:
%%writefile jina_lora_v7.py
"""Two-fold supervised LoRA of the proven Jina3.5 joint reranker.

No external labels/data, vocabulary logits, or neural weights in checkpoints.
Manual standard LoRA is deliberate: FP32 A/B, frozen FP16 backbone/projector,
zero B, alpha/r scaling, no merge into half-precision weights. Benchmark weights
are discarded before training. Only complete two-fold OOF/test streams return.
"""
from __future__ import annotations
import gc, hashlib, importlib.metadata, json, math, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.model_selection import GroupKFold
from safetensors.torch import save_file, load_file
from safetensors import safe_open
import jina_joint as jj
from jina_probe import _array_digest, _strings_digest

MODEL_NAME=jj.MODEL_NAME
MODEL_REVISION=jj.MODEL_REVISION
DEFAULT_CONFIG=dict(version='jina-lora-v7-1',r=8,alpha=16,dropout=.05,last_n_layers=0,
 epochs=1,accum=8,lr=1e-4,weight_decay=.01,warmup=.1,scale=20.,margin=.2,
 bce_weight=.1,distill_weight=.03,prior=.26,seed=42,
 train_doc_tokens=768,train_group_tokens=6144,eval_doc_tokens=1024,eval_group_tokens=8192,
 min_doc_tokens=64,max_query_tokens=256,benchmark_queries=32,safety=1.35,
 checkpoint_queries=32,grad_scaler_init=1024.,max_overflow_windows=5)
class LoRABudget(RuntimeError):pass

def _check(budget):
 try:jj._check_budget(budget)
 except jj.JointBudget as e:raise LoRABudget(str(e)) from e

def _sync(device):
 if str(device).startswith('cuda'):torch.cuda.synchronize()

class LoRALinear(nn.Module):
 def __init__(self,base,r=8,alpha=16,dropout=.05):
  super().__init__()
  if not isinstance(base,nn.Linear):raise TypeError('LoRA target must be nn.Linear')
  self.base=base
  for p in base.parameters():p.requires_grad_(False)
  self.A=nn.Parameter(torch.empty(r,base.in_features,device=base.weight.device,dtype=torch.float32))
  self.B=nn.Parameter(torch.zeros(base.out_features,r,device=base.weight.device,dtype=torch.float32))
  nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
  self.dropout=nn.Dropout(dropout);self.scaling=float(alpha)/r
 def forward(self,x):
  original=self.base(x)
  with torch.autocast(device_type=x.device.type,enabled=False):
   delta=F.linear(F.linear(self.dropout(x.float()),self.A),self.B)*self.scaling
  return original+delta.to(original.dtype)

def inject_lora(model,config=None):
 cfg={**DEFAULT_CONFIG,**(config or {})}
 for p in model.parameters():p.requires_grad_(False)
 backbone=model.model
 layers=int(getattr(backbone.config,'num_hidden_layers',0));last=int(cfg['last_n_layers'])
 selected=[]
 for name,module in list(backbone.named_modules()):
  if not isinstance(module,nn.Linear) or name.split('.')[-1] not in ('q_proj','v_proj'):continue
  if last:
   import re
   m=re.search(r'layers\.(\d+)\.',name)
   if m is None or int(m.group(1))<layers-last:continue
  parent,leaf=name.rsplit('.',1)
  setattr(backbone.get_submodule(parent),leaf,LoRALinear(module,cfg['r'],cfg['alpha'],cfg['dropout']))
  selected.append(name)
 if not selected:raise ValueError('No q_proj/v_proj LoRA targets found')
 model.projector.float()
 if hasattr(backbone,'gradient_checkpointing_enable'):
  backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
 backbone.config.use_cache=False
 return selected

def adapter_state(model):return {n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}

def set_adapter_state(model,state):
 named={n:p for n,p in model.named_parameters() if p.requires_grad}
 if set(named)!=set(state):raise ValueError('Adapter parameter names changed')
 with torch.no_grad():
  for n,p in named.items():
   if p.shape!=state[n].shape or not torch.isfinite(state[n]).all():raise ValueError('Invalid adapter state')
   p.copy_(state[n].to(p.device,dtype=p.dtype))

def forward_scores(model,input_ids,attention_mask=None):
 hidden=model.model(input_ids=input_ids,attention_mask=attention_mask,use_cache=False,
                    output_hidden_states=False,return_dict=True).last_hidden_state
 dm=input_ids.eq(model.doc_embed_token_id);qm=input_ids.eq(model.query_embed_token_id)
 if input_ids.shape[0]!=1 or int(qm.sum())!=1 or not int(dm.sum()):raise ValueError('Expected one complete query prompt')
 with torch.autocast(device_type=hidden.device.type,enabled=False):
  d=F.normalize(model.projector(hidden[dm].float()),dim=-1)
  q=F.normalize(model.projector(hidden[qm].float()),dim=-1)
  scores=(d*q).sum(-1)
 if not torch.isfinite(scores).all():raise FloatingPointError('Nonfinite Jina LoRA scores')
 return scores

def ranking_loss(scores,labels,teacher=None,config=None):
 cfg={**DEFAULT_CONFIG,**(config or {})};z=scores.float()*cfg['scale'];y=labels.float()
 if len(z)!=len(y) or not len(z):raise ValueError('Invalid query loss inputs')
 pos=torch.nonzero(y>.5).flatten();neg=torch.nonzero(y<=.5).flatten();rank=z.sum()*0
 if len(pos) and len(neg):
  with torch.no_grad():
   order=torch.argsort(z.detach(),descending=True,stable=True);ranks=torch.empty_like(order)
   ranks[order]=torch.arange(len(z),device=z.device)
   k=torch.arange(len(z),device=z.device);disc=(k<10).float()/torch.log2(k.float()+2)
   ideal=disc[:min(len(pos),10)].sum().clamp_min(1e-8)
   weight=(disc[ranks[pos]][:,None]-disc[ranks[neg]][None,:]).abs()/ideal
  rank=(F.softplus(cfg['margin']-(z[pos,None]-z[neg][None,:]))*weight).sum()/weight.sum().clamp_min(1e-8)
 centered=z-z.mean()+math.log(cfg['prior']/(1-cfg['prior']))
 bce=F.binary_cross_entropy_with_logits(centered,y)
 distill=z.sum()*0
 if teacher is not None:
  t=torch.as_tensor(teacher,dtype=torch.float32,device=z.device)
  if t.shape!=z.shape or not torch.isfinite(t).all():raise ValueError('Invalid frozen teacher')
  # teacher is the frozen raw-cosine mean of id/reverse, not probabilities.
  t=cfg['scale']*(t-t.mean())
  distill=F.kl_div(F.log_softmax(z-z.mean(),dim=0),F.softmax(t,dim=0),reduction='sum')
 return rank+cfg['bce_weight']*bce+cfg['distill_weight']*distill

class PromptStore:
 def __init__(self,frame,tokenizer,formatter,config):
  self.frame=frame;self.tok=tokenizer;self.formatter=formatter;self.cfg=config;self.cache={}
  self.groups={str(q):np.asarray(ix,dtype=np.int64) for q,ix in frame.groupby('query_id',sort=True).indices.items()}
  self.ids=frame.id.to_numpy(dtype=str)
 def get(self,q,order,training=False,seed=42):
  key=(str(q),order,bool(training),int(seed))
  if key in self.cache:return self.cache[key]
  rows=self.groups[str(q)];rows=rows[np.argsort(self.ids[rows],kind='stable')]
  if order=='reverse':rows=rows[::-1]
  elif order=='shuffle':
   rows=np.asarray(sorted(rows,key=lambda i:hashlib.sha256(f'{seed}\0{q}\0{self.ids[i]}'.encode()).digest()),dtype=np.int64)
  elif order!='id':raise ValueError('Unknown LoRA order')
  f=self.frame.iloc[rows];query=f.query_c.iloc[0]
  docs=[(t+'. ' if t else '')+x for t,x in zip(f.title_c,f.text_c)]
  for marker in ('<|embed_token|>','<|rerank_token|>'):
   query=query.replace(marker,'');docs=[x.replace(marker,'') for x in docs]
  query=self.tok.decode(self.tok.encode(query,add_special_tokens=False)[:self.cfg['max_query_tokens']],
                        skip_special_tokens=True,clean_up_tokenization_spaces=False)
  tokens=[self.tok.encode(x,add_special_tokens=False) for x in docs]
  cap=self.cfg['train_doc_tokens' if training else 'eval_doc_tokens'];limit=self.cfg['train_group_tokens' if training else 'eval_group_tokens']
  while True:
   strings=[self.tok.decode(x[:cap],skip_special_tokens=True,clean_up_tokenization_spaces=False) for x in tokens]
   prompt=self.formatter(query,strings,instruction=None,special_tokens={'query_embed_token':'<|rerank_token|>','doc_embed_token':'<|embed_token|>'},no_thinking=True)
   ids=np.asarray(self.tok.encode(prompt,add_special_tokens=True),dtype=np.int64)
   if len(ids)<=limit:break
   new=max(self.cfg['min_doc_tokens'],min(cap-1,int(cap*limit/len(ids)*.95)))
   if new>=cap:raise ValueError('Complete query markers cannot fit LoRA context')
   cap=new
  if np.count_nonzero(ids==151670)!=len(rows) or np.count_nonzero(ids==151671)!=1:raise ValueError('Prompt marker mismatch')
  result=dict(ids=ids,rows=rows,cap=cap);self.cache[key]=result;return result

def _batch(prompt,device):
 ids=torch.as_tensor(prompt['ids'][None,:],dtype=torch.long,device=device)
 return ids,torch.ones_like(ids)

def _load_model(device,config):
 torch.random.default_generator.manual_seed(config['seed'])
 if str(device).startswith('cuda'):torch.cuda.manual_seed(config['seed'])
 model,tok,formatter=jj._load_components(MODEL_NAME,MODEL_REVISION,device)
 model.lm_head=nn.Identity()
 selected=inject_lora(model,config)
 if any(p.dtype!=torch.float32 for p in model.parameters() if p.requires_grad):raise RuntimeError('Trainable LoRA weights must be FP32')
 return model,tok,formatter,selected

def _scaler(device,config):
 return torch.amp.GradScaler('cuda',enabled=str(device).startswith('cuda'),init_scale=config['grad_scaler_init'])

def _cpu_tree(x):
 if isinstance(x,torch.Tensor):return x.detach().cpu()
 if isinstance(x,dict):return {k:_cpu_tree(v) for k,v in x.items()}
 if isinstance(x,list):return [_cpu_tree(v) for v in x]
 if isinstance(x,tuple):return tuple(_cpu_tree(v) for v in x)
 return x

def _atomic_torch(path,value):
 tmp=Path(str(path)+'.tmp');torch.save(value,tmp);os.replace(tmp,path)

def _rng(device):
 return dict(cpu=torch.get_rng_state(),cuda=torch.cuda.get_rng_state(device) if str(device).startswith('cuda') else None)

def _restore_rng(rng,device):
 torch.set_rng_state(rng['cpu'])
 if rng['cuda'] is not None:torch.cuda.set_rng_state(rng['cuda'],device)

def make_lora_folds(tr,training_indices=None):
 if 'grp' not in tr:raise ValueError('tr.grp connected query/document components are required')
 if training_indices is None:
  original=list(GroupKFold(5).split(tr,groups=tr.grp));audit=np.sort(original[-1][1])
  dev=np.setdiff1d(np.arange(len(tr)),audit)
 else:
  dev=np.sort(np.asarray(training_indices,dtype=np.int64));audit=np.setdiff1d(np.arange(len(tr)),dev)
 if len(np.unique(dev))!=len(dev) or np.any(dev<0) or np.any(dev>=len(tr)):raise ValueError('Invalid training indices')
 if set(tr.iloc[dev].grp)&set(tr.iloc[audit].grp):raise ValueError('Component crosses development/audit boundary')
 if 'document_id' in tr and set(tr.iloc[dev].document_id)&set(tr.iloc[audit].document_id):raise ValueError('Document crosses development/audit boundary')
 from final_data import source_folds
 folds=source_folds(tr,dev)
 for a,b in folds:
  if set(tr.iloc[a].query_id)&set(tr.iloc[b].query_id) or set(tr.iloc[a].query_id)&set(tr.iloc[audit].query_id):raise ValueError('Query crosses LoRA fold/audit')
  if 'document_id' in tr and set(tr.iloc[a].document_id)&set(tr.iloc[b].document_id):raise ValueError('Document crosses LoRA fold')
 return dev,audit,folds

def _benchmark(model,store,train_keys,eval_keys,config,budget,device):
 initial=adapter_state(model);parameters=[p for p in model.parameters() if p.requires_grad]
 opt=torch.optim.AdamW(parameters,lr=config['lr'],weight_decay=config['weight_decay']);scaler=_scaler(device,config)
 def representatives(keys,order,training):
  lengths=[]
  for q in keys:
   _check(budget);p=store.get(q,order,training,config['seed']);lengths.append(len(p['ids']))
  ix=np.argsort(lengths,kind='stable')
  return [keys[int(ix[i])] for i in np.unique(np.linspace(0,len(ix)-1,min(config['benchmark_queries'],len(ix))).astype(int))]
 train_probe=representatives(train_keys,'shuffle',True);eval_probe=representatives(eval_keys,'id',False)
 try:
  model.train();opt.zero_grad(set_to_none=True);_sync(device);start=time.monotonic()
  for i,q in enumerate(train_probe):
   _check(budget);p=store.get(q,'shuffle',True,config['seed']);ids,att=_batch(p,device)
   # Synthetic labels are deliberate: benchmark cannot read held-out targets.
   labels=(torch.arange(len(p['rows']),device=device)%2==0).float()
   with torch.autocast(device_type=torch.device(device).type,dtype=torch.float16,enabled=str(device).startswith('cuda')):
    scores=forward_scores(model,ids,att);loss=ranking_loss(scores,labels,None,config)/config['accum']
   scaler.scale(loss).backward()
   if (i+1)%config['accum']==0 or i==len(train_probe)-1:
    scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(parameters,1.);scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
  _sync(device);train_seconds=(time.monotonic()-start)/max(1,len(train_probe))
  model.eval();_sync(device);start=time.monotonic()
  with torch.inference_mode():
   for q in eval_probe:
    _check(budget);ids,att=_batch(store.get(q,'id',False),device);forward_scores(model,ids,att)
  _sync(device);eval_seconds=(time.monotonic()-start)/max(1,len(eval_probe))
  return dict(train_seconds_per_query=train_seconds,inference_seconds_per_query=eval_seconds,
              train_probe_queries=len(train_probe),inference_probe_queries=len(eval_probe),
              train_max_tokens=max(len(store.get(q,'shuffle',True,config['seed'])['ids']) for q in train_probe),
              peak_cuda_gb=torch.cuda.max_memory_allocated()/2**30 if str(device).startswith('cuda') else None,
              benchmark_labels='synthetic; adapter restored, optimizer/scaler discarded before folds')
 finally:
  set_adapter_state(model,initial);opt.zero_grad(set_to_none=True);del opt,scaler

def _checkpoint_read(path,signature):
 if not path.is_file():return None
 c=torch.load(path,map_location='cpu',weights_only=True)
 if c.get('signature')!=signature:raise ValueError('LoRA training checkpoint signature changed; use a new output directory')
 return c

def _train_fold(model,store,keys,y,teacher,fold,config,path,signature,budget,device):
 rng=np.random.default_rng(config['seed']+fold);keys=[keys[i] for i in rng.permutation(len(keys))]
 params=[p for p in model.parameters() if p.requires_grad]
 opt=torch.optim.AdamW(params,lr=config['lr'],weight_decay=config['weight_decay']);scaler=_scaler(device,config)
 state=_checkpoint_read(path,signature);cursor=updates=overflows=0;loss_sum=0.
 if state is not None:
  set_adapter_state(model,state['adapter']);opt.load_state_dict(state['optimizer']);scaler.load_state_dict(state['scaler'])
  cursor=int(state['cursor']);updates=int(state['updates']);overflows=int(state['overflows']);loss_sum=float(state['loss_sum'])
  if state['query_order']!=keys:raise ValueError('LoRA training query order changed')
  if cursor<0 or cursor>len(keys) or (cursor%config['accum']!=0 and cursor!=len(keys)):raise ValueError('Invalid committed optimizer cursor')
  _restore_rng(state['rng'],device)
 else:
  torch.random.default_generator.manual_seed(config['seed']+fold)
  if str(device).startswith('cuda'):torch.cuda.manual_seed(config['seed']+fold)
 total_steps=math.ceil(len(keys)/config['accum']);last_save=time.monotonic();model.train()
 def save():
  _atomic_torch(path,dict(signature=signature,adapter=adapter_state(model),optimizer=_cpu_tree(opt.state_dict()),
    scaler=scaler.state_dict(),rng=_rng(device),cursor=cursor,updates=updates,overflows=overflows,
    loss_sum=loss_sum,query_order=keys,complete=cursor==len(keys)))
 while cursor<len(keys):
  _check(budget);end=min(cursor+config['accum'],len(keys));opt.zero_grad(set_to_none=True);window_loss=0.
  warm=max(1,int(total_steps*config['warmup']));step=cursor//config['accum']
  factor=(step+1)/warm if step<warm else max(0.,(total_steps-step)/max(1,total_steps-warm))
  for group in opt.param_groups:group['lr']=config['lr']*factor
  for j in range(cursor,end):
   _check(budget);p=store.get(keys[j],'shuffle',True,config['seed']+fold);ids,att=_batch(p,device)
   labels=torch.as_tensor(y[p['rows']],dtype=torch.float32,device=device)
   with torch.autocast(device_type=torch.device(device).type,dtype=torch.float16,enabled=str(device).startswith('cuda')):
    scores=forward_scores(model,ids,att);loss=ranking_loss(scores,labels,teacher[p['rows']],config)
   if not torch.isfinite(loss):raise FloatingPointError('Nonfinite LoRA training loss; incomplete fold not published')
   window_loss+=float(loss.detach());scaler.scale(loss/(end-cursor)).backward()
  scaler.unscale_(opt);norm=torch.nn.utils.clip_grad_norm_(params,1.);before=float(scaler.get_scale())
  if not torch.isfinite(norm) and not str(device).startswith('cuda'):raise FloatingPointError('Nonfinite CPU LoRA gradient')
  scaler.step(opt);scaler.update();after=float(scaler.get_scale())
  if after<before:
   overflows+=1
   if overflows>config['max_overflow_windows']:raise FloatingPointError('Repeated FP16 gradient overflow; trained fold not published')
  else:updates+=1
  cursor=end;loss_sum+=window_loss;opt.zero_grad(set_to_none=True)
  if cursor%config['checkpoint_queries']==0 or cursor==len(keys) or time.monotonic()-last_save>60:
   save();last_save=time.monotonic();print(f'LoRA fold{fold}: {cursor}/{len(keys)} train queries; updates={updates}, overflow={overflows}',flush=True)
 if updates<1:raise FloatingPointError('No successful LoRA optimizer updates')
 save();return dict(train_queries=len(keys),successful_updates=updates,overflow_windows=overflows,mean_loss=loss_sum/max(1,len(keys)))

def _predict_fold(model,store,keys,order,path,signature,budget,device):
 n=len(store.frame);prediction=np.full(n,np.nan,np.float32);done=set()
 if path.is_file():
  with np.load(path,allow_pickle=False) as z:
   if str(z['signature'].item())!=signature or not np.array_equal(z['ids'],store.ids):raise ValueError('LoRA score cache binding changed')
   prediction=z['prediction'].copy();done=set(z['done'].astype(str))
  if prediction.shape!=(n,) or not done<=set(keys):raise ValueError('Malformed LoRA score cache')
  for q in done:
   if not np.isfinite(prediction[store.groups[q]]).all():raise ValueError('Incomplete committed query')
 pending=0;last=time.monotonic()
 def save():jj._atomic_npz(path,ids=store.ids,prediction=prediction,done=np.asarray(sorted(done),dtype=str),signature=np.asarray(signature))
 model.eval()
 try:
  with torch.inference_mode():
   for q in keys:
    if q in done:continue
    _check(budget);p=store.get(q,order,False);ids,att=_batch(p,device)
    scores=forward_scores(model,ids,att).cpu().numpy().astype(np.float32)
    if scores.shape!=(len(p['rows']),) or not np.isfinite(scores).all():raise FloatingPointError('Invalid query inference')
    prediction[p['rows']]=scores;done.add(q);pending+=1
    if pending>=32 or time.monotonic()-last>30:save();pending=0;last=time.monotonic()
 finally:
  if pending:save()
 if done!=set(keys):raise LoRABudget('LoRA inference incomplete')
 return prediction

def _query_z(frame,score):
 s=pd.Series(np.asarray(score,dtype=float));g=s.groupby(frame.query_id.astype(str).to_numpy(),sort=False)
 return ((s-g.transform('mean'))/(g.transform('std').fillna(0)+1e-6)).to_numpy(np.float32)

def fit_jina_lora(tr,te,teacher_scores,out_dir,budget,config=None,train_rows=None,
                  device='cuda',second_order=True,preflight_only=False):
 """Return (modern_score_pairs, diagnostics), or an empty dict on budget stop.

 teacher_scores={'id':(train_raw_cos,test_raw_cos),'reverse':(...)}. Default
 training rows exclude the last original five-fold audit. Two new component
 folds split development rows only. Returned train scores are dev OOF plus
 audit predictions averaged from both audit-blind models. No audit labels
 enter gradients, benchmark, cache identity, or hyperparameter selection.
 """
 started=time.monotonic();cfg={**DEFAULT_CONFIG,**(config or {})};out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)
 if set(config or {})-set(DEFAULT_CONFIG):raise ValueError('Unknown LoRA configuration key')
 if cfg['epochs']!=1:raise ValueError('This fixed-budget implementation supports exactly one epoch')
 tr,te=tr.reset_index(drop=True).copy(),te.reset_index(drop=True).copy()
 for name,f in [('train',tr),('test',te)]:
  if not {'id','query_id','query_c','title_c','text_c'}<=set(f) or not f.id.is_unique or f.id.isna().any():raise ValueError(name+': missing/duplicate canonical pair data')
  f['id']=f.id.astype(str);f['query_id']=f.query_id.astype(str)
  for c in ['query_c','title_c','text_c']:f[c]=f[c].fillna('').astype(str)
 if set(tr.query_id)&set(te.query_id) or set(tr.id)&set(te.id):raise ValueError('Train/test overlap')
 dev,audit,folds=make_lora_folds(tr,train_rows)
 if not tr.iloc[dev].relevance.isin([0,1]).all():raise ValueError('Development labels must be binary')
 y=np.zeros(len(tr)+len(te),np.float32);y[dev]=tr.iloc[dev].relevance.to_numpy(np.float32)
 full=pd.concat([tr,te],ignore_index=True);full['query_id']=full.query_id.astype(str)
 if set(teacher_scores)!={'id','reverse'}:raise ValueError('Supply frozen raw id and reverse Jina cosine scores')
 teachers=[];teacher_hash={}
 for name,pair in teacher_scores.items():
  if len(pair)!=2:raise ValueError('Teacher requires train/test arrays')
  a,b=map(np.asarray,pair)
  if a.shape!=(len(tr),) or b.shape!=(len(te),) or not np.isfinite(a).all() or not np.isfinite(b).all():raise ValueError('Malformed teacher stream '+name)
  teachers.append(np.r_[a,b].astype(np.float32));teacher_hash[name]=[_array_digest(a),_array_digest(b)]
 teacher=np.mean(teachers,axis=0).astype(np.float32)
 versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','safetensors']}
 binding=dict(model=MODEL_NAME,revision=MODEL_REVISION,config=cfg,device=str(device),versions=versions,
  text_digest=jj._input_digest(full),ids=_strings_digest(full.id),dev_rows=_array_digest(dev),audit_rows=_array_digest(audit),
  development_labels=_array_digest(y[dev]),teacher_hash=teacher_hash,
  folds=[dict(a=_array_digest(a),b=_array_digest(b)) for a,b in folds])
 request_key=hashlib.sha256(json.dumps(binding,sort_keys=True).encode()).hexdigest();planpath=out/'plan.json'
 oldplan=json.loads(planpath.read_text()) if planpath.is_file() else None
 if oldplan and oldplan.get('request_key')!=request_key:raise ValueError('LoRA source/config/folds changed: select a new out_dir')
 resolved=oldplan['resolved_config'] if oldplan else dict(cfg)
 signature=oldplan['signature'] if oldplan else None
 diag=dict(status='initializing',request_key=request_key,preflight=None,train_development_queries=int(tr.iloc[dev].query_id.nunique()),
  untouched_audit_queries=int(tr.iloc[audit].query_id.nunique()),n_component_folds=2,no_audit_training=True,
  train_score_scope='development out-of-fold; audit mean of two audit-blind models',folds=[],
  validation_caveat='New LoRA excludes audit labels entirely. Downstream CV with learned OOF inputs remains approximate; frozen old sources can contain downstream-fold label dependence.')
 def final_cache(order):return out/f'complete_{order}.npz'
 def load_complete(order):
  path=final_cache(order)
  if not path.is_file() or signature is None:return None
  with np.load(path,allow_pickle=False) as z:
   if str(z['signature'].item())!=signature or not np.array_equal(z['train_ids'],tr.id.to_numpy(dtype=str)) or not np.array_equal(z['test_ids'],te.id.to_numpy(dtype=str)):raise ValueError('Completed LoRA score identity changed')
   a,b=z['train'].copy(),z['test'].copy();hashes=z['weight_hashes'].astype(str)
  if a.shape!=(len(tr),) or b.shape!=(len(te),) or not np.isfinite(a).all() or not np.isfinite(b).all():raise ValueError('Invalid complete LoRA scores')
  if hashes.shape!=(2,):raise ValueError('Exactly two LoRA adapter hashes are required')
  for k,h in enumerate(hashes):
   wp=out/f'fold{k}_adapter.safetensors'
   if not wp.is_file() or hashlib.sha256(wp.read_bytes()).hexdigest()!=h:raise ValueError('Completed LoRA scores have missing/changed adapter weights')
  return a,b
 def collect():
  pairs={};a=load_complete('id');b=load_complete('reverse')
  if a is not None:
   pairs['jina35_lora_id']=a
   pairs['jina35_lora']=a
  if a is not None and b is not None:
   pairs['jina35_lora_reverse']=b
   pairs['jina35_lora']=((.5*(_query_z(tr,a[0])+_query_z(tr,b[0]))).astype(np.float32),
                         (.5*(_query_z(te,a[1])+_query_z(te,b[1]))).astype(np.float32))
  return pairs
 def finish(status):
  diag.update(status=status,seconds=time.monotonic()-started,resolved_config=resolved)
  jj._atomic_json(out/'lora_diagnostics.json',diag)
  return collect(),diag
 if signature and load_complete('id') is not None and (not second_order or load_complete('reverse') is not None):return finish('complete_cached')
 model=store=None
 try:
  if oldplan:
   diag['preflight']=oldplan['benchmark'];rates=oldplan['benchmark']
  else:
   for cap in [cfg['train_group_tokens']]+([4096] if cfg['train_group_tokens']>4096 else []):
    resolved={**cfg,'train_group_tokens':int(cap)}
    if cap<cfg['train_group_tokens']:resolved['eval_group_tokens']=min(cfg['eval_group_tokens'],int(cap))
    try:
     _check(budget);model,tok,formatter,_=_load_model(device,resolved);store=PromptStore(full,tok,formatter,resolved)
     trainkeys=sorted(set(tr.iloc[dev].query_id));evalkeys=sorted(set(full.query_id))
     rates=_benchmark(model,store,trainkeys,evalkeys,resolved,budget,device);break
    except torch.cuda.OutOfMemoryError:
     if cap<=4096:raise
     model=store=None;gc.collect()
     if torch.cuda.is_available():torch.cuda.empty_cache()
     print('LoRA preflight OOM: retrying consistent 4096-token training context',flush=True)
   signature=hashlib.sha256((request_key+json.dumps(resolved,sort_keys=True)).encode()).hexdigest()
   oldplan=dict(request_key=request_key,signature=signature,resolved_config=resolved,benchmark=rates,binding=binding,
                audit_policy='Never train LoRA on audit targets',training='2 development component folds, one epoch')
   jj._atomic_json(planpath,oldplan);diag['preflight']=rates
  # Conservative estimate uses remaining train queries and one mandatory order.
  ntrain=0;neval=0
  for k,(a,b) in enumerate(folds):
   keys=sorted(set(tr.iloc[a].query_id));cp=_checkpoint_read(out/f'fold{k}_train.pt',signature)
   ntrain+=max(0,len(keys)-(int(cp['cursor']) if cp else 0))
   expected=set(tr.iloc[b].query_id)|set(tr.iloc[audit].query_id)|set(te.query_id)
   done=set()
   sp=out/f'fold{k}_id.npz'
   if sp.is_file():
    with np.load(sp,allow_pickle=False) as z:done=set(z['done'].astype(str))
   neval+=len(expected-done)
  estimate=resolved['safety']*(ntrain*rates['train_seconds_per_query']+neval*rates['inference_seconds_per_query'])
  diag['estimated_mandatory_seconds']=estimate;diag['available_neural_seconds']=jj._available(budget)
  print(f'LoRA preflight: {rates["train_seconds_per_query"]:.3f}s/train query, {rates["inference_seconds_per_query"]:.3f}s/inference; mandatory ETA {estimate/60:.1f}min, available {jj._available(budget)/60:.1f}min',flush=True)
  if preflight_only:return finish('preflight_only')
  if estimate>jj._available(budget):return finish('refused_eta')
  model=store=None;gc.collect()
  if torch.cuda.is_available():torch.cuda.empty_cache()
  def run_order(order,train_allowed):
   predictions=[];hashes=[]
   for k,(a,b) in enumerate(folds):
    _check(budget);model,tok,formatter,targets=_load_model(device,{**resolved,'seed':resolved['seed']+k});store=PromptStore(full,tok,formatter,resolved)
    wp=out/f'fold{k}_adapter.safetensors';cp=out/f'fold{k}_train.pt'
    if wp.is_file():
     with safe_open(str(wp),framework='pt',device='cpu') as handle:metadata=handle.metadata() or {}
     if metadata.get('signature')!=signature or metadata.get('fold')!=str(k) or metadata.get('model_revision')!=MODEL_REVISION:raise ValueError('LoRA adapter checkpoint metadata changed')
     set_adapter_state(model,load_file(str(wp),device='cpu'))
     record=dict(fold=k,cached_training=True)
    elif train_allowed:
     keys=sorted(set(tr.iloc[a].query_id))
     record=_train_fold(model,store,keys,y,teacher,k,resolved,cp,signature,budget,device)
     save_file(adapter_state(model),str(wp)+'.tmp',metadata={'signature':signature,'fold':str(k),'model_revision':MODEL_REVISION})
     os.replace(str(wp)+'.tmp',wp)
    else:raise ValueError('Missing trained LoRA adapter for optional inference')
    weight_hash=hashlib.sha256(wp.read_bytes()).hexdigest();hashes.append(weight_hash)
    evalkeys=sorted(set(tr.iloc[b].query_id)|set(tr.iloc[audit].query_id)|set(te.query_id))
    key=hashlib.sha256((signature+str(k)+order+weight_hash).encode()).hexdigest()
    p=_predict_fold(model,store,evalkeys,order,out/f'fold{k}_{order}.npz',key,budget,device)
    predictions.append(p);record.update(fold=k,trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),targets=len(targets))
    diag['folds'].append(record);del model,store;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
   train=np.full(len(tr),np.nan,np.float32)
   for k,(_,b) in enumerate(folds):train[b]=predictions[k][b]
   train[audit]=np.mean([p[audit] for p in predictions],axis=0)
   test=np.mean([p[len(tr):] for p in predictions],axis=0).astype(np.float32)
   if not np.isfinite(train).all() or not np.isfinite(test).all():raise RuntimeError('LoRA OOF/audit/test incomplete')
   jj._atomic_npz(final_cache(order),train_ids=tr.id.to_numpy(dtype=str),test_ids=te.id.to_numpy(dtype=str),
     train=train,test=test,weight_hashes=np.asarray(hashes,dtype=str),signature=np.asarray(signature))
  if load_complete('id') is None:run_order('id',True)
  if second_order and load_complete('reverse') is None:
   reverse_queries=sum(len(set(tr.iloc[b].query_id)|set(tr.iloc[audit].query_id)|set(te.query_id)) for _,b in folds)
   eta=resolved['safety']*reverse_queries*rates['inference_seconds_per_query'];diag['optional_reverse_eta_seconds']=eta
   if eta<=jj._available(budget):run_order('reverse',False)
   else:diag['optional_reverse_skipped']='Insufficient time; complete mandatory id stream retained'
  return finish('complete')
 except (LoRABudget,jj.JointBudget) as e:
  diag['stop_reason']=str(e);return finish('budget_stopped')
 except FloatingPointError as e:
  diag['stop_reason']=str(e);return finish('numerical_stopped')
 except torch.cuda.OutOfMemoryError:
  diag['stop_reason']='OOM after preflight: partial gradients discarded, resume only from a committed optimizer checkpoint; use a new directory for smaller global context'
  return finish('oom_stopped')
 finally:
  model=store=None;gc.collect()
  if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:
%%writefile gemma_v8.py
"""Independent Gemma2B-family ranking signal with a measured training gate.
Pinned BGE-v2-Gemma causal-LM YES-row scoring; no random classifier or full-vocab
logits. Frozen scores first, then optional audit-blind two-fold LoRA. Native
GemmaModel retains embedding scaling/final norm. No external training data.
"""
from __future__ import annotations
import gc,hashlib,importlib.metadata,json,math,os,re,time
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from transformers import AutoModelForCausalLM,AutoTokenizer
from safetensors import safe_open
from safetensors.torch import save_file,load_file
import jina_joint as jj
from jina_probe import _array_digest,_strings_digest
from jina_lora_v7 import LoRALinear,adapter_state,set_adapter_state,_atomic_torch,_cpu_tree,_rng,_restore_rng,make_lora_folds

MODEL_NAME='BAAI/bge-reranker-v2-gemma'
MODEL_REVISION='1787044f8b6fb740a9de4557c3a12377f84d9e17'
INSTRUCTION="Given a query A and a passage B, determine whether the passage contains an answer to the query by providing a prediction of either 'Yes' or 'No'."
CONFIG=dict(version='gemma-v8-1',frozen_body=768,train_body=512,eval_body=768,query_tokens=128,
 batch_size=16,pair_block=2,r=8,alpha=16,dropout=.05,last_n_layers=8,
 epochs=1,accum=4,lr=1e-4,weight_decay=.01,warmup=.05,seed=42,
 bce_weight=.1,distill_weight=.03,margin=.2,prior=.26,safety=1.35,
 benchmark_pairs=128,benchmark_queries=32,grad_scaler_init=1024.,max_overflow_windows=5,
 lora_normalization='query-z-each-source-before-mean-v1')
class GemmaBudget(RuntimeError):pass

def _check(budget):
 try:jj._check_budget(budget)
 except jj.JointBudget as e:raise GemmaBudget(str(e)) from e

def _sync(device):
 if str(device).startswith('cuda'):torch.cuda.synchronize()

def encode_pair(tokenizer,query,document,body_cap=768,query_cap=128):
 """Official BOS/A/B body truncation, then newline+instruction; no chat/EOS.
 body_cap excludes the appended instruction, as in the official model card.
 """
 sep=tokenizer.encode('\n',add_special_tokens=False)
 prompt=tokenizer.encode(INSTRUCTION,add_special_tokens=False)
 q=tokenizer.encode('A: '+str(query),add_special_tokens=False)[:query_cap]
 d=tokenizer.encode('B: '+str(document),add_special_tokens=False)
 head=[tokenizer.bos_token_id]+q+sep
 if len(head)>=body_cap:raise ValueError('Query leaves no room for a passage')
 return np.asarray(head+d[:body_cap-len(head)]+sep+prompt,dtype=np.int64)

class PairStore:
 def __init__(self,frame,tokenizer,config):
  self.frame=frame;self.tok=tokenizer;self.cfg=config;self.cache={};self.ids=frame.id.to_numpy(dtype=str)
  self.groups={str(q):np.asarray(ix,dtype=np.int64) for q,ix in frame.groupby('query_id',sort=True).indices.items()}
 def get(self,row,body):
  key=(int(row),int(body))
  if key not in self.cache:
   f=self.frame.iloc[int(row)];doc=(f.title_c+'. ' if f.title_c else '')+f.text_c
   self.cache[key]=encode_pair(self.tok,f.query_c,doc,body,self.cfg['query_tokens'])
  return self.cache[key]
 def batch(self,rows,body,device):
  seq=[self.get(i,body) for i in rows];width=int(math.ceil(max(map(len,seq))/8)*8)
  pad=self.tok.pad_token_id
  if pad is None:pad=self.tok.eos_token_id
  ids=torch.full((len(seq),width),int(pad),dtype=torch.long);mask=torch.zeros_like(ids)
  for j,s in enumerate(seq):ids[j,-len(s):]=torch.as_tensor(s);mask[j,-len(s):]=1
  return ids.to(device),mask.to(device)

class GemmaRanker(nn.Module):
 def __init__(self,backbone,yes_weight):
  super().__init__();self.backbone=backbone
  vector=yes_weight.detach().clone().float().reshape(1,-1)
  self.head=nn.Linear(vector.shape[1],1,bias=False,device=vector.device,dtype=torch.float32)
  with torch.no_grad():self.head.weight.copy_(vector)
  for p in self.parameters():p.requires_grad_(False)
 def forward(self,input_ids,attention_mask=None):
  if attention_mask is None:attention_mask=torch.ones_like(input_ids)
  if torch.any(attention_mask.sum(1)==0):raise ValueError('Empty Gemma sequence')
  h=self.backbone(input_ids=input_ids,attention_mask=attention_mask,use_cache=False,
     output_hidden_states=False,return_dict=True).last_hidden_state
  pos=(attention_mask.long()*torch.arange(h.shape[1],device=h.device)[None,:]).amax(1)
  last=h[torch.arange(len(h),device=h.device),pos]
  with torch.autocast(device_type=h.device.type,enabled=False):score=F.linear(last.float(),self.head.weight.float()).squeeze(-1)
  if not torch.isfinite(score).all():raise FloatingPointError('Nonfinite Gemma YES scores')
  return score

def inject_gemma_lora(model,config=None):
 cfg={**CONFIG,**(config or {})}
 for p in model.parameters():p.requires_grad_(False)
 layers=int(model.backbone.config.num_hidden_layers);last=int(cfg['last_n_layers']);targets=[]
 for name,module in list(model.backbone.named_modules()):
  if not isinstance(module,nn.Linear) or name.split('.')[-1] not in ('q_proj','v_proj'):continue
  m=re.search(r'layers\.(\d+)\.',name)
  if last and (m is None or int(m.group(1))<layers-last):continue
  parent,leaf=name.rsplit('.',1)
  setattr(model.backbone.get_submodule(parent),leaf,LoRALinear(module,cfg['r'],cfg['alpha'],cfg['dropout']));targets.append(name)
 if not targets:raise ValueError('No Gemma q/v LoRA targets')
 model.head.weight.requires_grad_(True)
 model.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
 model.backbone.config.use_cache=False
 return targets

def _load_model(device='cuda',training=False,config=None):
 cfg={**CONFIG,**(config or {})};torch.random.default_generator.manual_seed(cfg['seed'])
 if str(device).startswith('cuda'):torch.cuda.manual_seed(cfg['seed'])
 tok=AutoTokenizer.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,padding_side='left')
 yes=tok.encode('Yes',add_special_tokens=False)
 if len(yes)!=1:raise ValueError('Gemma Yes must be a single token')
 dtype=torch.float16 if str(device).startswith('cuda') else torch.float32
 base=AutoModelForCausalLM.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,torch_dtype=dtype,
   low_cpu_mem_usage=True,device_map={'':str(device)},attn_implementation='sdpa',trust_remote_code=False)
 ranker=GemmaRanker(base.model,base.get_output_embeddings().weight[yes[0]])
 del base
 if training:inject_gemma_lora(ranker,cfg)
 return ranker.eval(),tok

def gemma_rank_loss(scores,labels,teacher,config=None):
 cfg={**CONFIG,**(config or {})};z=scores.float();y=labels.float()
 if z.shape!=y.shape or not len(z):raise ValueError('Invalid whole-query loss')
 pos=torch.nonzero(y>.5).flatten();neg=torch.nonzero(y<=.5).flatten();rank=z.sum()*0
 if len(pos) and len(neg):
  with torch.no_grad():
   order=torch.argsort(z.detach(),descending=True,stable=True);ranks=torch.empty_like(order);ranks[order]=torch.arange(len(z),device=z.device)
   k=torch.arange(len(z),device=z.device);disc=(k<10).float()/torch.log2(k.float()+2)
   ideal=disc[:min(len(pos),10)].sum().clamp_min(1e-8)
   delta=(disc[ranks[pos]][:,None]-disc[ranks[neg]][None,:]).abs()/ideal
  rank=(F.softplus(cfg['margin']-(z[pos,None]-z[neg][None,:]))*delta).sum()/delta.sum().clamp_min(1e-8)
 bce=F.binary_cross_entropy_with_logits(z-z.mean()+math.log(cfg['prior']/(1-cfg['prior'])),y)
 t=torch.as_tensor(teacher,dtype=torch.float32,device=z.device)
 if t.shape!=z.shape or not torch.isfinite(t).all():raise ValueError('Invalid frozen Gemma teacher')
 scale=t.std(unbiased=False).detach().clamp_min(1.)
 kl=F.kl_div(F.log_softmax((z-z.mean())/scale,dim=0),F.softmax((t-t.mean())/scale,dim=0),reduction='sum')
 return rank+cfg['bce_weight']*bce+cfg['distill_weight']*kl

def _clean(tr,te):
 a,b=tr.reset_index(drop=True).copy(),te.reset_index(drop=True).copy()
 for f in (a,b):
  if not {'id','query_id','query_c','title_c','text_c'}<=set(f) or not f.id.is_unique or f.id.isna().any() or f.query_id.isna().any():raise ValueError('Canonical Gemma data missing/duplicate')
  f['id']=f.id.astype(str);f['query_id']=f.query_id.astype(str)
  for c in ['query_c','title_c','text_c']:f[c]=f[c].fillna('').astype(str)
 if set(a.id)&set(b.id) or set(a.query_id)&set(b.query_id):raise ValueError('Gemma train/test overlap')
 return a,b,pd.concat([a,b],ignore_index=True)

def _signature(full,cfg,phase,extra=None):
 payload=dict(model=MODEL_NAME,revision=MODEL_REVISION,phase=phase,config=cfg,
  data_digest=jj._input_digest(full),ids=_strings_digest(full.id),extra=extra,
  versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','safetensors']})
 return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest(),payload

def _manifest(path,key,payload):
 if path.is_file() and json.loads(path.read_text()).get('signature')!=key:raise ValueError('Gemma cache source/config changed; choose new directory')
 jj._atomic_json(path,dict(signature=key,**payload))

def _score_cache(path,ids,key):
 if not path.is_file():return np.full(len(ids),np.nan,np.float32)
 with np.load(path,allow_pickle=False) as z:
  if str(z['signature'].item())!=key or not np.array_equal(z['ids'],ids):raise ValueError('Gemma score cache ID/config mismatch')
  p=z['prediction'].copy()
 if p.shape!=(len(ids),) or np.isinf(p).any():raise ValueError('Malformed Gemma score cache')
 return p

def _save_score(path,ids,key,p):jj._atomic_npz(path,ids=ids,prediction=p,signature=np.asarray(key))

def _infer(model,store,rows,body,pred,path,key,budget,device,batch_size=16,phase='Gemma'):
 rows=np.asarray(rows,dtype=np.int64);pending=rows[~np.isfinite(pred[rows])]
 if not len(pending):return pred,batch_size
 order=pending[np.argsort([len(store.get(i,body)) for i in pending],kind='stable')]
 i=0;last=time.monotonic();started=last;last_done=0;bs=batch_size
 model.eval()
 try:
  with torch.inference_mode():
   while i<len(order):
    _check(budget);ix=order[i:i+bs]
    try:
     ids,mask=store.batch(ix,body,device);scores=model(ids,mask).cpu().numpy().astype(np.float32)
    except torch.cuda.OutOfMemoryError:
     if bs==1:raise
     bs=max(1,bs//2);gc.collect()
     if torch.cuda.is_available():torch.cuda.empty_cache()
     print(phase,'inference OOM; batch',bs,flush=True);continue
    pred[ix]=scores;i+=len(ix)
    if i-last_done>=256 or time.monotonic()-last>30 or i==len(order):
     _save_score(path,store.ids,key,pred);elapsed=time.monotonic()-started;eta=(len(order)-i)*elapsed/max(1,i)
     print(f'{phase}: {i}/{len(order)} remaining pairs; {elapsed/60:.1f}min elapsed; ETA {eta/60:.1f}min',flush=True)
     last=time.monotonic();last_done=i
 finally:_save_score(path,store.ids,key,pred)
 return pred,bs

def _frozen_cfg(cfg):return {k:cfg[k] for k in ['version','frozen_body','query_tokens']}

def run_frozen_gemma(tr,te,cache_dir,budget,config=None,device='cuda'):
 cfg={**CONFIG,**(config or {})};tr,te,full=_clean(tr,te);root=Path(cache_dir);root.mkdir(parents=True,exist_ok=True)
 key,payload=_signature(full,_frozen_cfg(cfg),'frozen');_manifest(root/'frozen_manifest.json',key,payload)
 path=root/'frozen_scores.npz';p=_score_cache(path,full.id.to_numpy(dtype=str),key)
 diag=dict(kind='zero_shot',phase='frozen',model=MODEL_NAME,revision=MODEL_REVISION,signature=key,status='initializing')
 def finish(status):
  diag['status']=status;jj._atomic_json(root/'frozen_diagnostics.json',diag)
  return ({'gemma_frozen':(p[:len(tr)].copy(),p[len(tr):].copy())} if np.isfinite(p).all() else {}),diag
 if np.isfinite(p).all():return finish('complete_cached')
 model=None
 try:
  _check(budget);model,tok=_load_model(device,False,cfg);store=PairStore(full,tok,cfg)
  pending=np.flatnonzero(~np.isfinite(p));length=np.array([len(store.get(i,cfg['frozen_body'])) for i in pending]);order=pending[np.argsort(length,kind='stable')]
  warm=order[len(order)//2:len(order)//2+min(16,len(order))]
  p,bs=_infer(model,store,warm,cfg['frozen_body'],p,path,key,budget,device,cfg['batch_size'],'Gemma warmup')
  pending=order[~np.isfinite(p[order])]
  probe=pending[np.unique(np.linspace(0,len(pending)-1,min(cfg['benchmark_pairs'],len(pending))).astype(int))] if len(pending) else []
  _sync(device);start=time.monotonic()
  p,bs=_infer(model,store,probe,cfg['frozen_body'],p,path,key,budget,device,bs,'Gemma frozen benchmark')
  _sync(device);rate=len(probe)/max(time.monotonic()-start,1e-6) if len(probe) else 1.
  remaining=int((~np.isfinite(p)).sum());eta=cfg['safety']*remaining/max(rate,1e-9)
  diag.update(pairs_per_second=rate,estimated_remaining_seconds=eta,available_seconds=jj._available(budget),batch_size=bs)
  print(f'Gemma frozen: {rate:.2f} pairs/s; ETA with reserve {eta/60:.1f}min; available {jj._available(budget)/60:.1f}min',flush=True)
  if eta>jj._available(budget):return finish('refused_eta')
  p,bs=_infer(model,store,np.arange(len(full)),cfg['frozen_body'],p,path,key,budget,device,bs,'Gemma frozen')
  diag['batch_size']=bs;return finish('complete')
 except (GemmaBudget,jj.JointBudget) as e:diag['reason']=str(e);return finish('budget_stopped')
 except (FloatingPointError,torch.cuda.OutOfMemoryError) as e:diag['reason']=str(e);return finish('numerical_or_oom_stopped')
 finally:
  model=None;gc.collect()
  if torch.cuda.is_available():torch.cuda.empty_cache()

def _query_scores(model,store,rows,cfg,device):
 scores=[]
 for start in range(0,len(rows),cfg['pair_block']):
  ids,mask=store.batch(rows[start:start+cfg['pair_block']],cfg['train_body'],device)
  scores.append(model(ids,mask))
 return torch.cat(scores)

def _scaler(device,cfg):return torch.amp.GradScaler('cuda',enabled=str(device).startswith('cuda'),init_scale=cfg['grad_scaler_init'])

def _bench_train(model,store,train_keys,teacher,cfg,budget,device):
 initial=adapter_state(model);params=[p for p in model.parameters() if p.requires_grad]
 opt=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=cfg['weight_decay']);scaler=_scaler(device,cfg)
 cost=[]
 for q in train_keys:
  _check(budget);cost.append(sum(len(store.get(i,cfg['train_body'])) for i in store.groups[q]))
 ix=np.argsort(cost,kind='stable');probe=[train_keys[int(ix[i])] for i in np.unique(np.linspace(0,len(ix)-1,min(cfg['benchmark_queries'],len(ix))).astype(int))]
 try:
  model.train();opt.zero_grad(set_to_none=True);_sync(device);start=time.monotonic()
  for j,q in enumerate(probe):
   _check(budget);rows=store.groups[q]
   with torch.autocast(device_type=torch.device(device).type,dtype=torch.float16,enabled=str(device).startswith('cuda')):
    scores=_query_scores(model,store,rows,cfg,device)
    labels=(torch.arange(len(rows),device=device)%2==0).float()
    loss=gemma_rank_loss(scores,labels,teacher[rows],cfg)/cfg['accum']
   scaler.scale(loss).backward()
   if (j+1)%cfg['accum']==0 or j==len(probe)-1:
    scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(params,1.);scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
  _sync(device);seconds=(time.monotonic()-start)/max(1,len(probe))
  # Measure the actual LoRA inference path, rather than assuming frozen speed.
  rows=np.arange(len(store.frame));length=np.array([len(store.get(i,cfg['eval_body'])) for i in rows]);ix=rows[np.argsort(length,kind='stable')]
  probe_rows=ix[np.unique(np.linspace(0,len(ix)-1,min(cfg['benchmark_pairs'],len(ix))).astype(int))]
  model.eval();batch=cfg['batch_size'];done=0;_sync(device);start=time.monotonic()
  with torch.inference_mode():
   while done<len(probe_rows):
    _check(budget);part=probe_rows[done:done+batch]
    try:ids,mask=store.batch(part,cfg['eval_body'],device);model(ids,mask)
    except torch.cuda.OutOfMemoryError:
     if batch==1:raise
     batch=max(1,batch//2);gc.collect()
     if torch.cuda.is_available():torch.cuda.empty_cache()
     continue
    done+=len(part)
  _sync(device);rate=len(probe_rows)/max(time.monotonic()-start,1e-6)
  return dict(train_seconds_per_query=seconds,inference_pairs_per_second=rate,inference_batch=batch,
    train_probe_queries=len(probe),pair_block=cfg['pair_block'],last_n_layers=cfg['last_n_layers'],
    peak_cuda_gb=torch.cuda.max_memory_allocated()/2**30 if str(device).startswith('cuda') else None,
    benchmark_labels='synthetic; adapters/head restored, optimizer discarded before source folds')
 finally:set_adapter_state(model,initial);opt.zero_grad(set_to_none=True)

def _checkpoint(path,key):
 if not path.is_file():return None
 x=torch.load(path,map_location='cpu',weights_only=True)
 if x.get('signature')!=key:raise ValueError('Gemma optimizer checkpoint identity changed')
 return x

def _train(model,store,keys,y,teacher,fold,cfg,path,key,budget,device):
 rng=np.random.default_rng(cfg['seed']+fold);keys=[keys[i] for i in rng.permutation(len(keys))]
 params=[p for p in model.parameters() if p.requires_grad];opt=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=cfg['weight_decay']);scaler=_scaler(device,cfg)
 cp=_checkpoint(path,key);cursor=updates=overflows=0;loss_sum=0.
 if cp:
  if cp['query_order']!=keys:raise ValueError('Gemma training order changed')
  set_adapter_state(model,cp['adapter']);opt.load_state_dict(cp['optimizer']);scaler.load_state_dict(cp['scaler'])
  cursor=int(cp['cursor']);updates=int(cp['updates']);overflows=int(cp['overflows']);loss_sum=float(cp['loss_sum']);_restore_rng(cp['rng'],device)
  if cursor<0 or cursor>len(keys) or (cursor%cfg['accum'] and cursor!=len(keys)):raise ValueError('Invalid committed optimizer cursor')
 else:
  torch.random.default_generator.manual_seed(cfg['seed']+fold)
  if str(device).startswith('cuda'):torch.cuda.manual_seed(cfg['seed']+fold)
 total_steps=math.ceil(len(keys)/cfg['accum']);started=time.monotonic();initial_cursor=cursor;last_save=started;model.train()
 def save():_atomic_torch(path,dict(signature=key,adapter=adapter_state(model),optimizer=_cpu_tree(opt.state_dict()),scaler=scaler.state_dict(),
   rng=_rng(device),cursor=cursor,updates=updates,overflows=overflows,loss_sum=loss_sum,query_order=keys,complete=cursor==len(keys)))
 while cursor<len(keys):
  _check(budget);end=min(cursor+cfg['accum'],len(keys));opt.zero_grad(set_to_none=True);step=cursor//cfg['accum'];warm=max(1,int(total_steps*cfg['warmup']))
  factor=(step+1)/warm if step<warm else max(0.,(total_steps-step)/max(1,total_steps-warm))
  for group in opt.param_groups:group['lr']=cfg['lr']*factor
  window=0.
  for j in range(cursor,end):
   _check(budget);rows=store.groups[keys[j]];labels=torch.as_tensor(y[rows],device=device,dtype=torch.float32)
   with torch.autocast(device_type=torch.device(device).type,dtype=torch.float16,enabled=str(device).startswith('cuda')):
    scores=_query_scores(model,store,rows,cfg,device);loss=gemma_rank_loss(scores,labels,teacher[rows],cfg)
   if not torch.isfinite(loss):raise FloatingPointError('Nonfinite Gemma LoRA loss')
   scaler.scale(loss/(end-cursor)).backward();window+=float(loss.detach())
  scaler.unscale_(opt);norm=torch.nn.utils.clip_grad_norm_(params,1.);before=float(scaler.get_scale())
  if not torch.isfinite(norm) and not str(device).startswith('cuda'):raise FloatingPointError('Nonfinite CPU gradient')
  scaler.step(opt);scaler.update()
  if float(scaler.get_scale())<before:
   overflows+=1
   if overflows>cfg['max_overflow_windows']:raise FloatingPointError('Repeated Gemma FP16 gradient overflow')
  else:updates+=1
  cursor=end;loss_sum+=window;opt.zero_grad(set_to_none=True)
  if cursor%32==0 or cursor==len(keys) or time.monotonic()-last_save>=30:
   save();elapsed=time.monotonic()-started;eta=(len(keys)-cursor)*elapsed/max(1,cursor-initial_cursor)
   print(f'Gemma LoRA fold{fold} TRAIN {cursor}/{len(keys)} queries; {elapsed/60:.1f}min; ETA {eta/60:.1f}min; updates={updates}, overflow={overflows}',flush=True);last_save=time.monotonic()
 if updates<1:raise FloatingPointError('No successful Gemma training updates')
 save();return dict(train_queries=len(keys),updates=updates,overflows=overflows,mean_loss=loss_sum/max(1,len(keys)))

def normalize_known_scores(frame,prediction,rows):
 """Normalize complete finite query groups only; unscored rows stay NaN."""
 rows=np.asarray(rows,dtype=np.int64);values=np.asarray(prediction)[rows]
 if not np.isfinite(values).all():raise ValueError('Cannot normalize incomplete source groups')
 sub=frame.iloc[rows].reset_index(drop=True);total=frame.groupby('query_id').size();count=sub.groupby('query_id').size()
 if not np.array_equal(total.loc[count.index].to_numpy(),count.to_numpy()):raise ValueError('Normalization requires every candidate of each known query')
 s=pd.Series(values,dtype=float);g=s.groupby(sub.query_id.astype(str).to_numpy(),sort=False)
 z=((s-g.transform('mean'))/(g.transform('std').fillna(0)+1e-6)).to_numpy(np.float32)
 out=np.full(len(frame),np.nan,np.float32);out[rows]=z;return out


def run_lora_gemma(tr,te,train_rows,cache_dir,budget,config=None,device='cuda'):
 cfg={**CONFIG,**(config or {})};tr,te,full=_clean(tr,te);root=Path(cache_dir);root.mkdir(parents=True,exist_ok=True)
 fkey,_=_signature(full,_frozen_cfg(cfg),'frozen');teacher=_score_cache(root/'frozen_scores.npz',full.id.to_numpy(dtype=str),fkey)
 if not np.isfinite(teacher).all():raise ValueError('Gemma LoRA requires the completed frozen score cache')
 dev,audit,folds=make_lora_folds(tr,train_rows)
 if not tr.iloc[dev].relevance.isin([0,1]).all():raise ValueError('Binary development labels required')
 y=np.zeros(len(full),np.float32);y[dev]=tr.iloc[dev].relevance.to_numpy(np.float32)
 extra=dict(development=_array_digest(dev),audit=_array_digest(audit),development_labels=_array_digest(y[dev]),
  frozen_teacher=_array_digest(teacher),folds=[dict(a=_array_digest(a),b=_array_digest(b)) for a,b in folds])
 request,payload=_signature(full,cfg,'lora-request',extra);planpath=root/'lora_plan.json'
 plan=json.loads(planpath.read_text()) if planpath.is_file() else None
 if plan and plan['request_key']!=request:raise ValueError('Gemma LoRA request inputs/config changed; use new directory')
 resolved=plan['config'] if plan else dict(cfg);key=plan['signature'] if plan else None
 diag=dict(status='initializing',kind='trained',phase='lora',model=MODEL_NAME,revision=MODEL_REVISION,
  no_audit_training=True,development_queries=int(tr.iloc[dev].query_id.nunique()),audit_queries=int(tr.iloc[audit].query_id.nunique()),
  score_scope='per-query z for each source model; development OOF; audit/test mean of two audit-blind models',folds=[])
 final=root/'lora_complete.npz'
 def complete():
  if not final.is_file() or key is None:return None
  with np.load(final,allow_pickle=False) as z:
   if str(z['signature'].item())!=key or not np.array_equal(z['train_ids'],tr.id.to_numpy(dtype=str)) or not np.array_equal(z['test_ids'],te.id.to_numpy(dtype=str)):raise ValueError('Completed Gemma LoRA cache changed')
   a,b=z['train'].copy(),z['test'].copy();hashes=z['weight_hashes'].astype(str)
  if hashes.shape!=(2,) or a.shape!=(len(tr),) or b.shape!=(len(te),) or not np.isfinite(a).all() or not np.isfinite(b).all():raise ValueError('Malformed complete Gemma LoRA scores')
  for k,h in enumerate(hashes):
   path=root/f'gemma_fold{k}.safetensors'
   if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest()!=h:raise ValueError('Gemma adapter/head weights changed')
  return a,b
 def finish(status):
  diag.update(status=status,resolved_config=resolved);jj._atomic_json(root/'lora_diagnostics.json',diag)
  pair=complete();return ({'gemma_lora':pair} if pair is not None else {}),diag
 if complete() is not None:return finish('complete_cached')
 model=store=None
 try:
  if not plan:
   for block in [cfg['pair_block']]+([1] if cfg['pair_block']>1 else []):
    resolved={**cfg,'pair_block':block}
    try:
     _check(budget);model,tok=_load_model(device,True,resolved);store=PairStore(full,tok,resolved)
     bench=_bench_train(model,store,sorted(set(tr.iloc[dev].query_id)),teacher,resolved,budget,device);break
    except torch.cuda.OutOfMemoryError:
     if block==1:raise
     model=store=None;gc.collect()
     if torch.cuda.is_available():torch.cuda.empty_cache()
     print('Gemma training preflight OOM: retry WHOLE query with pair_block=1',flush=True)
   key=hashlib.sha256((request+json.dumps(resolved,sort_keys=True)).encode()).hexdigest()
   plan=dict(request_key=request,signature=key,config=resolved,benchmark=bench,payload=payload,
    policy='Frozen phase complete first. Two DEV-only component folds, one epoch, no audit labels.')
   jj._atomic_json(planpath,plan)
  else:bench=plan['benchmark']
  diag['benchmark']=bench
  # Count the actual remaining work, including all OOF, audit and two-model test pairs.
  ntrain=0;neval=0
  for k,(a,b) in enumerate(folds):
   keys=sorted(set(tr.iloc[a].query_id));cp=_checkpoint(root/f'gemma_fold{k}_train.pt',key)
   ntrain+=max(0,len(keys)-(int(cp['cursor']) if cp else 0))
   rows=np.unique(np.r_[b,audit,np.arange(len(tr),len(full))]);done=0;sp=root/f'gemma_fold{k}_scores.npz'
   if sp.is_file():
    with np.load(sp,allow_pickle=False) as z:
     if not np.array_equal(z['ids'],full.id.to_numpy(dtype=str)):raise ValueError('Partial Gemma score IDs changed')
     done=int(np.isfinite(z['prediction'][rows]).sum())
   neval+=len(rows)-done
  eta=resolved['safety']*(ntrain*bench['train_seconds_per_query']+neval/max(bench['inference_pairs_per_second'],1e-9))
  diag.update(estimated_complete_seconds=eta,available_seconds=jj._available(budget))
  print(f'Gemma LoRA WHOLE remaining stage: {ntrain} train queries + {neval} validation/audit/test pairs; ETA {eta/60:.1f}min; available {jj._available(budget)/60:.1f}min',flush=True)
  if eta>jj._available(budget):return finish('refused_eta')
  model=store=None;gc.collect()
  if torch.cuda.is_available():torch.cuda.empty_cache()
  predictions=[];hashes=[]
  for k,(a,b) in enumerate(folds):
   _check(budget);model,tok=_load_model(device,True,{**resolved,'seed':resolved['seed']+k});store=PairStore(full,tok,resolved)
   wp=root/f'gemma_fold{k}.safetensors';cp=root/f'gemma_fold{k}_train.pt'
   if wp.is_file():
    with safe_open(str(wp),framework='pt',device='cpu') as handle:meta=handle.metadata() or {}
    if meta.get('signature')!=key or meta.get('fold')!=str(k) or meta.get('revision')!=MODEL_REVISION:raise ValueError('Gemma trained weight metadata changed')
    set_adapter_state(model,load_file(str(wp),device='cpu'));record=dict(cached_training=True)
   else:
    record=_train(model,store,sorted(set(tr.iloc[a].query_id)),y,teacher,k,resolved,cp,key,budget,device)
    save_file(adapter_state(model),str(wp)+'.tmp',metadata=dict(signature=key,fold=str(k),revision=MODEL_REVISION))
    os.replace(str(wp)+'.tmp',wp)
   whash=hashlib.sha256(wp.read_bytes()).hexdigest();hashes.append(whash)
   scorekey=hashlib.sha256((key+str(k)+whash).encode()).hexdigest();scorepath=root/f'gemma_fold{k}_scores.npz'
   p=_score_cache(scorepath,store.ids,scorekey);rows=np.unique(np.r_[b,audit,np.arange(len(tr),len(full))])
   p,bs=_infer(model,store,rows,resolved['eval_body'],p,scorepath,scorekey,budget,device,bench['inference_batch'],f'Gemma LoRA fold{k} OOF/audit/test')
   if not np.isfinite(p[rows]).all():raise GemmaBudget('Gemma source inference incomplete')
   predictions.append(normalize_known_scores(full,p,rows));record.update(fold=k,inference_batch=bs,trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad));diag['folds'].append(record)
   del model,store;gc.collect()
   if torch.cuda.is_available():torch.cuda.empty_cache()
  result=np.full(len(tr),np.nan,np.float32)
  for k,(_,b) in enumerate(folds):result[b]=predictions[k][b]
  result[audit]=np.mean([p[audit] for p in predictions],axis=0)
  test=np.mean([p[len(tr):] for p in predictions],axis=0).astype(np.float32)
  if not np.isfinite(result).all() or not np.isfinite(test).all():raise ValueError('Incomplete Gemma source score pair')
  jj._atomic_npz(final,train_ids=tr.id.to_numpy(dtype=str),test_ids=te.id.to_numpy(dtype=str),train=result,test=test,
    weight_hashes=np.asarray(hashes,dtype=str),signature=np.asarray(key))
  return finish('complete')
 except (GemmaBudget,jj.JointBudget) as e:diag['reason']=str(e);return finish('budget_stopped')
 except (FloatingPointError,torch.cuda.OutOfMemoryError) as e:diag['reason']=str(e);return finish('numerical_or_oom_stopped')
 finally:
  model=store=None;gc.collect()
  if torch.cuda.is_available():torch.cuda.empty_cache()

def run_gemma(tr,te,train_rows,cache_dir,budget,run_lora=True,config=None,device='cuda'):
 pairs,diag=run_frozen_gemma(tr,te,cache_dir,budget,config,device)
 report=dict(frozen=diag)
 if run_lora and 'gemma_frozen' in pairs:
  added,ld=run_lora_gemma(tr,te,train_rows,cache_dir,budget,config,device);pairs.update(added);report['lora']=ld
 return pairs,report


In [ ]:
%%writefile giga_encoder_v11.py
"""Pinned Russian Giga embeddings with atomic unique-item checkpoints.

The encoder is frozen. Queries alone receive the model-card instruction;
documents receive title and passage. All non-padding tokens participate in
FP32 mean pooling followed by L2 normalization. Only complete phases return
pair-aligned vectors; partial unique vectors remain resumable on disk.
"""
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import time
import uuid

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from transformers import AutoModel, AutoTokenizer

MODEL_NAME='ai-sage/Giga-Embeddings-instruct-3B-0826'
MODEL_REVISION='b71168088212f0a13688514e4fc288a86106c4e4'
QUERY_PREFIX='Instruct: Given a query, retrieve relevant passages\nQuery: '
CONFIG=dict(version='giga-unique-encoder-v11-1',dimension=2048,
            max_query_tokens=192,max_doc_tokens=768,batch_size=8,
            benchmark_items=64,checkpoint_items=256,checkpoint_seconds=30.,
            safety=1.35,query_prefix=QUERY_PREFIX,
            document_format='title_c + newline + text_c',
            pooling='all non-padding tokens; FP32 mean; row L2',
            add_special_tokens=True,padding_side='right')
FORMAT_VERSION=1

class GigaBudget(RuntimeError):pass

def _available(budget):
    if hasattr(budget,'for_neural'):return max(0.,float(budget.for_neural()))
    if hasattr(budget,'remaining'):return max(0.,float(budget.remaining())-1500.)
    raise TypeError('Budget must expose for_neural() or remaining()')

def _check(budget):
    if _available(budget)<10:raise GigaBudget('CPU/save reserve reached; incomplete unique vectors stay cached')

def _sync(device):
    if str(device).startswith('cuda'):torch.cuda.synchronize()

def _atomic_json(path,value):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,sort_keys=True,ensure_ascii=False,indent=2),encoding='utf8')
    os.replace(tmp,path)

def _atomic_npz(path,**values):
    path=Path(path);tmp=path.with_name(path.name+'.tmp')
    with tmp.open('wb') as handle:np.savez_compressed(handle,**values)
    os.replace(tmp,path)

def _strings_digest(values):
    h=hashlib.sha256()
    for v in values:
        b=str(v).encode('utf8');h.update(len(b).to_bytes(8,'little'));h.update(b)
    return h.hexdigest()

def _config(config=None):
    if set(config or {})-set(CONFIG):raise ValueError('Unknown Giga encoder configuration key')
    cfg={**CONFIG,**(config or {})}
    for k in ['dimension','max_query_tokens','max_doc_tokens','batch_size','benchmark_items','checkpoint_items']:
        if not isinstance(cfg[k],(int,np.integer)) or int(cfg[k])<1:raise ValueError('Positive integer required: '+k)
        cfg[k]=int(cfg[k])
    if not np.isfinite(cfg['safety']) or cfg['safety']<1:raise ValueError('Safety must be finite and at least one')
    if cfg['checkpoint_seconds']<=0:raise ValueError('Positive checkpoint time required')
    cfg['safety']=float(cfg['safety']);cfg['checkpoint_seconds']=float(cfg['checkpoint_seconds'])
    if not isinstance(cfg['query_prefix'],str) or not cfg['query_prefix']:raise ValueError('A nonempty query instruction prefix is required')
    if cfg['padding_side']!='right' or cfg['add_special_tokens'] is not True:
        raise ValueError('This pinned implementation uses official special tokens and right padding')
    return cfg

def _frame(frame):
    f=frame.reset_index(drop=True).copy()
    required={'id','query_c','title_c','text_c'}
    if not required<=set(f) or not len(f) or f.id.isna().any() or not f.id.is_unique:
        raise ValueError('Giga needs nonempty canonical data and present unique pair IDs')
    f['id']=f.id.astype(str)
    if not f.id.is_unique:raise ValueError('Pair IDs collide after string conversion')
    for c in ['query_c','title_c','text_c']:f[c]=f[c].fillna('').astype(str)
    if f.query_c.str.strip().eq('').any():raise ValueError('Empty canonical query')
    return f

def role_texts(frame,config=None):
    """Return exact encoder strings, without reading labels or numeric IDs."""
    cfg=_config(config);f=_frame(frame)
    return {'query':[cfg['query_prefix']+s for s in f.query_c],
            'document':[t+'\n'+x for t,x in zip(f.title_c,f.text_c)]}

def _unique(texts,role):
    mapping={};unique=[];keys=[];rows=[]
    for text in texts:
        if text not in mapping:
            mapping[text]=len(unique);unique.append(text)
            keys.append(hashlib.sha256((role+'\0'+text).encode('utf8')).hexdigest())
        rows.append(mapping[text])
    if len(set(keys))!=len(keys):raise ValueError('Unexpected unique-text hash collision')
    return unique,np.asarray(keys,dtype=str),np.asarray(rows,dtype=np.int64)

def count_unique_items(frame,config=None):
    texts=role_texts(frame,config)
    return {role+'_items':len(set(values)) for role,values in texts.items()}

def mean_pool(hidden,attention_mask):
    """Model-card pooling including instruction/BOS/EOS, excluding padding."""
    if hidden.ndim!=3 or tuple(attention_mask.shape)!=tuple(hidden.shape[:2]):raise ValueError('Embedding/mask shape mismatch')
    count=attention_mask.sum(-1,keepdim=True)
    if torch.any(count<=0):raise ValueError('Cannot pool an empty padded sequence')
    if torch.any((attention_mask!=0)&(attention_mask!=1)):raise ValueError('Pooling mask must contain only zero/one values')
    # Mask before the reduction so irrelevant padded NaNs cannot contaminate
    # valid tokens through IEEE NaN*0 arithmetic. Valid-token NaNs still fail.
    values=hidden.float().masked_fill(attention_mask.eq(0).unsqueeze(-1),0.)
    pooled=values.sum(1)/count.float()
    norm=pooled.norm(p=2,dim=-1,keepdim=True)
    if not torch.isfinite(pooled).all() or torch.any(norm<1e-10):raise FloatingPointError('Giga pooled nonfinite/zero vectors')
    return F.normalize(pooled,p=2,dim=-1,eps=1e-10)

def _load_model(device='cuda',config=None):
    cfg=_config(config)
    tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,
        padding_side='right',trust_remote_code=True)
    dtype=torch.float16 if str(device).startswith('cuda') else torch.float32
    model=AutoModel.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,code_revision=MODEL_REVISION,
        trust_remote_code=True,torch_dtype=dtype,attn_implementation='sdpa',
        low_cpu_mem_usage=True,device_map={'':str(device)}).eval()
    if int(model.config.hidden_size)!=int(cfg['dimension']):raise ValueError('Pinned Giga embedding width changed')
    if any(bool(layer.self_attn.is_causal) for layer in model.layers):raise ValueError('Expected genuinely bidirectional Giga attention')
    for p in model.parameters():p.requires_grad_(False)
    return model,tokenizer

def encode_batch(model,tokenizer,texts,max_length,device='cuda'):
    """Call ONLY supported encoder kwargs: remote class hardcodes use_cache."""
    batch=tokenizer(list(texts),padding=True,truncation=True,max_length=int(max_length),
        pad_to_multiple_of=8,return_tensors='pt',return_token_type_ids=False,add_special_tokens=True)
    ids=batch['input_ids'].to(device);mask=batch['attention_mask'].to(device)
    with torch.inference_mode():
        output=model(input_ids=ids,attention_mask=mask)
        result=mean_pool(output.last_hidden_state,mask).cpu().numpy().astype(np.float32)
    if result.shape[0]!=len(texts) or not np.isfinite(result).all():raise FloatingPointError('Incomplete/nonfinite Giga embedding batch')
    return result

def _binding(frame,texts,keys,cfg,phase,device):
    payload=dict(format_version=FORMAT_VERSION,model=MODEL_NAME,revision=MODEL_REVISION,
        phase=phase,config=cfg,requested_dtype='float16' if str(device).startswith('cuda') else 'float32',
        pair_ids=_strings_digest(frame.id),role_text={r:_strings_digest(v) for r,v in texts.items()},
        role_unique_keys={r:_strings_digest(v) for r,v in keys.items()},
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy','safetensors']})
    key=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()
    return key,payload

def _validate_vectors(values,n,dim):
    if values.shape!=(n,dim) or values.dtype!=np.float32 or not np.isfinite(values).all():
        raise ValueError('Malformed complete Giga chunk')
    norms=np.sqrt(np.einsum('ij,ij->i',values,values,dtype=np.float64))
    if np.any(np.abs(norms-1)>2e-3):raise ValueError('Giga cached vectors are not row-normalized')

def _open_role(root,role,keys,signature,dim):
    directory=Path(root)/role;directory.mkdir(parents=True,exist_ok=True)
    state=dict(directory=directory,manifest=directory/'manifest.json',role=role,keys=keys,
        signature=signature,dimension=int(dim),values=np.full((len(keys),int(dim)),np.nan,np.float32),
        done=np.zeros(len(keys),bool),pending=[],chunks=[])
    path=state['manifest']
    if not path.is_file():return state
    m=json.loads(path.read_text())
    if (m.get('format_version')!=FORMAT_VERSION or m.get('signature')!=signature or m.get('role')!=role
            or m.get('dimension')!=int(dim) or m.get('items')!=len(keys)
            or m.get('keys_digest')!=_strings_digest(keys)):
        raise ValueError('Giga chunk manifest source/config changed; use a new cache directory')
    lookup={str(k):i for i,k in enumerate(keys)}
    for name in m['chunks']:
        if Path(name).name!=name or not name.startswith('chunk_') or not name.endswith('.npz'):raise ValueError('Unsafe Giga chunk filename')
        with np.load(directory/name,allow_pickle=False) as z:
            if str(z['signature'].item())!=signature or str(z['role'].item())!=role:raise ValueError('Giga chunk binding changed')
            item_keys=z['item_keys'].astype(str);vectors=z['vectors'].copy()
        _validate_vectors(vectors,len(item_keys),int(dim))
        if len(set(item_keys))!=len(item_keys) or not set(item_keys)<=set(lookup):raise ValueError('Unknown/duplicate Giga chunk keys')
        rows=np.asarray([lookup[k] for k in item_keys],np.int64)
        if state['done'][rows].any():raise ValueError('A Giga item appears in multiple chunks')
        state['values'][rows]=vectors.astype(np.float32);state['done'][rows]=True;state['chunks'].append(name)
    if int(m['complete_items'])!=int(state['done'].sum()):raise ValueError('Giga manifest item count differs from committed chunks')
    return state

def _commit(state):
    if not state['pending']:return
    rows=np.asarray(state['pending'],dtype=np.int64)
    if len(set(rows))!=len(rows):raise ValueError('Duplicate pending Giga items')
    values=state['values'][rows];_validate_vectors(values,len(rows),state['dimension'])
    name='chunk_'+uuid.uuid4().hex+'.npz'
    _atomic_npz(state['directory']/name,signature=np.asarray(state['signature']),role=np.asarray(state['role']),
                item_keys=state['keys'][rows],vectors=values)
    chunks=state['chunks']+[name]
    _atomic_json(state['manifest'],dict(format_version=FORMAT_VERSION,signature=state['signature'],role=state['role'],
        dimension=state['dimension'],items=len(state['keys']),keys_digest=_strings_digest(state['keys']),
        complete_items=int(state['done'].sum()),chunks=chunks))
    state['chunks']=chunks;state['pending']=[]

def _lengths(tokenizer,texts,cap,budget):
    result=[]
    for i,text in enumerate(texts):
        if i%64==0:_check(budget)
        ids=tokenizer.encode(text,add_special_tokens=True,truncation=True,max_length=int(cap))
        result.append(len(ids))
    return np.asarray(result,np.int64)

def _run_items(state,indices,texts,lengths,model,tokenizer,cap,cfg,budget,device,batch_size,label,store=True):
    ix=np.asarray(indices,np.int64)
    if store:ix=ix[~state['done'][ix]]
    if not len(ix):return int(batch_size)
    ordered=ix[np.argsort(lengths[ix],kind='stable')]
    i=last_count=0;bs=int(batch_size);started=last=time.monotonic()
    try:
        while i<len(ordered):
            _check(budget);rows=ordered[i:i+bs];values=None
            try:
                values=encode_batch(model,tokenizer,[texts[j] for j in rows],cap,device)
            except torch.cuda.OutOfMemoryError:
                values=None;gc.collect()
                if torch.cuda.is_available():torch.cuda.empty_cache()
                if bs<=1:raise
                bs=max(1,bs//2);print(f'{label}: OOM, retry batch={bs}',flush=True);continue
            _validate_vectors(values,len(rows),state['dimension'])
            if store:
                state['values'][rows]=values;state['done'][rows]=True;state['pending'].extend(rows.tolist())
            i+=len(rows)
            if i-last_count>=cfg['checkpoint_items'] or time.monotonic()-last>=cfg['checkpoint_seconds'] or i==len(ordered):
                if store:_commit(state)
                elapsed=time.monotonic()-started;eta=(len(ordered)-i)*elapsed/max(1,i)
                print(f'{label}: {i}/{len(ordered)} unique items; elapsed {elapsed/60:.1f} min; ETA {eta/60:.1f} min',flush=True)
                last=time.monotonic();last_count=i
    finally:
        if store:_commit(state)
    return bs

def _representatives(indices,lengths,count):
    indices=np.asarray(indices,np.int64)
    order=indices[np.argsort(lengths[indices],kind='stable')]
    return order[np.unique(np.linspace(0,len(order)-1,min(int(count),len(order))).astype(int))]

def _benchmark_role(state,texts,lengths,model,tokenizer,cap,cfg,budget,device,future_items):
    pending=np.flatnonzero(~state['done']);bs=cfg['batch_size'];readonly=not len(pending)
    if readonly and not future_items:return None,bs,False
    candidates=np.arange(len(texts)) if readonly else pending
    probe=_representatives(candidates,lengths,cfg['benchmark_items'])
    # Do not consume all pending work in warm-up, so tiny fixtures still have
    # representative timed work. Readonly benchmarks never alter native caches.
    warm=probe[len(probe)//2:len(probe)//2+min(4,max(1,len(probe)//4))]
    bs=_run_items(state,warm,texts,lengths,model,tokenizer,cap,cfg,budget,device,bs,
                  'Giga '+state['role']+' warmup',store=not readonly)
    if not readonly:
        probe=probe[~state['done'][probe]]
        if not len(probe):
            readonly=True;probe=_representatives(np.arange(len(texts)),lengths,cfg['benchmark_items'])
    _sync(device);start=time.monotonic()
    bs=_run_items(state,probe,texts,lengths,model,tokenizer,cap,cfg,budget,device,bs,
                  'Giga '+state['role']+' benchmark',store=not readonly)
    _sync(device);rate=len(probe)/max(time.monotonic()-start,1e-6)
    return rate,bs,readonly

def encode_giga(frame,cache_dir,budget,phase='train',config=None,device='cuda',future_workload=None):
    """Return complete pair-aligned {'query','document'} arrays and diagnostics.

    TRAIN and TEST must use separate cache directories. ``future_workload``
    contains query_items/document_items counts only, and changes budgeting,
    never model prompts/cache identity. A complete phase replays without GPU.
    """
    cfg=_config(config);f=_frame(frame)
    if phase not in ['train','test']:raise ValueError('Phase must be train or test')
    future=dict(future_workload or {})
    if set(future)-{'query_items','document_items'}:raise ValueError('Unknown future-workload role')
    for k,v in future.items():
        if not isinstance(v,(int,np.integer)) or int(v)<0:raise ValueError('Future item counts must be nonnegative integers')
    text_rows=role_texts(f,cfg);texts={};keys={};maps={}
    for role,values in text_rows.items():texts[role],keys[role],maps[role]=_unique(values,role)
    signature,payload=_binding(f,text_rows,keys,cfg,phase,device)
    del text_rows
    root=Path(cache_dir);root.mkdir(parents=True,exist_ok=True);path=root/'input_manifest.json'
    if path.is_file() and json.loads(path.read_text()).get('signature')!=signature:
        raise ValueError('Giga phase/model/inputs/config changed; choose a new cache directory')
    _atomic_json(path,dict(signature=signature,**payload))
    states={r:_open_role(root,r,keys[r],signature,cfg['dimension']) for r in ['query','document']}
    diag=dict(status='initializing',model=MODEL_NAME,revision=MODEL_REVISION,phase=phase,signature=signature,
              config=cfg,pairs=len(f),unique_items={r:len(keys[r]) for r in keys},future_workload={k:int(v) for k,v in future.items()},
              source_kind='zero_shot',pooling=cfg['pooling'])
    def finish(status):
        for state in states.values():_commit(state)
        diag['status']=status;diag['complete_items']={r:int(s['done'].sum()) for r,s in states.items()}
        _atomic_json(root/'encoder_diagnostics.json',diag)
        if status in ['refused_eta','budget_stopped','numerical_or_oom_stopped']:return {},diag
        if not all(s['done'].all() for s in states.values()):return {},diag
        result={r:states[r]['values'][maps[r]].copy() for r in ['query','document']}
        return result,diag
    if all(s['done'].all() for s in states.values()):return finish('complete_cached')
    model=tokenizer=None
    try:
        _check(budget);model,tokenizer=_load_model(device,cfg)
        caps={'query':cfg['max_query_tokens'],'document':cfg['max_doc_tokens']};lengths={};rates={};batches={};readonly=[]
        for role in ['query','document']:
            print(f'Giga {phase}: preparing {len(texts[role])} unique {role} texts',flush=True)
            lengths[role]=_lengths(tokenizer,texts[role],caps[role],budget)
            rate,bs,ro=_benchmark_role(states[role],texts[role],lengths[role],model,tokenizer,caps[role],cfg,budget,device,int(future.get(role+'_items',0)))
            rates[role]=rate;batches[role]=bs
            if ro and rate is not None:readonly.append(role)
        phase_eta=future_eta=0.
        for role in ['query','document']:
            missing=int((~states[role]['done']).sum());rate=max(rates[role] or 1.,1e-9)
            phase_eta+=missing/rate;future_eta+=int(future.get(role+'_items',0))/rate
        phase_eta*=cfg['safety'];future_eta*=cfg['safety'];whole_eta=phase_eta+future_eta
        diag.update(benchmark_rates=rates,batch_sizes=batches,readonly_rebenchmarks=readonly,
                    estimated_remaining_seconds=phase_eta,future_eta_seconds=future_eta,
                    estimated_whole_seconds=whole_eta,available_neural_seconds=_available(budget))
        print(f'Giga {phase}: phase ETA {phase_eta/60:.1f} min; with future test {whole_eta/60:.1f} min; available {_available(budget)/60:.1f} min',flush=True)
        if whole_eta>_available(budget):return finish('refused_eta')
        for role in ['query','document']:
            batches[role]=_run_items(states[role],np.arange(len(texts[role])),texts[role],lengths[role],model,tokenizer,
                caps[role],cfg,budget,device,batches[role],f'Giga {phase} {role}')
        diag['batch_sizes']=batches;return finish('complete')
    except GigaBudget as exc:diag['reason']=str(exc);return finish('budget_stopped')
    except (FloatingPointError,torch.cuda.OutOfMemoryError) as exc:diag['reason']=str(exc);return finish('numerical_or_oom_stopped')
    finally:
        model=tokenizer=None;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


In [ ]:
%%writefile giga_heads_v11.py
"""Competition-only relevance heads over frozen Giga embeddings.

The encoder never receives gradients. Four source folds cover development only;
one full-development head predicts untouched audit and future test. Returned
learned scores are query-normalized. Downstream ordinary stacking CV remains
approximate, because source OOF features are not nested inside its folds.
"""
from __future__ import annotations
import hashlib,importlib.metadata,json,math,os,time,warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from safetensors import safe_open
from safetensors.torch import save_file,load_file
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from jina_probe import _array_digest,_strings_digest,_atomic_json,_atomic_npz
from meta_frozen_v4 import query_z
from cache_core import ExactNDCG

LINEAR_CONFIG=dict(schema='giga-linear-dev-only-v1',dimension=2048,C=.05,max_iter=600,tol=1e-4,
    solver='lbfgs',class_weight=None,features=['q_times_d','abs_q_minus_d','cosine'],
    standardization='fit-on-source-training-fold-only',score_normalization='query-z',seed=42)
RANK_CONFIG=dict(schema='giga-role-residual-rank-v1',dimension=2048,hidden=64,dropout=.1,
    epochs=4,query_batch_size=32,lr=3e-4,weight_decay=.05,scale=20.,bce_weight=.1,
    residual_l2=.01,positive_prior=.26,seed=42,max_seconds=900.,score_normalization='query-z')
HEAD_SCHEMA='giga-heads-v11-dev-only-v1'

class HeadsBudget(RuntimeError):pass

def _remaining(budget):
    if budget is None:return math.inf
    if not hasattr(budget,'remaining'):raise TypeError('Head budget must expose remaining()')
    return max(0.,float(budget.remaining())-600.)

def _check(deadline):
    if time.monotonic()>=deadline:raise HeadsBudget('Reserve ten minutes for final fusion/save; incomplete head stays cached')

def _file_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def _json(value):return json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':'))
def _key(value):return hashlib.sha256(_json(value).encode()).hexdigest()

def _vectors(q,d,n):
    arrays=[];digests={};dimension=int(LINEAR_CONFIG['dimension'])
    if int(RANK_CONFIG['dimension'])!=dimension:raise ValueError('Head dimensions disagree')
    for name,value in [('query',q),('document',d)]:
        a=np.asarray(value)
        if a.shape!=(n,dimension) or not np.issubdtype(a.dtype,np.floating) or not np.isfinite(a).all():
            raise ValueError(f'{name}: expected complete finite ({n},{dimension}) vectors')
        digests[name]=_array_digest(a)
        norm=np.sqrt(np.einsum('ij,ij->i',a,a,dtype=np.float64))
        if np.any(norm<1e-10):raise ValueError('A zero Giga vector cannot be normalized')
        arrays.append((a/norm[:,None]).astype(np.float32))
    return arrays[0],arrays[1],digests

def _validate_train(tr,folds):
    tr=tr.reset_index(drop=True).copy()
    if not {'id','query_id','grp','relevance'}<=set(tr) or tr.id.isna().any() or not tr.id.is_unique or tr.query_id.isna().any():
        raise ValueError('Complete training IDs, queries, components and labels are required')
    if (tr.groupby('query_id').grp.nunique()!=1).any():raise ValueError('A query has multiple component labels')
    if 'document_id' in tr and (tr.groupby('document_id').grp.nunique()!=1).any():raise ValueError('A document has multiple component labels')
    parts=[(np.asarray(a,np.int64),np.asarray(b,np.int64)) for a,b in folds]
    if len(parts)!=5:raise ValueError('Use the five original connected-component folds')
    for a,b in parts:
        if a.ndim!=1 or b.ndim!=1 or not len(a) or not len(b) or len(np.unique(a))!=len(a) or np.any(a<0) or np.any(a>=len(tr)) or set(a)&set(b):
            raise ValueError('Malformed source fold indices')
    allval=np.concatenate([b for a,b in parts])
    if not np.array_equal(np.sort(allval),np.arange(len(tr))):raise ValueError('Fold validation rows must partition train')
    audit=np.sort(parts[-1][1]);dev=np.setdiff1d(np.arange(len(tr)),audit)
    if set(tr.iloc[dev].grp)&set(tr.iloc[audit].grp) or set(tr.iloc[dev].query_id)&set(tr.iloc[audit].query_id):raise ValueError('A query/component crosses development/audit')
    active=[]
    for a,b in parts[:-1]:
        aa=np.intersect1d(a,dev);bb=np.intersect1d(b,dev)
        if not len(aa) or not len(bb) or set(tr.iloc[aa].grp)&set(tr.iloc[bb].grp) or set(tr.iloc[aa].query_id)&set(tr.iloc[bb].query_id):
            raise ValueError('A query/component crosses source folds')
        active.append((aa,bb))
    if not np.array_equal(np.sort(np.concatenate([b for a,b in active])),dev):raise ValueError('Source folds must cover all development rows')
    if not tr.iloc[dev].relevance.isin([0,1]).all() or tr.iloc[dev].relevance.nunique()!=2:raise ValueError('Development requires both binary classes')
    if any(tr.iloc[a].relevance.nunique()!=2 for a,b in active):raise ValueError('Each source-training fold requires both classes')
    y=np.zeros(len(tr),np.float32);y[dev]=tr.iloc[dev].relevance.to_numpy(np.float32)
    return tr,dev,audit,active,y

def _feature_matrix(q,d):
    out=np.empty((len(q),2*q.shape[1]+1),np.float32)
    dim=q.shape[1];np.multiply(q,d,out=out[:,:dim]);np.subtract(q,d,out=out[:,dim:2*dim]);np.abs(out[:,dim:2*dim],out=out[:,dim:2*dim])
    out[:,-1]=out[:,:dim].sum(1)
    if not np.isfinite(out).all():raise FloatingPointError('Nonfinite Giga probe features')
    return out

def _save_joblib(path,model):
    temp=Path(str(path)+'.tmp');joblib.dump(model,temp,compress=3);os.replace(temp,path)

def _fit_linear(X,y,rows):
    cfg=LINEAR_CONFIG
    model=Pipeline([('scale',StandardScaler(copy=False)),('logistic',LogisticRegression(
        C=cfg['C'],solver=cfg['solver'],max_iter=cfg['max_iter'],tol=cfg['tol'],
        class_weight=cfg['class_weight'],random_state=cfg['seed']))])
    with threadpool_limits(limits=2),warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always',ConvergenceWarning);model.fit(X[np.asarray(rows)],y[np.asarray(rows)])
    convergence=[str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning)]
    for w in caught:
        if not issubclass(w.category,ConvergenceWarning):warnings.warn(str(w.message),w.category,stacklevel=2)
    if convergence:raise RuntimeError('Fixed Giga linear probe did not converge; no incomplete scores published: '+'; '.join(convergence))
    lr=model.named_steps['logistic']
    if not np.isfinite(lr.coef_).all() or not np.isfinite(lr.intercept_).all():raise FloatingPointError('Nonfinite linear coefficients')
    return model,int(np.max(lr.n_iter_))

def _linear_predict(model,X,rows):
    # Fancy indexing copies before StandardScaler(copy=False), preserving X.
    with threadpool_limits(limits=2):out=np.asarray(model.decision_function(X[np.asarray(rows)]),np.float64)
    if out.shape!=(len(rows),) or not np.isfinite(out).all():raise FloatingPointError('Malformed linear scores')
    return out

class RoleResidualMetric(nn.Module):
    def __init__(self,dimension=None,hidden=None):
        super().__init__();cfg=RANK_CONFIG;dim=int(dimension or cfg['dimension']);h=int(hidden or cfg['hidden'])
        def tower():
            net=nn.Sequential(nn.Linear(dim,h,bias=False),nn.GELU(),nn.Dropout(cfg['dropout']),nn.Linear(h,dim,bias=False))
            nn.init.zeros_(net[-1].weight);return net
        self.query=tower();self.document=tower()
    def forward(self,q,d):
        rq=self.query(q);rd=self.document(d)
        aq=F.normalize(q+rq,dim=-1,eps=1e-8);ad=F.normalize(d+rd,dim=-1,eps=1e-8)
        return (aq*ad).sum(-1),.5*(rq.square().mean()+rd.square().mean())

def query_rank_objective(cosines,labels,sizes):
    cfg=RANK_CONFIG;off=0;rank_terms=[];bce_terms=[]
    prior=math.log(cfg['positive_prior']/(1-cfg['positive_prior']))
    for n in sizes:
        if n<1:raise ValueError('Query groups must be nonempty')
        z=cfg['scale']*cosines[off:off+n].float();y=labels[off:off+n].float();off+=n
        if len(z)!=n:raise ValueError('Incomplete query scores')
        bce_terms.append(F.binary_cross_entropy_with_logits(z-z.mean()+prior,y))
        pos=torch.nonzero(y>.5).flatten();neg=torch.nonzero(y<=.5).flatten()
        if not len(pos) or not len(neg):rank_terms.append(z.sum()*0.);continue
        with torch.no_grad():
            order=torch.argsort(z.detach(),descending=True,stable=True);ranks=torch.empty_like(order);ranks[order]=torch.arange(n,device=z.device)
            i=torch.arange(n,device=z.device);disc=(i<10).float()/torch.log2(i.float()+2.)
            ideal=disc[:min(10,len(pos))].sum().clamp_min(1e-8)
            delta=(disc[ranks[pos]][:,None]-disc[ranks[neg]][None,:]).abs()/ideal
        loss=F.softplus(-(z[pos,None]-z[neg][None,:]))
        rank_terms.append((loss*delta).sum()/delta.sum().clamp_min(1e-8))
    if off!=len(cosines) or len(labels)!=len(cosines):raise ValueError('Query sizes do not cover rows')
    return torch.stack(rank_terms).mean()+cfg['bce_weight']*torch.stack(bce_terms).mean()

def _groups(frame,rows):
    rows=np.asarray(rows,np.int64);sub=frame.iloc[rows].reset_index(drop=True)
    return [rows[v] for v in sub.groupby('query_id',sort=False).indices.values()]

def _fit_rank(q,d,y,groups,seed,deadline,device):
    _check(deadline);cfg=RANK_CONFIG;torch.manual_seed(seed)
    if str(device).startswith('cuda'):torch.cuda.manual_seed(seed)
    model=RoleResidualMetric().to(device);opt=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    rng=np.random.default_rng(seed);history=[]
    for epoch in range(cfg['epochs']):
        model.train();order=rng.permutation(len(groups));total=0.;count=0
        for start in range(0,len(order),cfg['query_batch_size']):
            _check(deadline);chosen=[groups[int(i)] for i in order[start:start+cfg['query_batch_size']]];rows=np.concatenate(chosen)
            qq=torch.as_tensor(q[rows],dtype=torch.float32,device=device);dd=torch.as_tensor(d[rows],dtype=torch.float32,device=device)
            yy=torch.as_tensor(y[rows],dtype=torch.float32,device=device);opt.zero_grad(set_to_none=True)
            score,reg=model(qq,dd);loss=query_rank_objective(score,yy,[len(v) for v in chosen])+cfg['residual_l2']*reg
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite Giga rank objective')
            loss.backward();norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            if not torch.isfinite(norm):raise FloatingPointError('Nonfinite Giga rank gradient')
            opt.step();total+=float(loss.detach())*len(chosen);count+=len(chosen)
        history.append(total/max(1,count));print(f'Giga rank epoch{epoch+1}/{cfg["epochs"]}: loss={history[-1]:.6f}',flush=True)
    if not all(torch.isfinite(p).all() for p in model.parameters()):raise FloatingPointError('Nonfinite Giga rank weights')
    return model,history

@torch.no_grad()
def _rank_predict(model,q,d,rows,deadline,device):
    model.eval();out=np.empty(len(rows),np.float32)
    for start in range(0,len(rows),2048):
        _check(deadline);ix=np.asarray(rows[start:start+2048]);a=torch.as_tensor(q[ix],dtype=torch.float32,device=device);b=torch.as_tensor(d[ix],dtype=torch.float32,device=device)
        out[start:start+len(ix)]=model(a,b)[0].cpu().numpy()
    if not np.isfinite(out).all():raise FloatingPointError('Invalid rank predictions')
    return out

def _save_rank(path,model,signature):
    temp=Path(str(path)+'.tmp');save_file({n:p.detach().cpu().contiguous() for n,p in model.state_dict().items()},str(temp),metadata={'signature':signature,'config':_json(RANK_CONFIG)})
    os.replace(temp,path)

def _load_rank(path,signature,device):
    with safe_open(str(path),framework='pt',device='cpu') as f:meta=f.metadata() or {}
    if meta.get('signature')!=signature or meta.get('config')!=_json(RANK_CONFIG):raise ValueError('Rank model metadata changed')
    state=load_file(str(path),device='cpu')
    if not all(torch.isfinite(v).all() for v in state.values()):raise ValueError('Nonfinite cached rank weights')
    model=RoleResidualMetric();model.load_state_dict(state,strict=True);return model.to(device)

def _read_piece(path,model_path,key,ids):
    if not path.is_file():return None
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item())!=key or not np.array_equal(z['ids'],ids):raise ValueError('Giga head score IDs/config changed')
        p=z['prediction'].copy();h=str(z['model_sha256'].item());details=json.loads(str(z['details'].item()))
        if str(z['prediction_digest'].item())!=_array_digest(p):raise ValueError('Giga head score payload changed')
    if p.shape!=(len(ids),) or not np.isfinite(p).all():raise ValueError('Incomplete cached head scores')
    if not model_path.is_file() or _file_hash(model_path)!=h:raise ValueError('Giga head model missing or changed')
    return p,details

def _save_piece(path,model_path,key,ids,p,details):
    p=np.asarray(p,np.float32)
    _atomic_npz(path,ids=np.asarray(ids,dtype=str),prediction=p,prediction_digest=np.asarray(_array_digest(p)),signature=np.asarray(key),
                model_sha256=np.asarray(_file_hash(model_path)),details=np.asarray(_json(details)))

def _verified_models(out,diag):
    for name,head in diag.get('heads',{}).items():
        filename=head.get('model_filename','')
        if Path(filename).name!=filename or not filename:raise ValueError('Invalid Giga head model path')
        path=out/filename
        if not path.is_file() or _file_hash(path)!=head['model_sha256']:raise ValueError('Giga full-development model is missing or changed')

def _cached_train(out,key,ids):
    path=out/'train_scores.npz';report=out/'train_diagnostics.json'
    if not path.is_file() or not report.is_file():return None
    diag=json.loads(report.read_text())
    if diag.get('signature')!=key:raise ValueError('Giga trained-source report signature changed')
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item())!=key or not np.array_equal(z['ids'],ids):raise ValueError('Giga trained-source identities changed')
        names=z['source_names'].astype(str).tolist();scores=z['scores'].copy()
        if str(z['scores_digest'].item())!=_array_digest(scores):raise ValueError('Giga trained-source payload changed')
    if names!=diag.get('train_sources') or scores.shape!=(len(ids),len(names)) or not np.isfinite(scores).all():raise ValueError('Incomplete Giga trained sources')
    if not names or names[0]!='giga_cosine' or len(set(names))!=len(names) or not set(names)<=set(['giga_cosine','giga_linear','giga_rank']):raise ValueError('Unexpected trained source schema')
    _verified_models(out,diag)
    for n in ['linear','rank']:
        if ('giga_'+n in names)!=(n in diag.get('heads',{})):raise ValueError('Head artifact/source schema differs')
    return {n:scores[:,i] for i,n in enumerate(names)},diag

def _source_fit(tr,q,d,X,y,dev,audit,folds,out,key,source,deadline,device):
    isrank=source=='rank';raw=np.full(len(tr),np.nan,np.float32);details=[];elapsed=[]
    for k,(a,b) in enumerate(folds):
        model_path=out/f'{source}_fold{k}.{"safetensors" if isrank else "joblib"}';path=out/f'{source}_fold{k}_scores.npz'
        cached=_read_piece(path,model_path,key,tr.iloc[b].id.to_numpy(dtype=str))
        if cached is not None:p,detail=cached
        else:
            _check(deadline)
            if elapsed and 1.35*np.mean(elapsed)*(len(folds)-k+1)>deadline-time.monotonic():raise HeadsBudget(f'{source} remaining complete folds do not fit')
            print(f'Giga {source} fold{k+1}/{len(folds)}: fitting',flush=True);started=time.monotonic()
            if isrank:
                model,history=_fit_rank(q,d,y,_groups(tr,a),RANK_CONFIG['seed']+k,deadline,device)
                pred=_rank_predict(model,q,d,b,deadline,device);_save_rank(model_path,model,key);detail=dict(history=history)
            else:
                model,n_iter=_fit_linear(X,y,a);pred=_linear_predict(model,X,b);_save_joblib(model_path,model);detail=dict(n_iter=n_iter)
            p=query_z(tr.iloc[b].reset_index(drop=True),pred);detail.update(seconds=time.monotonic()-started,cached=False)
            _save_piece(path,model_path,key,tr.iloc[b].id.to_numpy(dtype=str),p,detail);del model
            elapsed.append(detail['seconds'])
        raw[b]=p;details.append(detail)
        print(f'Giga {source} fold{k+1}: development NDCG={ExactNDCG(tr.iloc[b]).score(p):.6f}',flush=True)
    model_path=out/f'{source}_full_dev.{"safetensors" if isrank else "joblib"}';path=out/f'{source}_audit_scores.npz'
    cached=_read_piece(path,model_path,key,tr.iloc[audit].id.to_numpy(dtype=str))
    if cached is not None:p,detail=cached
    else:
        _check(deadline);started=time.monotonic();print(f'Giga {source}: full-development fit for audit/test',flush=True)
        if isrank:
            model,history=_fit_rank(q,d,y,_groups(tr,dev),RANK_CONFIG['seed']+100,deadline,device)
            pred=_rank_predict(model,q,d,audit,deadline,device);_save_rank(model_path,model,key);detail=dict(history=history)
        else:
            model,n_iter=_fit_linear(X,y,dev);pred=_linear_predict(model,X,audit);_save_joblib(model_path,model);detail=dict(n_iter=n_iter)
        p=query_z(tr.iloc[audit].reset_index(drop=True),pred);detail.update(seconds=time.monotonic()-started,cached=False)
        _save_piece(path,model_path,key,tr.iloc[audit].id.to_numpy(dtype=str),p,detail);del model
    raw[audit]=p
    if not np.isfinite(raw).all():raise ValueError('Head OOF/audit source is incomplete')
    return raw,dict(folds=details,full_dev=detail,model_filename=model_path.name,model_sha256=_file_hash(model_path),
                    development_ndcg=ExactNDCG(tr.iloc[dev]).score(raw[dev]),normalization='query-z')

def fit_giga_heads_train(tr,folds,qtr,dtr,out_dir,budget=None):
    """Return complete train source arrays, without any test embedding input."""
    started=time.monotonic();out=Path(out_dir);out.mkdir(parents=True,exist_ok=True)
    tr,dev,audit,active,y=_validate_train(tr,folds);q,d,digests=_vectors(qtr,dtr,len(tr))
    binding=dict(schema=HEAD_SCHEMA,linear_config=LINEAR_CONFIG,rank_config=RANK_CONFIG,
        versions={n:importlib.metadata.version(n) for n in ['numpy','scikit-learn','joblib','torch','safetensors']},
        embeddings=digests,train_ids=_strings_digest(tr.id),train_query_ids=sorted(tr.query_id.astype(str).unique().tolist()),
        train_queries=_strings_digest(tr.query_id),development=_array_digest(dev),audit=_array_digest(audit),development_labels=_array_digest(y[dev]),
        folds=[dict(train=_array_digest(a),validation=_array_digest(b)) for a,b in active])
    signature=_key(binding);manifest=out/'train_manifest.json'
    if manifest.is_file() and json.loads(manifest.read_text()).get('signature')!=signature:raise ValueError('Giga head data/config/folds changed; select a new directory')
    _atomic_json(manifest,dict(signature=signature,**binding))
    cached=_cached_train(out,signature,tr.id.to_numpy(dtype=str))
    if cached is not None and ('giga_rank' in cached[0] or _remaining(budget)<=60):
        cached[1]['cache_replayed']=True
        return cached
    sources={'giga_cosine':np.einsum('ij,ij->i',q,d).astype(np.float32)}
    diag=dict(schema=HEAD_SCHEMA,signature=signature,kind_by_source={'giga_cosine':'zero_shot'},
        no_audit_training=True,score_scope='development OOF; audit/test full-development head',heads={},
        train_sources=['giga_cosine'],validation_caveat='New heads exclude audit labels. Downstream CV with learned OOF features remains approximate.')
    deadline=time.monotonic()+_remaining(budget);X=None
    if cached is not None and 'giga_linear' in cached[0]:
        sources['giga_linear']=cached[0]['giga_linear'];diag['heads']['linear']=cached[1]['heads']['linear'];diag['kind_by_source']['giga_linear']='trained'
    else:
        try:
            _check(deadline);X=_feature_matrix(q,d)
            p,detail=_source_fit(tr,q,d,X,y,dev,audit,active,out,signature,'linear',deadline,'cpu')
            sources['giga_linear']=p;diag['heads']['linear']=detail;diag['kind_by_source']['giga_linear']='trained'
        except HeadsBudget as exc:diag['linear_status']='budget_stopped';diag['linear_reason']=str(exc)
        except (RuntimeError,FloatingPointError) as exc:diag['linear_status']='numerical_stopped';diag['linear_reason']=str(exc)
    del X
    if 'giga_linear' in sources:
        rank_deadline=time.monotonic()+min(RANK_CONFIG['max_seconds'],_remaining(budget));device='cuda' if torch.cuda.is_available() else 'cpu'
        old_threads=torch.get_num_threads()
        try:
            _check(rank_deadline);torch.set_num_threads(2)
            with threadpool_limits(limits=2):p,detail=_source_fit(tr,q,d,None,y,dev,audit,active,out,signature,'rank',rank_deadline,device)
            sources['giga_rank']=p;diag['heads']['rank']=detail;diag['kind_by_source']['giga_rank']='trained'
        except HeadsBudget as exc:diag['rank_status']='budget_stopped';diag['rank_reason']=str(exc)
        except (FloatingPointError,torch.cuda.OutOfMemoryError) as exc:diag['rank_status']='numerical_stopped';diag['rank_reason']=str(exc)
        finally:
            torch.set_num_threads(old_threads)
            if torch.cuda.is_available():torch.cuda.empty_cache()
    diag.update(status='complete' if 'giga_rank' in sources else ('linear_complete_rank_unavailable' if 'giga_linear' in sources else 'cosine_only'),
                train_sources=list(sources),seconds=time.monotonic()-started)
    matrix=np.column_stack(list(sources.values())).astype(np.float32)
    _atomic_npz(out/'train_scores.npz',ids=tr.id.to_numpy(dtype=str),signature=np.asarray(signature),source_names=np.asarray(list(sources),dtype=str),
                scores=matrix,scores_digest=np.asarray(_array_digest(matrix)))
    _atomic_json(out/'train_diagnostics.json',diag)
    return sources,diag

def predict_giga_heads_test(te,qte,dte,out_dir,train_diag):
    """Use the already fitted full-development heads after a positive gate."""
    te=te.reset_index(drop=True).copy();out=Path(out_dir)
    if not {'id','query_id'}<=set(te) or te.id.isna().any() or not te.id.is_unique or te.query_id.isna().any():raise ValueError('Malformed test IDs/queries')
    manifest=json.loads((out/'train_manifest.json').read_text());saved=json.loads((out/'train_diagnostics.json').read_text())
    signature=manifest['signature']
    if train_diag.get('signature')!=signature or saved.get('signature')!=signature or train_diag.get('train_sources')!=saved.get('train_sources'):raise ValueError('Training head decision changed')
    if manifest['linear_config']!=LINEAR_CONFIG or manifest['rank_config']!=RANK_CONFIG:raise ValueError('Head configs changed')
    if set(te.query_id.astype(str))&set(manifest['train_query_ids']):raise ValueError('Test queries overlap training')
    q,d,digests=_vectors(qte,dte,len(te));hashes={};_verified_models(out,saved)
    for name,head in saved['heads'].items():
        p=out/head['model_filename']
        if Path(head['model_filename']).name!=head['model_filename'] or not p.is_file() or _file_hash(p)!=head['model_sha256']:raise ValueError('Giga full-development model is missing or changed')
        hashes[name]=head['model_sha256']
    test_binding=dict(train=signature,test_ids=_strings_digest(te.id),test_queries=_strings_digest(te.query_id),embeddings=digests,models=hashes,sources=saved['train_sources'])
    test_signature=_key(test_binding)
    cache=out/'test_scores.npz'
    if cache.is_file():
        with np.load(cache,allow_pickle=False) as z:
            old_signature=str(z['signature'].item())
            if not np.array_equal(z['ids'],te.id.to_numpy(dtype=str)):raise ValueError('Giga test score IDs changed')
            names=z['source_names'].astype(str).tolist();scores=z['scores'].copy()
            if str(z['scores_digest'].item())!=_array_digest(scores):raise ValueError('Cached Giga test payload changed')
            previous=json.loads(str(z['binding'].item()))
        if scores.shape!=(len(te),len(names)) or not np.isfinite(scores).all() or names!=previous.get('sources') or _key(previous)!=old_signature:raise ValueError('Incomplete or malformed cached test scores')
        if old_signature==test_signature:
            if names!=saved['train_sources']:raise ValueError('Cached test source schema changed')
            return {n:scores[:,i] for i,n in enumerate(names)},dict(status='complete_cached',signature=test_signature,sources=names)
        # A newly completed optional head may extend an otherwise identical
        # test cache; changed examples/vectors or earlier models still reject.
        stable=[k for k in test_binding if k not in ('models','sources')]
        upgrade=(all(previous.get(k)==test_binding[k] for k in stable)
                 and names==saved['train_sources'][:len(names)]
                 and all(hashes.get(k)==v for k,v in previous.get('models',{}).items()))
        if not upgrade:raise ValueError('Giga test source data/config changed')
    sources={'giga_cosine':np.einsum('ij,ij->i',q,d).astype(np.float32)}
    if 'giga_linear' in saved['train_sources']:
        model=joblib.load(out/saved['heads']['linear']['model_filename']);X=_feature_matrix(q,d)
        raw=_linear_predict(model,X,np.arange(len(te)));sources['giga_linear']=query_z(te,raw);del model,X
    if 'giga_rank' in saved['train_sources']:
        device='cuda' if torch.cuda.is_available() else 'cpu';model=_load_rank(out/saved['heads']['rank']['model_filename'],signature,device)
        raw=_rank_predict(model,q,d,np.arange(len(te)),math.inf,device);sources['giga_rank']=query_z(te,raw);del model
        if torch.cuda.is_available():torch.cuda.empty_cache()
    if list(sources)!=saved['train_sources'] or any(v.shape!=(len(te),) or not np.isfinite(v).all() for v in sources.values()):raise ValueError('Test head source schema/inference incomplete')
    matrix=np.column_stack(list(sources.values())).astype(np.float32)
    _atomic_npz(cache,ids=te.id.to_numpy(dtype=str),signature=np.asarray(test_signature),source_names=np.asarray(list(sources),dtype=str),scores=matrix,scores_digest=np.asarray(_array_digest(matrix)),binding=np.asarray(_json(test_binding)))
    return sources,dict(status='complete',signature=test_signature,sources=list(sources),no_test_labels_used=True)


In [ ]:
%%writefile giga_lora_v13.py
"""Budget-gated, audit-blind two-fold training of the pinned Giga encoder.

Only LoRA q/v matrices in the last eight bidirectional layers are learned.
A complete source contains development OOF cosines and the raw-cosine mean of
both audit/test predictions, normalized within queries only after combination.
Unique inference embeddings and optimizer-boundary training states resume
atomically. No incomplete source is returned.
"""
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.model_selection import GroupKFold
from transformers import AutoModel, AutoTokenizer
import giga_encoder_v11 as ge

MODEL_NAME = ge.MODEL_NAME
MODEL_REVISION = ge.MODEL_REVISION
QUERY_PREFIX = ge.QUERY_PREFIX
CONFIG = dict(version='giga-lora-v13-1', r=8, alpha=16, dropout=.05,
    last_n_layers=8, epochs=1, accum=4, lr=1e-4, weight_decay=.01,
    warmup=.05, temperature=.05, distill_weight=.01, seed=42,
    train_doc_tokens=384, eval_doc_tokens=512, max_query_tokens=128,
    train_forward_batch=2, eval_batch_size=8, max_train_query_caps=[None,1000,700],
    benchmark_queries=8, benchmark_items=16, safety=1.45,
    reserve_seconds=1200., checkpoint_updates=16, checkpoint_seconds=30.,
    checkpoint_items=128, grad_scaler_init=1024., max_overflow_windows=5)


class GigaLoRABudget(RuntimeError):
    pass


def _config(config=None):
    if set(config or {}) - set(CONFIG):
        raise ValueError('Unknown Giga LoRA configuration key')
    c = {**CONFIG, **(config or {})}
    for k in ['r','alpha','last_n_layers','epochs','accum','train_doc_tokens',
              'eval_doc_tokens','max_query_tokens','train_forward_batch',
              'eval_batch_size','benchmark_queries','benchmark_items',
              'checkpoint_updates','checkpoint_items','max_overflow_windows']:
        if not isinstance(c[k], (int,np.integer)) or int(c[k]) < 1:
            raise ValueError('Positive integer required: '+k)
        c[k] = int(c[k])
    if c['epochs'] != 1 or c['last_n_layers'] not in (6,8):
        raise ValueError('Use one epoch and the last six or eight layers')
    if c['train_doc_tokens'] not in (384,448) or c['eval_doc_tokens'] not in (512,640):
        raise ValueError('Fixed training/evaluation context choices required')
    if c['max_query_tokens'] != 128 or c['accum'] != 4:
        raise ValueError('Fixed 128-token queries and four-query accumulation required')
    if c['max_train_query_caps'] != [None,1000,700]:
        raise ValueError('Query caps must be ALL, 1000, 700 in that order')
    for k in ['lr','temperature','safety','checkpoint_seconds','grad_scaler_init']:
        if not np.isfinite(c[k]) or c[k] <= 0:
            raise ValueError('Positive finite value required: '+k)
    if c['safety'] < 1 or not 0 <= c['dropout'] < 1 or c['distill_weight'] < 0:
        raise ValueError('Invalid safety/dropout/distillation configuration')
    if c['reserve_seconds'] < 1200:
        raise ValueError('Keep at least twenty minutes for CPU fitting and submission')
    return c


def _available(budget, c):
    if hasattr(budget, 'remaining'):
        return max(0., float(budget.remaining())-c['reserve_seconds'])
    if hasattr(budget, 'for_neural'):
        return max(0., float(budget.for_neural()))
    raise TypeError('Budget must expose remaining() or reserved for_neural()')


def _check(budget, c):
    if _available(budget,c) < 10:
        raise GigaLoRABudget('Twenty-minute CPU/save reserve reached; resume saved checkpoints')


def _sync(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize(device)


def _autocast(device):
    return torch.autocast(device_type=torch.device(device).type,
        dtype=torch.float16, enabled=str(device).startswith('cuda'))


def _digest(a):
    a = np.ascontiguousarray(a)
    return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()


def _key(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()


def _atomic_torch(path,value):
    tmp = Path(str(path)+'.tmp')
    torch.save(value,tmp)
    os.replace(tmp,path)


def _cpu(value):
    if isinstance(value,torch.Tensor): return value.detach().cpu()
    if isinstance(value,dict): return {k:_cpu(v) for k,v in value.items()}
    if isinstance(value,list): return [_cpu(v) for v in value]
    if isinstance(value,tuple): return tuple(_cpu(v) for v in value)
    return value


def _rng(device):
    n = np.random.get_state()
    return dict(torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state(device) if str(device).startswith('cuda') else None,
        python=random.getstate(), numpy=(n[0],torch.from_numpy(n[1].astype(np.int64)),n[2],n[3],n[4]))


def _restore_rng(state,device):
    torch.set_rng_state(state['torch'])
    if state['cuda'] is not None: torch.cuda.set_rng_state(state['cuda'],device)
    random.setstate(state['python'])
    n = state['numpy']
    np.random.set_state((n[0],n[1].numpy().astype(np.uint32),n[2],n[3],n[4]))


def _seed(seed,device):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if str(device).startswith('cuda'): torch.cuda.manual_seed(seed)


class LoRALinear(nn.Module):
    """Frozen base linear plus an unmerged, FP32 standard LoRA residual."""
    def __init__(self,base,r=8,alpha=16,dropout=.05):
        super().__init__()
        if not isinstance(base,nn.Linear): raise TypeError('LoRA target must be Linear')
        self.base = base
        for p in base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(base.out_features,r,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout)
        self.scaling = float(alpha)/r

    def forward(self,x):
        original = self.base(x)
        with torch.autocast(device_type=x.device.type,enabled=False):
            delta = F.linear(F.linear(self.dropout(x.float()),self.A),self.B)*self.scaling
        return original+delta.to(original.dtype)


def inject_lora(model,config=None):
    c = {**CONFIG,**(config or {})}
    for p in model.parameters(): p.requires_grad_(False)
    layers = int(model.config.num_hidden_layers)
    chosen = []
    for name,module in list(model.named_modules()):
        m = re.search(r'(?:^|\.)layers\.(\d+)\.self_attn\.(q_proj|v_proj)$',name)
        if m is None or int(m.group(1)) < max(0,layers-c['last_n_layers']): continue
        parent,leaf = name.rsplit('.',1)
        setattr(model.get_submodule(parent),leaf,LoRALinear(module,c['r'],c['alpha'],c['dropout']))
        chosen.append(name)
    if len(chosen) != 2*min(layers,c['last_n_layers']):
        raise ValueError('Pinned Giga q/v layer structure changed')
    if any(layer.self_attn.is_causal for layer in model.layers):
        raise ValueError('Giga attention must remain bidirectional')
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    model.config.use_cache = False
    for layer in model.layers:
        if not getattr(layer,'gradient_checkpointing',False):
            raise RuntimeError('Native decoder layer does not enable gradient checkpointing')
        if layer._gradient_checkpointing_func.keywords.get('use_reentrant') is not False:
            raise RuntimeError('Nonreentrant gradient checkpointing is required')
    return chosen


def adapter_state(model):
    return {n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}


def set_adapter_state(model,state):
    params = {n:p for n,p in model.named_parameters() if p.requires_grad}
    if set(params) != set(state): raise ValueError('LoRA parameter names changed')
    with torch.no_grad():
        for n,p in params.items():
            if state[n].shape != p.shape or not torch.isfinite(state[n]).all():
                raise ValueError('Invalid adapter tensor')
            p.copy_(state[n].to(p.device,dtype=p.dtype))


def _load_model(device,c):
    _seed(c['seed'],device)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,
        trust_remote_code=True,padding_side='right')
    model = AutoModel.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,
        code_revision=MODEL_REVISION,trust_remote_code=True,
        torch_dtype=torch.float16 if str(device).startswith('cuda') else torch.float32,
        attn_implementation='sdpa',low_cpu_mem_usage=True,device_map={'':str(device)})
    if int(model.config.hidden_size) != 2048: raise ValueError('Pinned Giga width changed')
    targets = inject_lora(model,c)
    if any(p.dtype != torch.float32 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Trainable LoRA parameters must be FP32')
    return model,tok,targets


def embed(model,tokenizer,texts,cap,device='cuda'):
    """Differentiable official pooling; pass only the native supported kwargs."""
    batch = tokenizer(list(texts),padding=True,truncation=True,max_length=int(cap),
        pad_to_multiple_of=8,return_tensors='pt',return_token_type_ids=False,add_special_tokens=True)
    ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
    with _autocast(device):
        output = model(input_ids=ids,attention_mask=mask)
        return ge.mean_pool(output.last_hidden_state,mask)


def ranking_loss(scores,labels,teacher=None,config=None):
    c = {**CONFIG,**(config or {})}
    z = scores.float(); y = labels.float()
    if z.ndim != 1 or z.shape != y.shape or not torch.isfinite(z).all():
        raise ValueError('Invalid cosine scores/labels')
    pos = z[y>.5]; neg = z[y<=.5]
    if not len(pos) or not len(neg): raise ValueError('A ranking query needs positives and negatives')
    loss = F.softplus(-(pos[:,None]-neg[None,:])/c['temperature']).mean()
    if teacher is not None and c['distill_weight']:
        t = torch.as_tensor(teacher,device=z.device,dtype=torch.float32)
        if t.shape != z.shape or not torch.isfinite(t).all(): raise ValueError('Invalid frozen cosine teacher')
        loss = loss+c['distill_weight']*F.mse_loss(
            (z-z.mean())/c['temperature'],(t-t.mean())/c['temperature'])
    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite Giga LoRA loss')
    return loss


def _clean(tr,te):
    frames = []
    for f in (tr,te):
        f = f.reset_index(drop=True).copy()
        if not {'id','query_id','query_c','title_c','text_c'} <= set(f):
            raise ValueError('Canonical pair columns required')
        if f.id.isna().any() or f.query_id.isna().any(): raise ValueError('Missing pair/query IDs')
        f['id'] = f.id.astype(str); f['query_id'] = f.query_id.astype(str)
        if not f.id.is_unique: raise ValueError('Duplicate pair IDs')
        for col in ['query_c','title_c','text_c']: f[col] = f[col].fillna('').astype(str)
        if f.query_c.str.strip().eq('').any(): raise ValueError('Empty query text')
        if (f.groupby('query_id').query_c.nunique()>1).any(): raise ValueError('Query ID has conflicting texts')
        frames.append(f)
    a,b = frames
    if set(a.id)&set(b.id) or set(a.query_id)&set(b.query_id): raise ValueError('Train/test overlap')
    return a,b


def make_folds(tr,folds):
    original = list(folds)
    if len(original) != 5 or 'grp' not in tr: raise ValueError('Original five folds and connected grp required')
    audit = np.sort(np.asarray(original[-1][1],dtype=np.int64))
    if len(np.unique(audit))!=len(audit) or np.any(audit<0) or np.any(audit>=len(tr)):
        raise ValueError('Invalid audit rows')
    dev = np.setdiff1d(np.arange(len(tr)),audit)
    if not len(dev) or not len(audit): raise ValueError('Development/audit must be nonempty')
    for col in ['grp','query_id']+(['document_id'] if 'document_id' in tr else []):
        if set(tr.iloc[dev][col])&set(tr.iloc[audit][col]): raise ValueError(col+' crosses audit boundary')
    from final_data import source_folds
    result=source_folds(tr,dev)
    for a,b in result:
        for col in ['grp','query_id']+(['document_id'] if 'document_id' in tr else []):
            if set(tr.iloc[a][col])&set(tr.iloc[b][col]): raise ValueError(col+' crosses source folds')
    return dev,audit,result


def _teacher(teacher_scores,n):
    if not isinstance(teacher_scores,dict) or set(teacher_scores) != {'giga_cosine'}:
        raise ValueError('Supply the fixed zero-shot teacher as {giga_cosine: train_cosines}')
    values = teacher_scores['giga_cosine']
    if isinstance(values,tuple) and len(values)==2: values = values[0]
    a = np.asarray(values,dtype=np.float32)
    if a.shape != (n,) or not np.isfinite(a).all() or np.any(np.abs(a)>1.001):
        raise ValueError('Teacher must be finite, unnormalized zero-shot Giga raw cosines')
    return a


def _binding(tr,te,dev,audit,folds,c,teacher):
    full = pd.concat([tr,te],ignore_index=True)
    binding = dict(model=MODEL_NAME,revision=MODEL_REVISION,config=c,
        columns={col:ge._strings_digest(full[col]) for col in ['id','query_id','query_c','title_c','text_c']},
        groups=ge._strings_digest(tr.grp),dev=_digest(dev),audit=_digest(audit),
        development_labels=_digest(tr.iloc[dev].relevance.to_numpy(np.float32)),
        frozen_teacher=dict(kind='zero_shot_giga_cosine',digest=_digest(teacher)),
        folds=[dict(train=_digest(a),heldout=_digest(b)) for a,b in folds],
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy']})
    return _key(binding),binding


class QueryStore:
    def __init__(self,frame,teacher):
        self.frame = frame; self.teacher = teacher
        self.groups = {str(q):np.asarray(rows,np.int64) for q,rows in frame.groupby('query_id',sort=True).indices.items()}
        self.queries = (QUERY_PREFIX+frame.query_c).tolist()
        self.documents = (frame.title_c+'\n'+frame.text_c).tolist()

    def keys(self,rows):
        return sorted(set(self.frame.iloc[rows].query_id))

    def selected(self,q,seed):
        rows = self.groups[q]
        y = self.frame.iloc[rows].relevance.to_numpy(np.float32)
        pos = rows[y>.5]; neg = rows[y<=.5]
        pos = sorted(pos,key=lambda i:hashlib.sha256(f'{seed}\0{self.frame.id.iloc[i]}'.encode()).digest())[:2]
        neg = sorted(neg,key=lambda i:(-float(self.teacher[i]),self.frame.id.iloc[i]))[:2]
        if not pos or not neg: return None
        selected = np.asarray(pos+neg,np.int64)
        return selected,np.asarray([1.]*len(pos)+[0.]*len(neg),np.float32)


def _query_loss(model,tok,store,q,c,device,batch_size,seed,synthetic=False):
    if synthetic:
        rows = store.groups[q]
        rows = sorted(rows,key=lambda i:len(store.documents[i]),reverse=True)[:4]
        if len(rows)<2: raise ValueError('Benchmark query requires at least two documents')
        labels = np.asarray([float(i<len(rows)//2) for i in range(len(rows))],np.float32)
        rows = np.asarray(rows,np.int64)
    else:
        rows,labels = store.selected(q,seed)
    query = embed(model,tok,[store.queries[rows[0]]],c['max_query_tokens'],device)
    documents = []
    for start in range(0,len(rows),batch_size):
        documents.append(embed(model,tok,[store.documents[i] for i in rows[start:start+batch_size]],c['train_doc_tokens'],device))
    scores = (query*torch.cat(documents)).sum(-1)
    teacher = None if synthetic else store.teacher[rows]
    return ranking_loss(scores,torch.as_tensor(labels,device=device),teacher,c)


def _scaler(device,c):
    return torch.amp.GradScaler('cuda',enabled=str(device).startswith('cuda'),init_scale=c['grad_scaler_init'])


def _read_checkpoint(path,signature):
    if not Path(path).is_file(): return None
    s = torch.load(path,map_location='cpu',weights_only=True)
    if s.get('signature') != signature: raise ValueError('Giga training checkpoint binding changed')
    return s


def _training_keys(store,rows,seed,cap):
    keys = [q for q in store.keys(rows) if store.selected(q,seed) is not None]
    rng = np.random.default_rng(seed)
    keys = [keys[i] for i in rng.permutation(len(keys))]
    return keys if cap is None else keys[:cap]


def _train_fold(model,tok,store,keys,c,path,signature,budget,device,fold):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params,lr=c['lr'],weight_decay=c['weight_decay'])
    scaler = _scaler(device,c)
    cursor=updates=overflows=0; loss_sum=0.; bs=c['train_forward_batch']
    _seed(c['seed']+fold,device)
    state = _read_checkpoint(path,signature)
    if state is not None:
        if state['query_order'] != keys: raise ValueError('Committed query order changed')
        cursor=int(state['cursor']); updates=int(state['updates']); overflows=int(state['overflows'])
        if cursor<0 or cursor>len(keys) or (cursor%c['accum'] and cursor!=len(keys)):
            raise ValueError('Checkpoint must be at an optimizer boundary')
        set_adapter_state(model,state['adapter']); opt.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler']); loss_sum=float(state['loss_sum'])
        bs=int(state['forward_batch']); _restore_rng(state['rng'],device)
    committed_rng = _rng(device); last_save=time.monotonic(); last_updates=updates
    model.train(); opt.zero_grad(set_to_none=True)
    def save():
        _atomic_torch(path,dict(signature=signature,adapter=adapter_state(model),
            optimizer=_cpu(opt.state_dict()),scaler=scaler.state_dict(),rng=committed_rng,
            cursor=cursor,updates=updates,overflows=overflows,loss_sum=loss_sum,
            query_order=keys,forward_batch=bs,complete=cursor==len(keys)))
    def window(start,end):
        total=0.
        for j in range(start,end):
            _check(budget,c)
            loss = _query_loss(model,tok,store,keys[j],c,device,bs,c['seed']+fold)
            total+=float(loss.detach()); scaler.scale(loss/(end-start)).backward()
        return total
    try:
        while cursor<len(keys):
            _check(budget,c); model.train(); end=min(len(keys),cursor+c['accum'])
            total_steps=math.ceil(len(keys)/c['accum']); step=cursor//c['accum']
            warm=max(1,int(total_steps*c['warmup']))
            factor=(step+1)/warm if step<warm else max(0.,(total_steps-step)/max(1,total_steps-warm))
            for group in opt.param_groups: group['lr']=c['lr']*factor
            try:
                value=window(cursor,end)
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True); _restore_rng(committed_rng,device)
                if bs<=1: raise
                bs=1; gc.collect()
                if str(device).startswith('cuda'): torch.cuda.empty_cache()
                continue
            scaler.unscale_(opt)
            norm=torch.nn.utils.clip_grad_norm_(params,1.)
            finite=bool(torch.isfinite(norm)) and all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in params)
            before=float(scaler.get_scale())
            if not finite and not str(device).startswith('cuda'):
                raise FloatingPointError('Nonfinite Giga CPU adapter gradients')
            scaler.step(opt); scaler.update()
            if not finite or float(scaler.get_scale())<before:
                overflows+=1; opt.zero_grad(set_to_none=True); _restore_rng(committed_rng,device)
                if overflows>c['max_overflow_windows']:
                    raise FloatingPointError('Repeated FP16 gradient overflow')
                continue
            cursor=end; updates+=1; loss_sum+=value
            opt.zero_grad(set_to_none=True); committed_rng=_rng(device)
            if updates-last_updates>=c['checkpoint_updates'] or time.monotonic()-last_save>=c['checkpoint_seconds'] or cursor==len(keys):
                save(); last_save=time.monotonic(); last_updates=updates
                print(f'Giga LoRA fold {fold}: {cursor}/{len(keys)} queries, {updates} updates',flush=True)
        if not updates: raise FloatingPointError('No successful optimizer update')
        save()
        return dict(train_queries=len(keys),updates=updates,overflow_windows=overflows,
            mean_loss=loss_sum/max(1,len(keys)),forward_batch=bs)
    finally:
        opt.zero_grad(set_to_none=True)
        _restore_rng(committed_rng,device)
        # Uncommitted partial-query gradients are discarded; all tensors and RNG
        # describe the last successful optimizer boundary.
        save()
        del opt,scaler


def _representatives(texts,count):
    if not texts: return []
    order=np.argsort([len(s) for s in texts],kind='stable')
    positions=np.unique(np.linspace(max(0,len(order)//2),len(order)-1,min(count,len(order))).astype(int))
    return [texts[int(order[i])] for i in positions]


def _benchmark(model,tok,store,dev,phases,c,out,budget,device,load_seconds):
    initial=adapter_state(model); initial_rng=_rng(device); initial_mode=model.training
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=c['lr'],weight_decay=c['weight_decay']); scaler=_scaler(device,c)
    keys=[q for q in store.keys(dev) if len(store.groups[q])>=2]
    # Upper-half length samples cover long documents, rather than quick short queries.
    _check(budget,c)
    keys=sorted(keys,key=lambda q:sum(sorted((min(len(tok.encode(store.documents[i],add_special_tokens=True)),c['train_doc_tokens']) for i in store.groups[q]),reverse=True)[:4]))
    _check(budget,c)
    ix=np.unique(np.linspace(len(keys)//2,len(keys)-1,min(c['benchmark_queries'],len(keys))).astype(int))
    probes=[keys[i] for i in ix]
    if not probes: raise ValueError('No benchmark queries')
    bs=c['train_forward_batch']
    try:
        while True:
            set_adapter_state(model,initial); opt.zero_grad(set_to_none=True); model.train()
            try:
                # One warm-up backward, including optimizer kernels; all labels synthetic.
                _check(budget,c)
                loss=_query_loss(model,tok,store,probes[-1],c,device,bs,c['seed'],True)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                norm=torch.nn.utils.clip_grad_norm_(params,1.)
                if not bool(torch.isfinite(norm)): raise FloatingPointError('Benchmark gradient nonfinite')
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                _sync(device); start=time.monotonic()
                for i,q in enumerate(probes):
                    _check(budget,c)
                    loss=_query_loss(model,tok,store,q,c,device,bs,c['seed'],True)
                    scaler.scale(loss/c['accum']).backward()
                    if (i+1)%c['accum']==0 or i==len(probes)-1:
                        scaler.unscale_(opt); norm=torch.nn.utils.clip_grad_norm_(params,1.)
                        if not bool(torch.isfinite(norm)): raise FloatingPointError('Benchmark gradient nonfinite')
                        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                _sync(device); train_rate=(time.monotonic()-start)/len(probes)
                break
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True)
                if bs<=1: raise
                bs=1; gc.collect()
                if str(device).startswith('cuda'): torch.cuda.empty_cache()
        # Actual eval on representative real texts, with inference mode/dropout disabled.
        rates={}; model.eval(); eval_bs=c['eval_batch_size']
        with torch.inference_mode():
            # Long documents establish the microbatch before shorter queries.
            # Time the same batching used by real inference, not single items.
            for role,cap in [('document',c['eval_doc_tokens']),('query',c['max_query_tokens'])]:
                texts=list(dict.fromkeys(t for phase in phases for t in phase[role]))
                texts=_representatives(texts,c['benchmark_items'])
                while True:
                    try:
                        _check(budget,c); embed(model,tok,texts[-eval_bs:],cap,device)
                        _sync(device); start=time.monotonic()
                        for i in range(0,len(texts),eval_bs):
                            _check(budget,c); embed(model,tok,texts[i:i+eval_bs],cap,device)
                        _sync(device); rates[role]=(time.monotonic()-start)/max(1,len(texts))
                        break
                    except torch.cuda.OutOfMemoryError:
                        if eval_bs<=1: raise
                        eval_bs=max(1,eval_bs//2); gc.collect()
                        if str(device).startswith('cuda'): torch.cuda.empty_cache()
        io_path=out/'benchmark_io.pt'; start=time.monotonic()
        _atomic_torch(io_path,dict(adapter=adapter_state(model),optimizer=_cpu(opt.state_dict()),
            scaler=scaler.state_dict(),rng=_rng(device)))
        torch.load(io_path,map_location='cpu',weights_only=True); io_seconds=time.monotonic()-start
        io_path.unlink(missing_ok=True)
        return dict(train_seconds_per_query=train_rate,inference_seconds_per_item=rates,
            model_load_seconds=load_seconds,checkpoint_io_seconds=io_seconds,
            train_probe_queries=len(probes),train_forward_batch=bs,eval_batch_size=eval_bs,
            length_sampling='upper-half through maximum document lengths; real full eval context',
            benchmark_labels='synthetic only; adapter, RNG and model mode restored',
            peak_cuda_gb=torch.cuda.max_memory_allocated(device)/2**30 if str(device).startswith('cuda') else None)
    finally:
        opt.zero_grad(set_to_none=True); set_adapter_state(model,initial)
        _restore_rng(initial_rng,device); model.train(initial_mode)
        del opt,scaler


def _phase_texts(frame):
    return dict(query=(QUERY_PREFIX+frame.query_c).tolist(),
                document=(frame.title_c+'\n'+frame.text_c).tolist())


def _phase_signature(signature,weight_digest,phase,frame,c):
    return _key(dict(signature=signature,weight_digest=weight_digest,phase=phase,
        ids=ge._strings_digest(frame.id),texts={r:ge._strings_digest(t) for r,t in _phase_texts(frame).items()},
        query_cap=c['max_query_tokens'],document_cap=c['eval_doc_tokens']))


def _load_cosines(path,key,ids):
    if not path.is_file(): return None
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item())!=key or not np.array_equal(z['ids'],np.asarray(ids,dtype=str)):
            raise ValueError('Giga cosine cache binding changed')
        values=z['cosines'].copy()
    if values.shape!=(len(ids),) or not np.isfinite(values).all(): raise ValueError('Incomplete cosine cache')
    return values


def _infer_phase(model,tok,frame,root,signature,weight_digest,phase,c,budget,device):
    root.mkdir(parents=True,exist_ok=True)
    key=_phase_signature(signature,weight_digest,phase,frame,c)
    cached=_load_cosines(root/'cosines.npz',key,frame.id)
    if cached is not None: return cached
    vectors={}; maps={}; texts=_phase_texts(frame); mode=model.training; model.eval()
    try:
        with torch.inference_mode():
            for role,cap in [('query',c['max_query_tokens']),('document',c['eval_doc_tokens'])]:
                unique,keys,mapping=ge._unique(texts[role],role)
                state=ge._open_role(root,role,keys,key,int(model.config.hidden_size))
                maps[role]=mapping
                # Token lengths are measured in chunks to keep a budget boundary.
                pending=np.flatnonzero(~state['done']); lengths=[]
                for start in range(0,len(pending),64):
                    _check(budget,c)
                    lengths.extend(len(tok.encode(unique[j],add_special_tokens=True,truncation=True,max_length=cap)) for j in pending[start:start+64])
                order=pending[np.argsort(lengths,kind='stable')]
                i=0; bs=c['eval_batch_size']; started=last=time.monotonic()
                print(f'Giga LoRA {root.name} {role}: {int(state["done"].sum())}/{len(unique)} unique cached',flush=True)
                try:
                    while i<len(order):
                        _check(budget,c); rows=order[i:i+bs]
                        try:
                            values=embed(model,tok,[unique[j] for j in rows],cap,device).cpu().numpy().astype(np.float32)
                        except torch.cuda.OutOfMemoryError:
                            if bs<=1: raise
                            bs=max(1,bs//2); gc.collect()
                            if str(device).startswith('cuda'): torch.cuda.empty_cache()
                            continue
                        ge._validate_vectors(values,len(rows),int(model.config.hidden_size))
                        state['values'][rows]=values; state['done'][rows]=True
                        state['pending'].extend(rows.tolist()); i+=len(rows)
                        if len(state['pending'])>=c['checkpoint_items'] or time.monotonic()-last>=c['checkpoint_seconds']:
                            ge._commit(state); last=time.monotonic()
                            eta=(len(order)-i)*(last-started)/max(1,i)
                            print(f'Giga LoRA {root.name} {role}: {int(state["done"].sum())}/{len(unique)} unique; ETA {eta/60:.1f} min',flush=True)
                finally:
                    ge._commit(state)
                if not state['done'].all(): raise GigaLoRABudget('Incomplete inference role')
                vectors[role]=state['values']
        # No pair-aligned vectors are serialized: cosine rows are produced here.
        values=np.empty(len(frame),np.float32)
        for start in range(0,len(frame),4096):
            stop=min(len(frame),start+4096)
            values[start:stop]=np.einsum('ij,ij->i',vectors['query'][maps['query'][start:stop]],vectors['document'][maps['document'][start:stop]])
        if not np.isfinite(values).all(): raise FloatingPointError('Nonfinite Giga LoRA cosines')
        ge._atomic_npz(root/'cosines.npz',signature=np.asarray(key),ids=frame.id.to_numpy(dtype=str),cosines=values)
        return values
    finally:
        model.train(mode)


def _query_z(frame,values):
    s=pd.Series(np.asarray(values,dtype=np.float64))
    group=s.groupby(frame.query_id.to_numpy(),sort=False)
    return ((s-group.transform('mean'))/(group.transform('std').fillna(0)+1e-6)).to_numpy(np.float32)


def _adapter_digest(model):
    return _key({n:_digest(p.numpy()) for n,p in adapter_state(model).items()})


def run_giga_lora(tr,te,folds,cache_dir,budget,teacher_scores,config=None,device='cuda'):
    """Train two DEV-only models; return complete query-normalized score pairs.

    ``folds`` are the original five component folds; the last held-out fold is
    the immutable audit. ``teacher_scores['giga_cosine']`` is a fixed zero-shot
    raw cosine array for train rows. Audit labels are never hashed or read.
    """
    started=time.monotonic(); c=_config(config); tr,te=_clean(tr,te)
    dev,audit,source_folds=make_folds(tr,folds)
    if 'relevance' not in tr or not tr.iloc[dev].relevance.isin([0,1]).all():
        raise ValueError('Development labels must be binary')
    teacher=_teacher(teacher_scores,len(tr)); request_key,binding=_binding(tr,te,dev,audit,source_folds,c,teacher)
    out=Path(cache_dir); out.mkdir(parents=True,exist_ok=True)
    plan_path=out/'plan.json'; plan=json.loads(plan_path.read_text()) if plan_path.is_file() else None
    if plan and plan['request_key']!=request_key: raise ValueError('Giga LoRA inputs/config changed; use a new cache directory')
    resolved=plan['resolved_config'] if plan else dict(c)
    signature=plan['signature'] if plan else None
    report=dict(status='initializing',model=MODEL_NAME,revision=MODEL_REVISION,
        request_key=request_key,folds=[],no_audit_training=True,
        training='two GroupKFold source models on DEV only; one epoch',
        score_scope='DEV OOF; audit/test mean raw cosines, then within-query z',
        validation_caveat='Downstream meta-CV with learned OOF inputs is approximate; each source held-out target is excluded from that source training.')
    def complete():
        path=out/'complete.npz'
        if signature is None or not path.is_file(): return {}
        with np.load(path,allow_pickle=False) as z:
            if str(z['signature'].item())!=signature or not np.array_equal(z['train_ids'],tr.id.to_numpy(dtype=str)) or not np.array_equal(z['test_ids'],te.id.to_numpy(dtype=str)):
                raise ValueError('Complete Giga LoRA score identity changed')
            a=z['train'].copy(); b=z['test'].copy(); hashes=z['weight_digests'].astype(str)
        if a.shape!=(len(tr),) or b.shape!=(len(te),) or not np.isfinite(a).all() or not np.isfinite(b).all() or len(hashes)!=2:
            raise ValueError('Incomplete Giga LoRA final source')
        for k,h in enumerate(hashes):
            s=_read_checkpoint(out/f'fold{k}_train.pt',signature)
            if s is None or not s['complete'] or _key({n:_digest(p.numpy()) for n,p in s['adapter'].items()})!=h:
                raise ValueError('Complete source adapter checkpoint changed')
        return {'giga_lora':(a,b)}
    def finish(status):
        report.update(status=status,signature=signature,seconds=time.monotonic()-started,resolved_config=resolved,
            available_neural_seconds=_available(budget,c))
        ge._atomic_json(out/'report.json',report)
        return complete(),report
    if complete():
        old_report=out/'report.json'
        if old_report.is_file():
            previous=json.loads(old_report.read_text())
            if previous.get('signature')==signature: report.update(previous)
        return finish('complete_cached')
    if str(device).startswith('cuda') and not torch.cuda.is_available():
        report['reason']='A CUDA GPU is required for the 3B training experiment'
        return finish('no_cuda')
    # QueryStore carries only DEV labels. Audit slots are explicitly erased.
    safe=tr.copy(); safe['relevance']=0.; safe.loc[dev,'relevance']=tr.iloc[dev].relevance.to_numpy(np.float32)
    store=QueryStore(safe,teacher)
    phase_frames=[]; all_phase_text=[]
    for k,(train_rows,held_rows) in enumerate(source_folds):
        phases=[('heldout',tr.iloc[held_rows].reset_index(drop=True)),
                ('audit',tr.iloc[audit].reset_index(drop=True)),('test',te)]
        phase_frames.append(phases); all_phase_text.extend(_phase_texts(frame) for _,frame in phases)
    model=tok=None
    try:
        _check(budget,c)
        if plan is None:
            start=time.monotonic(); model,tok,targets=_load_model(device,c); _sync(device)
            load_seconds=time.monotonic()-start
            rates=_benchmark(model,tok,store,dev,all_phase_text,c,out,budget,device,load_seconds)
            report['preflight']=rates
            best=None
            for cap in [None]:
                train_count=sum(len(_training_keys(store,a,c['seed']+k,cap)) for k,(a,b) in enumerate(source_folds))
                counts={r:sum(len(set(phase[r])) for phase in all_phase_text) for r in ['query','document']}
                updates=sum(math.ceil(len(_training_keys(store,a,c['seed']+k,cap))/c['accum']) for k,(a,b) in enumerate(source_folds))
                estimate=c['safety']*(train_count*rates['train_seconds_per_query']+
                    sum(counts[r]*rates['inference_seconds_per_item'][r] for r in counts)+
                    2*rates['model_load_seconds']+rates['checkpoint_io_seconds']*(2+math.ceil(updates/c['checkpoint_updates'])))
                report.setdefault('cap_forecasts',[]).append(dict(cap=cap,train_queries=train_count,unique_eval_items=counts,seconds=estimate))
                if estimate<=_available(budget,c): best=cap; break
            if not report['cap_forecasts'] or report['cap_forecasts'][-1]['seconds']>_available(budget,c):
                return finish('refused_eta')
            resolved={**c,'max_train_queries':best,'train_forward_batch':rates['train_forward_batch'],
                'eval_batch_size':rates['eval_batch_size']}
            signature=_key(dict(request_key=request_key,resolved_config=resolved))
            plan=dict(request_key=request_key,signature=signature,resolved_config=resolved,benchmark=rates,binding=binding)
            ge._atomic_json(plan_path,plan)
        else:
            rates=plan['benchmark']; report['preflight']=rates
        # Recompute the whole remaining workload, including both audit/test passes.
        train_remaining=0; eval_seconds=0.; checkpoints=0
        for k,(a,b) in enumerate(source_folds):
            keys=_training_keys(store,a,c['seed']+k,resolved['max_train_queries'])
            state=_read_checkpoint(out/f'fold{k}_train.pt',signature)
            cursor=int(state['cursor']) if state else 0
            if state and state['query_order']!=keys: raise ValueError('Resume query order differs')
            train_remaining+=len(keys)-cursor
            checkpoints+=math.ceil(max(0,len(keys)-cursor)/c['accum']/c['checkpoint_updates'])
            weight=_key({n:_digest(p.numpy()) for n,p in state['adapter'].items()}) if state and state['complete'] else None
            for phase,frame in phase_frames[k]:
                root=out/f'fold{k}_{phase}'
                key=_phase_signature(signature,weight,phase,frame,c) if weight else None
                if key and _load_cosines(root/'cosines.npz',key,frame.id) is not None: continue
                for role,texts in _phase_texts(frame).items():
                    n=len(set(texts)); manifest=root/role/'manifest.json'
                    if key and manifest.is_file():
                        m=json.loads(manifest.read_text())
                        if m['signature']!=key: raise ValueError('Inference resume weight binding changed')
                        n-=int(m['complete_items'])
                    eval_seconds+=n*rates['inference_seconds_per_item'][role]
        estimate=c['safety']*(train_remaining*rates['train_seconds_per_query']+eval_seconds+
            2*rates['model_load_seconds']+(checkpoints+2)*rates['checkpoint_io_seconds'])
        report.update(estimated_mandatory_seconds=estimate,remaining_train_queries=train_remaining)
        print(f'Giga LoRA whole two-fold ETA {estimate/60:.1f} min; available {_available(budget,c)/60:.1f} min; cap {resolved["max_train_queries"]}',flush=True)
        if estimate>_available(budget,c): return finish('refused_eta')
        # Benchmark model is never a training initialization or a fold artifact.
        model=tok=None; gc.collect()
        if str(device).startswith('cuda'): torch.cuda.empty_cache()
        raw_train=np.full(len(tr),np.nan,np.float32); audits=[]; tests=[]; hashes=[]
        for k,(train_rows,held_rows) in enumerate(source_folds):
            _check(budget,c)
            model,tok,targets=_load_model(device,{**resolved,'seed':c['seed']+k})
            keys=_training_keys(store,train_rows,c['seed']+k,resolved['max_train_queries'])
            record=_train_fold(model,tok,store,keys,resolved,out/f'fold{k}_train.pt',signature,budget,device,k)
            weight=_adapter_digest(model); hashes.append(weight)
            predictions={phase:_infer_phase(model,tok,frame,out/f'fold{k}_{phase}',signature,weight,phase,resolved,budget,device) for phase,frame in phase_frames[k]}
            raw_train[held_rows]=predictions['heldout']; audits.append(predictions['audit']); tests.append(predictions['test'])
            report['folds'].append(dict(fold=k,targets=targets,weight_digest=weight,**record))
            model=tok=None; gc.collect()
            if str(device).startswith('cuda'): torch.cuda.empty_cache()
        raw_train[audit]=np.mean(audits,axis=0); raw_test=np.mean(tests,axis=0)
        if not np.isfinite(raw_train).all() or not np.isfinite(raw_test).all(): raise FloatingPointError('Incomplete two-fold source')
        ge._atomic_npz(out/'complete.npz',signature=np.asarray(signature),
            train_ids=tr.id.to_numpy(dtype=str),test_ids=te.id.to_numpy(dtype=str),
            train=_query_z(tr,raw_train),test=_query_z(te,raw_test),weight_digests=np.asarray(hashes))
        return finish('complete')
    except GigaLoRABudget as exc:
        report['reason']=str(exc); return finish('budget_stopped')
    except (FloatingPointError,torch.cuda.OutOfMemoryError) as exc:
        report['reason']=str(exc); return finish('numerical_or_oom_stopped')
    finally:
        model=tok=None; gc.collect()
        if str(device).startswith('cuda'): torch.cuda.empty_cache()


In [ ]:
%%writefile giga_lora_v15.py
"""Budget-gated, audit-blind Giga training with frozen Jina 3.5 distillation.

Only LoRA q/v matrices in the last eight bidirectional layers are learned.
A complete source contains development OOF cosines and the raw-cosine mean of
both audit/test predictions, normalized within queries only after combination.
Unique inference embeddings and optimizer-boundary training states resume
atomically. No incomplete source is returned.
"""
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import random
import re
import time

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.model_selection import GroupKFold
from transformers import AutoModel, AutoTokenizer
import giga_encoder_v11 as ge

MODEL_NAME = ge.MODEL_NAME
MODEL_REVISION = ge.MODEL_REVISION
QUERY_PREFIX = ge.QUERY_PREFIX
CONFIG = dict(version='giga-jina-v15-1', r=8, alpha=16, dropout=.05,
    last_n_layers=8, epochs=1, accum=4, lr=1e-4, weight_decay=.01,
    warmup=.05, temperature=.05, distill_weight=.15, seed=42,
    train_doc_tokens=384, eval_doc_tokens=512, max_query_tokens=128,
    train_forward_batch=2, eval_batch_size=8, max_train_query_caps=[None,1000,700],
    benchmark_queries=8, benchmark_items=16, safety=1.45,
    reserve_seconds=1200., checkpoint_updates=16, checkpoint_seconds=30.,
    checkpoint_items=128, grad_scaler_init=1024., max_overflow_windows=5)


class GigaLoRABudget(RuntimeError):
    pass


def _config(config=None):
    if set(config or {}) - set(CONFIG):
        raise ValueError('Unknown Giga LoRA configuration key')
    c = {**CONFIG, **(config or {})}
    for k in ['r','alpha','last_n_layers','epochs','accum','train_doc_tokens',
              'eval_doc_tokens','max_query_tokens','train_forward_batch',
              'eval_batch_size','benchmark_queries','benchmark_items',
              'checkpoint_updates','checkpoint_items','max_overflow_windows']:
        if not isinstance(c[k], (int,np.integer)) or int(c[k]) < 1:
            raise ValueError('Positive integer required: '+k)
        c[k] = int(c[k])
    if c['epochs'] != 1 or c['last_n_layers'] not in (6,8):
        raise ValueError('Use one epoch and the last six or eight layers')
    if c['train_doc_tokens'] not in (384,448) or c['eval_doc_tokens'] not in (512,640):
        raise ValueError('Fixed training/evaluation context choices required')
    if c['max_query_tokens'] != 128 or c['accum'] != 4:
        raise ValueError('Fixed 128-token queries and four-query accumulation required')
    if c['max_train_query_caps'] != [None,1000,700]:
        raise ValueError('Query caps must be ALL, 1000, 700 in that order')
    for k in ['lr','temperature','safety','checkpoint_seconds','grad_scaler_init']:
        if not np.isfinite(c[k]) or c[k] <= 0:
            raise ValueError('Positive finite value required: '+k)
    if c['safety'] < 1 or not 0 <= c['dropout'] < 1 or c['distill_weight'] != .15:
        raise ValueError('Invalid safety/dropout/distillation configuration')
    if c['temperature'] != .05:
        raise ValueError('Use the fixed .05 student temperature')
    if c['reserve_seconds'] < 1200:
        raise ValueError('Keep at least twenty minutes for CPU fitting and submission')
    return c


def _available(budget, c):
    if hasattr(budget, 'remaining'):
        return max(0., float(budget.remaining())-c['reserve_seconds'])
    if hasattr(budget, 'for_neural'):
        return max(0., float(budget.for_neural()))
    raise TypeError('Budget must expose remaining() or reserved for_neural()')


def _check(budget, c):
    if _available(budget,c) < 10:
        raise GigaLoRABudget('Twenty-minute CPU/save reserve reached; resume saved checkpoints')


def _sync(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize(device)


def _autocast(device):
    return torch.autocast(device_type=torch.device(device).type,
        dtype=torch.float16, enabled=str(device).startswith('cuda'))


def _digest(a):
    a = np.ascontiguousarray(a)
    return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()


def _key(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()


def _atomic_torch(path,value):
    tmp = Path(str(path)+'.tmp')
    torch.save(value,tmp)
    os.replace(tmp,path)


def _cpu(value):
    if isinstance(value,torch.Tensor): return value.detach().cpu()
    if isinstance(value,dict): return {k:_cpu(v) for k,v in value.items()}
    if isinstance(value,list): return [_cpu(v) for v in value]
    if isinstance(value,tuple): return tuple(_cpu(v) for v in value)
    return value


def _rng(device):
    n = np.random.get_state()
    return dict(torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state(device) if str(device).startswith('cuda') else None,
        python=random.getstate(), numpy=(n[0],torch.from_numpy(n[1].astype(np.int64)),n[2],n[3],n[4]))


def _restore_rng(state,device):
    torch.set_rng_state(state['torch'])
    if state['cuda'] is not None: torch.cuda.set_rng_state(state['cuda'],device)
    random.setstate(state['python'])
    n = state['numpy']
    np.random.set_state((n[0],n[1].numpy().astype(np.uint32),n[2],n[3],n[4]))


def _seed(seed,device):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if str(device).startswith('cuda'): torch.cuda.manual_seed(seed)


class LoRALinear(nn.Module):
    """Frozen base linear plus an unmerged, FP32 standard LoRA residual."""
    def __init__(self,base,r=8,alpha=16,dropout=.05):
        super().__init__()
        if not isinstance(base,nn.Linear): raise TypeError('LoRA target must be Linear')
        self.base = base
        for p in base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B = nn.Parameter(torch.zeros(base.out_features,r,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout)
        self.scaling = float(alpha)/r

    def forward(self,x):
        original = self.base(x)
        with torch.autocast(device_type=x.device.type,enabled=False):
            delta = F.linear(F.linear(self.dropout(x.float()),self.A),self.B)*self.scaling
        return original+delta.to(original.dtype)


def inject_lora(model,config=None):
    c = {**CONFIG,**(config or {})}
    for p in model.parameters(): p.requires_grad_(False)
    layers = int(model.config.num_hidden_layers)
    chosen = []
    for name,module in list(model.named_modules()):
        m = re.search(r'(?:^|\.)layers\.(\d+)\.self_attn\.(q_proj|v_proj)$',name)
        if m is None or int(m.group(1)) < max(0,layers-c['last_n_layers']): continue
        parent,leaf = name.rsplit('.',1)
        setattr(model.get_submodule(parent),leaf,LoRALinear(module,c['r'],c['alpha'],c['dropout']))
        chosen.append(name)
    if len(chosen) != 2*min(layers,c['last_n_layers']):
        raise ValueError('Pinned Giga q/v layer structure changed')
    if any(layer.self_attn.is_causal for layer in model.layers):
        raise ValueError('Giga attention must remain bidirectional')
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
    model.config.use_cache = False
    for layer in model.layers:
        if not getattr(layer,'gradient_checkpointing',False):
            raise RuntimeError('Native decoder layer does not enable gradient checkpointing')
        if layer._gradient_checkpointing_func.keywords.get('use_reentrant') is not False:
            raise RuntimeError('Nonreentrant gradient checkpointing is required')
    return chosen


def adapter_state(model):
    return {n:p.detach().cpu().clone() for n,p in model.named_parameters() if p.requires_grad}


def set_adapter_state(model,state):
    params = {n:p for n,p in model.named_parameters() if p.requires_grad}
    if set(params) != set(state): raise ValueError('LoRA parameter names changed')
    with torch.no_grad():
        for n,p in params.items():
            if state[n].shape != p.shape or not torch.isfinite(state[n]).all():
                raise ValueError('Invalid adapter tensor')
            p.copy_(state[n].to(p.device,dtype=p.dtype))


def _load_model(device,c):
    _seed(c['seed'],device)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,
        trust_remote_code=True,padding_side='right')
    model = AutoModel.from_pretrained(MODEL_NAME,revision=MODEL_REVISION,
        code_revision=MODEL_REVISION,trust_remote_code=True,
        torch_dtype=torch.float16 if str(device).startswith('cuda') else torch.float32,
        attn_implementation='sdpa',low_cpu_mem_usage=True,device_map={'':str(device)})
    if int(model.config.hidden_size) != 2048: raise ValueError('Pinned Giga width changed')
    targets = inject_lora(model,c)
    if any(p.dtype != torch.float32 for p in model.parameters() if p.requires_grad):
        raise RuntimeError('Trainable LoRA parameters must be FP32')
    return model,tok,targets


def embed(model,tokenizer,texts,cap,device='cuda'):
    """Differentiable official pooling; pass only the native supported kwargs."""
    batch = tokenizer(list(texts),padding=True,truncation=True,max_length=int(cap),
        pad_to_multiple_of=8,return_tensors='pt',return_token_type_ids=False,add_special_tokens=True)
    ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
    with _autocast(device):
        output = model(input_ids=ids,attention_mask=mask)
        return ge.mean_pool(output.last_hidden_state,mask)


def ranking_loss(scores,labels,teacher=None,config=None):
    c = {**CONFIG,**(config or {})}
    z = scores.float(); y = labels.float()
    if z.ndim != 1 or z.shape != y.shape or not torch.isfinite(z).all():
        raise ValueError('Invalid cosine scores/labels')
    pos = z[y>.5]; neg = z[y<=.5]
    if not len(pos) or not len(neg): raise ValueError('A ranking query needs positives and negatives')
    loss = F.softplus(-(pos[:,None]-neg[None,:])/c['temperature']).mean()
    if teacher is not None and c['distill_weight']:
        t = torch.as_tensor(teacher,device=z.device,dtype=torch.float32)
        if t.shape != z.shape or not torch.isfinite(t).all(): raise ValueError('Invalid frozen Jina teacher')
        # The fixed per-query Jina z scores use teacher temperature 1.0.
        # Sum over this query, rather than dividing KL again by document count.
        t = t.detach().clamp(-8.,8.)
        loss = loss+c['distill_weight']*F.kl_div(
            F.log_softmax(z/c['temperature'],dim=0),F.softmax(t,dim=0),reduction='sum')
    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite Giga LoRA loss')
    return loss


def _clean(tr,te):
    frames = []
    for f in (tr,te):
        f = f.reset_index(drop=True).copy()
        if not {'id','query_id','query_c','title_c','text_c'} <= set(f):
            raise ValueError('Canonical pair columns required')
        if f.id.isna().any() or f.query_id.isna().any(): raise ValueError('Missing pair/query IDs')
        f['id'] = f.id.astype(str); f['query_id'] = f.query_id.astype(str)
        if not f.id.is_unique: raise ValueError('Duplicate pair IDs')
        for col in ['query_c','title_c','text_c']: f[col] = f[col].fillna('').astype(str)
        if f.query_c.str.strip().eq('').any(): raise ValueError('Empty query text')
        if (f.groupby('query_id').query_c.nunique()>1).any(): raise ValueError('Query ID has conflicting texts')
        frames.append(f)
    a,b = frames
    if set(a.id)&set(b.id) or set(a.query_id)&set(b.query_id): raise ValueError('Train/test overlap')
    return a,b


def make_folds(tr,folds):
    original = list(folds)
    if len(original) != 5 or 'grp' not in tr: raise ValueError('Original five folds and connected grp required')
    audit = np.sort(np.asarray(original[-1][1],dtype=np.int64))
    if len(np.unique(audit))!=len(audit) or np.any(audit<0) or np.any(audit>=len(tr)):
        raise ValueError('Invalid audit rows')
    dev = np.setdiff1d(np.arange(len(tr)),audit)
    if not len(dev) or not len(audit): raise ValueError('Development/audit must be nonempty')
    for col in ['grp','query_id']+(['document_id'] if 'document_id' in tr else []):
        if set(tr.iloc[dev][col])&set(tr.iloc[audit][col]): raise ValueError(col+' crosses audit boundary')
    from final_data import source_folds
    result=source_folds(tr,dev)
    for a,b in result:
        for col in ['grp','query_id']+(['document_id'] if 'document_id' in tr else []):
            if set(tr.iloc[a][col])&set(tr.iloc[b][col]): raise ValueError(col+' crosses source folds')
    return dev,audit,result


def _teacher(teacher_scores,n,query_ids):
    """Copy and validate frozen sources; normalize Jina once without labels."""
    if not isinstance(teacher_scores,dict) or set(teacher_scores) != {'giga_cosine','jina35_mean'}:
        raise ValueError('Supply only frozen zero-shot giga_cosine and jina35_mean train arrays')
    q = np.asarray(query_ids)
    if q.shape != (n,) or pd.isna(q).any():
        raise ValueError('Aligned nonmissing teacher query IDs required')
    result = {}
    for name in ['giga_cosine','jina35_mean']:
        a = np.array(teacher_scores[name],dtype=np.float32,copy=True)
        if a.shape != (n,) or not np.isfinite(a).all():
            raise ValueError('Teacher must be a finite, aligned frozen train array: '+name)
        if name == 'giga_cosine' and np.any(np.abs(a)>1.):
            raise ValueError('Giga teacher must contain unnormalized raw cosines in [-1,1]')
        if name == 'jina35_mean':
            # Already-normalized frozen scores are safe to standardize again.
            # float64 statistics handle constant/singleton queries stably.
            s = pd.Series(a.astype(np.float64))
            grouped = s.groupby(q,sort=False)
            scale = grouped.transform('std').fillna(0.)
            a = ((s-grouped.transform('mean'))/(scale+1e-6)).clip(-8.,8.).to_numpy(np.float32)
        a.setflags(write=False)
        result[name] = a
    return result


def _binding(tr,te,dev,audit,folds,c,teacher,teacher_scores=None):
    full = pd.concat([tr,te],ignore_index=True)
    binding = dict(model=MODEL_NAME,revision=MODEL_REVISION,config=c,
        columns={col:ge._strings_digest(full[col]) for col in ['id','query_id','query_c','title_c','text_c']},
        groups=ge._strings_digest(tr.grp),dev=_digest(dev),audit=_digest(audit),
        development_labels=_digest(tr.iloc[dev].relevance.to_numpy(np.float32)),
        frozen_teacher=dict(kind='zero_shot_giga_cosine_and_frozen_jina35_mean',
            digests={name:_digest(values) for name,values in sorted(teacher.items())},
            input_digests={name:_digest(np.asarray(values,dtype=np.float32))
                for name,values in sorted((teacher_scores if teacher_scores is not None else teacher).items())}),
        folds=[dict(train=_digest(a),heldout=_digest(b)) for a,b in folds],
        versions={p:importlib.metadata.version(p) for p in ['torch','transformers','numpy']})
    return _key(binding),binding


class QueryStore:
    def __init__(self,frame,teacher):
        self.frame = frame; self.teacher = teacher
        self.groups = {str(q):np.asarray(rows,np.int64) for q,rows in frame.groupby('query_id',sort=True).indices.items()}
        self.queries = (QUERY_PREFIX+frame.query_c).tolist()
        self.documents = (frame.title_c+'\n'+frame.text_c).tolist()

    def keys(self,rows):
        return sorted(set(self.frame.iloc[rows].query_id))

    def selected(self,q,seed):
        rows = self.groups[q]
        y = self.frame.iloc[rows].relevance.to_numpy(np.float32)
        pos = rows[y>.5]; neg = rows[y<=.5]
        pos = sorted(pos,key=lambda i:hashlib.sha256(f'{seed}\0{self.frame.id.iloc[i]}'.encode()).digest())[:2]
        giga = sorted(neg,key=lambda i:(-float(self.teacher['giga_cosine'][i]),self.frame.id.iloc[i]))
        jina = sorted(neg,key=lambda i:(-float(self.teacher['jina35_mean'][i]),self.frame.id.iloc[i]))
        # Union the two top-2 lists, then fill shared selections with Jina's
        # next-highest distinct negatives. At most 2 positives + 4 negatives.
        neg = list(dict.fromkeys(giga[:2]+jina[:2]))
        neg.extend(i for i in jina if i not in neg)
        neg = neg[:4]
        if not pos or not neg: return None
        selected = np.asarray(pos+neg,np.int64)
        return selected,np.asarray([1.]*len(pos)+[0.]*len(neg),np.float32)


def _query_loss(model,tok,store,q,c,device,batch_size,seed,synthetic=False):
    if synthetic:
        rows = store.groups[q]
        rows = sorted(rows,key=lambda i:len(store.documents[i]),reverse=True)[:6]
        if len(rows)<2: raise ValueError('Benchmark query requires at least two documents')
        labels = np.asarray([float(i<len(rows)//2) for i in range(len(rows))],np.float32)
        rows = np.asarray(rows,np.int64)
    else:
        rows,labels = store.selected(q,seed)
    query = embed(model,tok,[store.queries[rows[0]]],c['max_query_tokens'],device)
    documents = []
    for start in range(0,len(rows),batch_size):
        documents.append(embed(model,tok,[store.documents[i] for i in rows[start:start+batch_size]],c['train_doc_tokens'],device))
    scores = (query*torch.cat(documents)).sum(-1)
    teacher = None if synthetic else store.teacher['jina35_mean'][rows]
    return ranking_loss(scores,torch.as_tensor(labels,device=device),teacher,c)


def _scaler(device,c):
    return torch.amp.GradScaler('cuda',enabled=str(device).startswith('cuda'),init_scale=c['grad_scaler_init'])


def _read_checkpoint(path,signature):
    if not Path(path).is_file(): return None
    s = torch.load(path,map_location='cpu',weights_only=True)
    if s.get('signature') != signature: raise ValueError('Giga training checkpoint binding changed')
    return s


def _training_keys(store,rows,seed,cap):
    keys = [q for q in store.keys(rows) if store.selected(q,seed) is not None]
    rng = np.random.default_rng(seed)
    keys = [keys[i] for i in rng.permutation(len(keys))]
    return keys if cap is None else keys[:cap]


def _train_fold(model,tok,store,keys,c,path,signature,budget,device,fold):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params,lr=c['lr'],weight_decay=c['weight_decay'])
    scaler = _scaler(device,c)
    cursor=updates=overflows=0; loss_sum=0.; bs=c['train_forward_batch']
    _seed(c['seed']+fold,device)
    state = _read_checkpoint(path,signature)
    if state is not None:
        if state['query_order'] != keys: raise ValueError('Committed query order changed')
        cursor=int(state['cursor']); updates=int(state['updates']); overflows=int(state['overflows'])
        if cursor<0 or cursor>len(keys) or (cursor%c['accum'] and cursor!=len(keys)):
            raise ValueError('Checkpoint must be at an optimizer boundary')
        set_adapter_state(model,state['adapter']); opt.load_state_dict(state['optimizer'])
        scaler.load_state_dict(state['scaler']); loss_sum=float(state['loss_sum'])
        bs=int(state['forward_batch']); _restore_rng(state['rng'],device)
    committed_rng = _rng(device); last_save=time.monotonic(); last_updates=updates
    model.train(); opt.zero_grad(set_to_none=True)
    def save():
        _atomic_torch(path,dict(signature=signature,adapter=adapter_state(model),
            optimizer=_cpu(opt.state_dict()),scaler=scaler.state_dict(),rng=committed_rng,
            cursor=cursor,updates=updates,overflows=overflows,loss_sum=loss_sum,
            query_order=keys,forward_batch=bs,complete=cursor==len(keys)))
    def window(start,end):
        total=0.
        for j in range(start,end):
            _check(budget,c)
            loss = _query_loss(model,tok,store,keys[j],c,device,bs,c['seed']+fold)
            total+=float(loss.detach()); scaler.scale(loss/(end-start)).backward()
        return total
    try:
        while cursor<len(keys):
            _check(budget,c); model.train(); end=min(len(keys),cursor+c['accum'])
            total_steps=math.ceil(len(keys)/c['accum']); step=cursor//c['accum']
            warm=max(1,int(total_steps*c['warmup']))
            factor=(step+1)/warm if step<warm else max(0.,(total_steps-step)/max(1,total_steps-warm))
            for group in opt.param_groups: group['lr']=c['lr']*factor
            try:
                value=window(cursor,end)
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True); _restore_rng(committed_rng,device)
                if bs<=1: raise
                bs=1; gc.collect()
                if str(device).startswith('cuda'): torch.cuda.empty_cache()
                continue
            scaler.unscale_(opt)
            norm=torch.nn.utils.clip_grad_norm_(params,1.)
            finite=bool(torch.isfinite(norm)) and all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in params)
            before=float(scaler.get_scale())
            if not finite and not str(device).startswith('cuda'):
                raise FloatingPointError('Nonfinite Giga CPU adapter gradients')
            scaler.step(opt); scaler.update()
            if not finite or float(scaler.get_scale())<before:
                overflows+=1; opt.zero_grad(set_to_none=True); _restore_rng(committed_rng,device)
                if overflows>c['max_overflow_windows']:
                    raise FloatingPointError('Repeated FP16 gradient overflow')
                continue
            cursor=end; updates+=1; loss_sum+=value
            opt.zero_grad(set_to_none=True); committed_rng=_rng(device)
            if updates-last_updates>=c['checkpoint_updates'] or time.monotonic()-last_save>=c['checkpoint_seconds'] or cursor==len(keys):
                save(); last_save=time.monotonic(); last_updates=updates
                print(f'Giga LoRA fold {fold}: {cursor}/{len(keys)} queries, {updates} updates',flush=True)
        if not updates: raise FloatingPointError('No successful optimizer update')
        save()
        return dict(train_queries=len(keys),updates=updates,overflow_windows=overflows,
            mean_loss=loss_sum/max(1,len(keys)),forward_batch=bs)
    finally:
        opt.zero_grad(set_to_none=True)
        _restore_rng(committed_rng,device)
        # Uncommitted partial-query gradients are discarded; all tensors and RNG
        # describe the last successful optimizer boundary.
        save()
        del opt,scaler


def _representatives(texts,count):
    if not texts: return []
    order=np.argsort([len(s) for s in texts],kind='stable')
    positions=np.unique(np.linspace(max(0,len(order)//2),len(order)-1,min(count,len(order))).astype(int))
    return [texts[int(order[i])] for i in positions]


def _benchmark(model,tok,store,dev,phases,c,out,budget,device,load_seconds):
    initial=adapter_state(model); initial_rng=_rng(device); initial_mode=model.training
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=c['lr'],weight_decay=c['weight_decay']); scaler=_scaler(device,c)
    keys=[q for q in store.keys(dev) if len(store.groups[q])>=2]
    # Upper-half length samples cover long documents, rather than quick short queries.
    _check(budget,c)
    keys=sorted(keys,key=lambda q:sum(sorted((min(len(tok.encode(store.documents[i],add_special_tokens=True)),c['train_doc_tokens']) for i in store.groups[q]),reverse=True)[:6]))
    _check(budget,c)
    ix=np.unique(np.linspace(len(keys)//2,len(keys)-1,min(c['benchmark_queries'],len(keys))).astype(int))
    probes=[keys[i] for i in ix]
    if not probes: raise ValueError('No benchmark queries')
    bs=c['train_forward_batch']
    try:
        while True:
            set_adapter_state(model,initial); opt.zero_grad(set_to_none=True); model.train()
            try:
                # One warm-up backward, including optimizer kernels; all labels synthetic.
                _check(budget,c)
                loss=_query_loss(model,tok,store,probes[-1],c,device,bs,c['seed'],True)
                scaler.scale(loss).backward(); scaler.unscale_(opt)
                norm=torch.nn.utils.clip_grad_norm_(params,1.)
                if not bool(torch.isfinite(norm)): raise FloatingPointError('Benchmark gradient nonfinite')
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                _sync(device); start=time.monotonic()
                for i,q in enumerate(probes):
                    _check(budget,c)
                    loss=_query_loss(model,tok,store,q,c,device,bs,c['seed'],True)
                    scaler.scale(loss/c['accum']).backward()
                    if (i+1)%c['accum']==0 or i==len(probes)-1:
                        scaler.unscale_(opt); norm=torch.nn.utils.clip_grad_norm_(params,1.)
                        if not bool(torch.isfinite(norm)): raise FloatingPointError('Benchmark gradient nonfinite')
                        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                _sync(device); train_rate=(time.monotonic()-start)/len(probes)
                break
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True)
                if bs<=1: raise
                bs=1; gc.collect()
                if str(device).startswith('cuda'): torch.cuda.empty_cache()
        # Actual eval on representative real texts, with inference mode/dropout disabled.
        rates={}; model.eval(); eval_bs=c['eval_batch_size']
        with torch.inference_mode():
            # Long documents establish the microbatch before shorter queries.
            # Time the same batching used by real inference, not single items.
            for role,cap in [('document',c['eval_doc_tokens']),('query',c['max_query_tokens'])]:
                texts=list(dict.fromkeys(t for phase in phases for t in phase[role]))
                texts=_representatives(texts,c['benchmark_items'])
                while True:
                    try:
                        _check(budget,c); embed(model,tok,texts[-eval_bs:],cap,device)
                        _sync(device); start=time.monotonic()
                        for i in range(0,len(texts),eval_bs):
                            _check(budget,c); embed(model,tok,texts[i:i+eval_bs],cap,device)
                        _sync(device); rates[role]=(time.monotonic()-start)/max(1,len(texts))
                        break
                    except torch.cuda.OutOfMemoryError:
                        if eval_bs<=1: raise
                        eval_bs=max(1,eval_bs//2); gc.collect()
                        if str(device).startswith('cuda'): torch.cuda.empty_cache()
        io_path=out/'benchmark_io.pt'; start=time.monotonic()
        _atomic_torch(io_path,dict(adapter=adapter_state(model),optimizer=_cpu(opt.state_dict()),
            scaler=scaler.state_dict(),rng=_rng(device)))
        torch.load(io_path,map_location='cpu',weights_only=True); io_seconds=time.monotonic()-start
        io_path.unlink(missing_ok=True)
        return dict(train_seconds_per_query=train_rate,inference_seconds_per_item=rates,
            model_load_seconds=load_seconds,checkpoint_io_seconds=io_seconds,
            train_probe_queries=len(probes),train_forward_batch=bs,eval_batch_size=eval_bs,
            length_sampling='upper-half through maximum document lengths; real full eval context',
            benchmark_labels='synthetic only; adapter, RNG and model mode restored',
            peak_cuda_gb=torch.cuda.max_memory_allocated(device)/2**30 if str(device).startswith('cuda') else None)
    finally:
        opt.zero_grad(set_to_none=True); set_adapter_state(model,initial)
        _restore_rng(initial_rng,device); model.train(initial_mode)
        del opt,scaler


def _phase_texts(frame):
    return dict(query=(QUERY_PREFIX+frame.query_c).tolist(),
                document=(frame.title_c+'\n'+frame.text_c).tolist())


def _phase_signature(signature,weight_digest,phase,frame,c):
    return _key(dict(signature=signature,weight_digest=weight_digest,phase=phase,
        ids=ge._strings_digest(frame.id),texts={r:ge._strings_digest(t) for r,t in _phase_texts(frame).items()},
        query_cap=c['max_query_tokens'],document_cap=c['eval_doc_tokens']))


def _load_cosines(path,key,ids):
    if not path.is_file(): return None
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item())!=key or not np.array_equal(z['ids'],np.asarray(ids,dtype=str)):
            raise ValueError('Giga cosine cache binding changed')
        values=z['cosines'].copy()
    if values.shape!=(len(ids),) or not np.isfinite(values).all(): raise ValueError('Incomplete cosine cache')
    return values


def _infer_phase(model,tok,frame,root,signature,weight_digest,phase,c,budget,device):
    root.mkdir(parents=True,exist_ok=True)
    key=_phase_signature(signature,weight_digest,phase,frame,c)
    cached=_load_cosines(root/'cosines.npz',key,frame.id)
    if cached is not None: return cached
    vectors={}; maps={}; texts=_phase_texts(frame); mode=model.training; model.eval()
    try:
        with torch.inference_mode():
            for role,cap in [('query',c['max_query_tokens']),('document',c['eval_doc_tokens'])]:
                unique,keys,mapping=ge._unique(texts[role],role)
                state=ge._open_role(root,role,keys,key,int(model.config.hidden_size))
                maps[role]=mapping
                # Token lengths are measured in chunks to keep a budget boundary.
                pending=np.flatnonzero(~state['done']); lengths=[]
                for start in range(0,len(pending),64):
                    _check(budget,c)
                    lengths.extend(len(tok.encode(unique[j],add_special_tokens=True,truncation=True,max_length=cap)) for j in pending[start:start+64])
                order=pending[np.argsort(lengths,kind='stable')]
                i=0; bs=c['eval_batch_size']; started=last=time.monotonic()
                print(f'Giga LoRA {root.name} {role}: {int(state["done"].sum())}/{len(unique)} unique cached',flush=True)
                try:
                    while i<len(order):
                        _check(budget,c); rows=order[i:i+bs]
                        try:
                            values=embed(model,tok,[unique[j] for j in rows],cap,device).cpu().numpy().astype(np.float32)
                        except torch.cuda.OutOfMemoryError:
                            if bs<=1: raise
                            bs=max(1,bs//2); gc.collect()
                            if str(device).startswith('cuda'): torch.cuda.empty_cache()
                            continue
                        ge._validate_vectors(values,len(rows),int(model.config.hidden_size))
                        state['values'][rows]=values; state['done'][rows]=True
                        state['pending'].extend(rows.tolist()); i+=len(rows)
                        if len(state['pending'])>=c['checkpoint_items'] or time.monotonic()-last>=c['checkpoint_seconds']:
                            ge._commit(state); last=time.monotonic()
                            eta=(len(order)-i)*(last-started)/max(1,i)
                            print(f'Giga LoRA {root.name} {role}: {int(state["done"].sum())}/{len(unique)} unique; ETA {eta/60:.1f} min',flush=True)
                finally:
                    ge._commit(state)
                if not state['done'].all(): raise GigaLoRABudget('Incomplete inference role')
                vectors[role]=state['values']
        # No pair-aligned vectors are serialized: cosine rows are produced here.
        values=np.empty(len(frame),np.float32)
        for start in range(0,len(frame),4096):
            stop=min(len(frame),start+4096)
            values[start:stop]=np.einsum('ij,ij->i',vectors['query'][maps['query'][start:stop]],vectors['document'][maps['document'][start:stop]])
        if not np.isfinite(values).all(): raise FloatingPointError('Nonfinite Giga LoRA cosines')
        ge._atomic_npz(root/'cosines.npz',signature=np.asarray(key),ids=frame.id.to_numpy(dtype=str),cosines=values)
        return values
    finally:
        model.train(mode)


def _query_z(frame,values):
    s=pd.Series(np.asarray(values,dtype=np.float64))
    group=s.groupby(frame.query_id.to_numpy(),sort=False)
    return ((s-group.transform('mean'))/(group.transform('std').fillna(0)+1e-6)).to_numpy(np.float32)


def _adapter_digest(model):
    return _key({n:_digest(p.numpy()) for n,p in adapter_state(model).items()})


def run_giga_jina_lora(tr,te,folds,cache_dir,budget,teacher_scores,config=None,device='cuda'):
    """Train two DEV-only models; return complete query-normalized score pairs.

    ``folds`` are the original five component folds; the last held-out fold is
    the immutable audit. Teachers are fixed zero-shot raw ``giga_cosine`` and
    frozen ``jina35_mean`` train arrays. Jina is standardized within query once,
    without labels. Audit labels are never hashed or read.
    """
    started=time.monotonic(); c=_config(config); tr,te=_clean(tr,te)
    dev,audit,source_folds=make_folds(tr,folds)
    if 'relevance' not in tr or not tr.iloc[dev].relevance.isin([0,1]).all():
        raise ValueError('Development labels must be binary')
    teacher=_teacher(teacher_scores,len(tr),tr.query_id.to_numpy())
    request_key,binding=_binding(tr,te,dev,audit,source_folds,c,teacher,teacher_scores)
    out=Path(cache_dir); out.mkdir(parents=True,exist_ok=True)
    plan_path=out/'plan.json'; plan=json.loads(plan_path.read_text()) if plan_path.is_file() else None
    if plan and plan['request_key']!=request_key: raise ValueError('Giga LoRA inputs/config changed; use a new cache directory')
    resolved=plan['resolved_config'] if plan else dict(c)
    signature=plan['signature'] if plan else None
    report=dict(status='initializing',source='giga_jina_lora',model=MODEL_NAME,revision=MODEL_REVISION,
        request_key=request_key,folds=[],no_audit_training=True,
        training='two GroupKFold source models on DEV only; one epoch',
        objective='RankNet temperature .05 + .15 KL(frozen Jina query-z || student cosine/.05)',
        sampling='two hash positives; union of top-two Giga/top-two Jina negatives, Jina fill to four',
        score_scope='DEV OOF; audit/test mean raw cosines, then within-query z',
        validation_caveat='Downstream meta-CV with learned OOF inputs is approximate; each source held-out target is excluded from that source training.')
    def complete():
        path=out/'complete.npz'
        if signature is None or not path.is_file(): return {}
        with np.load(path,allow_pickle=False) as z:
            if str(z['signature'].item())!=signature or not np.array_equal(z['train_ids'],tr.id.to_numpy(dtype=str)) or not np.array_equal(z['test_ids'],te.id.to_numpy(dtype=str)):
                raise ValueError('Complete Giga LoRA score identity changed')
            a=z['train'].copy(); b=z['test'].copy(); hashes=z['weight_digests'].astype(str)
        if a.shape!=(len(tr),) or b.shape!=(len(te),) or not np.isfinite(a).all() or not np.isfinite(b).all() or len(hashes)!=2:
            raise ValueError('Incomplete Giga LoRA final source')
        for k,h in enumerate(hashes):
            s=_read_checkpoint(out/f'fold{k}_train.pt',signature)
            if s is None or not s['complete'] or _key({n:_digest(p.numpy()) for n,p in s['adapter'].items()})!=h:
                raise ValueError('Complete source adapter checkpoint changed')
        return {'giga_jina_lora':(a,b)}
    def finish(status):
        report.update(status=status,signature=signature,seconds=time.monotonic()-started,resolved_config=resolved,
            available_neural_seconds=_available(budget,c))
        ge._atomic_json(out/'report.json',report)
        return complete(),report
    if complete():
        old_report=out/'report.json'
        if old_report.is_file():
            previous=json.loads(old_report.read_text())
            if previous.get('signature')==signature: report.update(previous)
        return finish('complete_cached')
    if str(device).startswith('cuda') and not torch.cuda.is_available():
        report['reason']='A CUDA GPU is required for the 3B training experiment'
        return finish('no_cuda')
    # QueryStore carries only DEV labels. Audit slots are explicitly erased.
    safe=tr.copy(); safe['relevance']=0.; safe.loc[dev,'relevance']=tr.iloc[dev].relevance.to_numpy(np.float32)
    store=QueryStore(safe,teacher)
    phase_frames=[]; all_phase_text=[]
    for k,(train_rows,held_rows) in enumerate(source_folds):
        phases=[('heldout',tr.iloc[held_rows].reset_index(drop=True)),
                ('audit',tr.iloc[audit].reset_index(drop=True)),('test',te)]
        phase_frames.append(phases); all_phase_text.extend(_phase_texts(frame) for _,frame in phases)
    model=tok=None
    try:
        _check(budget,c)
        if plan is None:
            start=time.monotonic(); model,tok,targets=_load_model(device,c); _sync(device)
            load_seconds=time.monotonic()-start
            rates=_benchmark(model,tok,store,dev,all_phase_text,c,out,budget,device,load_seconds)
            report['preflight']=rates
            best=None
            for cap in [None]:
                train_count=sum(len(_training_keys(store,a,c['seed']+k,cap)) for k,(a,b) in enumerate(source_folds))
                counts={r:sum(len(set(phase[r])) for phase in all_phase_text) for r in ['query','document']}
                updates=sum(math.ceil(len(_training_keys(store,a,c['seed']+k,cap))/c['accum']) for k,(a,b) in enumerate(source_folds))
                estimate=c['safety']*(train_count*rates['train_seconds_per_query']+
                    sum(counts[r]*rates['inference_seconds_per_item'][r] for r in counts)+
                    2*rates['model_load_seconds']+rates['checkpoint_io_seconds']*(2+math.ceil(updates/c['checkpoint_updates'])))
                report.setdefault('cap_forecasts',[]).append(dict(cap=cap,train_queries=train_count,unique_eval_items=counts,seconds=estimate))
                if estimate<=_available(budget,c): best=cap; break
            if not report['cap_forecasts'] or report['cap_forecasts'][-1]['seconds']>_available(budget,c):
                return finish('refused_eta')
            resolved={**c,'max_train_queries':best,'train_forward_batch':rates['train_forward_batch'],
                'eval_batch_size':rates['eval_batch_size']}
            signature=_key(dict(request_key=request_key,resolved_config=resolved))
            plan=dict(request_key=request_key,signature=signature,resolved_config=resolved,benchmark=rates,binding=binding)
            ge._atomic_json(plan_path,plan)
        else:
            rates=plan['benchmark']; report['preflight']=rates
        # Recompute the whole remaining workload, including both audit/test passes.
        train_remaining=0; eval_seconds=0.; checkpoints=0
        for k,(a,b) in enumerate(source_folds):
            keys=_training_keys(store,a,c['seed']+k,resolved['max_train_queries'])
            state=_read_checkpoint(out/f'fold{k}_train.pt',signature)
            cursor=int(state['cursor']) if state else 0
            if state and state['query_order']!=keys: raise ValueError('Resume query order differs')
            train_remaining+=len(keys)-cursor
            checkpoints+=math.ceil(max(0,len(keys)-cursor)/c['accum']/c['checkpoint_updates'])
            weight=_key({n:_digest(p.numpy()) for n,p in state['adapter'].items()}) if state and state['complete'] else None
            for phase,frame in phase_frames[k]:
                root=out/f'fold{k}_{phase}'
                key=_phase_signature(signature,weight,phase,frame,c) if weight else None
                if key and _load_cosines(root/'cosines.npz',key,frame.id) is not None: continue
                for role,texts in _phase_texts(frame).items():
                    n=len(set(texts)); manifest=root/role/'manifest.json'
                    if key and manifest.is_file():
                        m=json.loads(manifest.read_text())
                        if m['signature']!=key: raise ValueError('Inference resume weight binding changed')
                        n-=int(m['complete_items'])
                    eval_seconds+=n*rates['inference_seconds_per_item'][role]
        estimate=c['safety']*(train_remaining*rates['train_seconds_per_query']+eval_seconds+
            2*rates['model_load_seconds']+(checkpoints+2)*rates['checkpoint_io_seconds'])
        report.update(estimated_mandatory_seconds=estimate,remaining_train_queries=train_remaining)
        print(f'Giga LoRA whole two-fold ETA {estimate/60:.1f} min; available {_available(budget,c)/60:.1f} min; cap {resolved["max_train_queries"]}',flush=True)
        if estimate>_available(budget,c): return finish('refused_eta')
        # Benchmark model is never a training initialization or a fold artifact.
        model=tok=None; gc.collect()
        if str(device).startswith('cuda'): torch.cuda.empty_cache()
        raw_train=np.full(len(tr),np.nan,np.float32); audits=[]; tests=[]; hashes=[]
        for k,(train_rows,held_rows) in enumerate(source_folds):
            _check(budget,c)
            model,tok,targets=_load_model(device,{**resolved,'seed':c['seed']+k})
            keys=_training_keys(store,train_rows,c['seed']+k,resolved['max_train_queries'])
            record=_train_fold(model,tok,store,keys,resolved,out/f'fold{k}_train.pt',signature,budget,device,k)
            weight=_adapter_digest(model); hashes.append(weight)
            predictions={phase:_infer_phase(model,tok,frame,out/f'fold{k}_{phase}',signature,weight,phase,resolved,budget,device) for phase,frame in phase_frames[k]}
            raw_train[held_rows]=predictions['heldout']; audits.append(predictions['audit']); tests.append(predictions['test'])
            report['folds'].append(dict(fold=k,targets=targets,weight_digest=weight,**record))
            model=tok=None; gc.collect()
            if str(device).startswith('cuda'): torch.cuda.empty_cache()
        raw_train[audit]=np.mean(audits,axis=0); raw_test=np.mean(tests,axis=0)
        if not np.isfinite(raw_train).all() or not np.isfinite(raw_test).all(): raise FloatingPointError('Incomplete two-fold source')
        ge._atomic_npz(out/'complete.npz',signature=np.asarray(signature),
            train_ids=tr.id.to_numpy(dtype=str),test_ids=te.id.to_numpy(dtype=str),
            train=_query_z(tr,raw_train),test=_query_z(te,raw_test),weight_digests=np.asarray(hashes))
        return finish('complete')
    except GigaLoRABudget as exc:
        report['reason']=str(exc); return finish('budget_stopped')
    except (FloatingPointError,torch.cuda.OutOfMemoryError) as exc:
        report['reason']=str(exc); return finish('numerical_or_oom_stopped')
    finally:
        model=tok=None; gc.collect()
        if str(device).startswith('cuda'): torch.cuda.empty_cache()


In [ ]:
%%writefile qwen8_focus_v17.py
"""Label-free, whole-query sparse NF4 Qwen3-8B frozen encoder cascade.

Only the native backbone runs: no full vocabulary logits, fine tuning, labels,
external inference, or dataset downloads. Query caches publish complete groups
only; interrupted groups remain private until resumed and completed.
"""
from __future__ import annotations
import gc
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import time
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen3-Reranker-8B'
MODEL_REVISION = '77d193c791ed757ca307ee72715aa132723da912'
HIDDEN_SIZE = 4096
TASK = 'Given a web search query, retrieve relevant passages that answer the query'
PREFIX = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
SUFFIX = '<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'
PRIORITY_SOURCES = ('jina35_mean', 'giga_cosine', 'rr4')
QUANTIZATION = dict(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype='float16',
    llm_int8_skip_modules=['lm_head'])
CONFIG = dict(version='qwen8-focus-v17-1', max_total_tokens=768,
    max_query_tokens=128, eval_batch_size=8, benchmark_items=16,
    reserve_seconds=1800., safety=1.35, progress_seconds=30.)
POLICY = dict(priority='.5 mean(abs(rank(Jina)-rank(Giga))) + .3 mean(abs(rank(Jina)-rank(rr4))) + .2 (topJina!=topGiga)',
    rank='descending average ties, normalized by n-1', eligibility='at least two original candidates',
    within_split='difficulty descending, then string query ID',
    merge='zero-based query rank / eligible split query count; train first on exact ties',
    scope='every original candidate of each selected query; only complete query groups published')

class FocusBudget(RuntimeError):
    pass


def _config(config=None):
    if set(config or {}) - set(CONFIG):
        raise ValueError('Unknown Qwen8 focus configuration')
    c = {**CONFIG, **(config or {})}
    if c['version'] != CONFIG['version'] or c['max_total_tokens'] != 768 or c['max_query_tokens'] != 128 or c['eval_batch_size'] != 8:
        raise ValueError('Fixed V17 policy requires total768, query128, initial batch8')
    if not isinstance(c['benchmark_items'], (int, np.integer)) or c['benchmark_items'] < 1:
        raise ValueError('Positive benchmark_items required')
    for k in ('reserve_seconds', 'safety', 'progress_seconds'):
        if not np.isfinite(c[k]) or c[k] <= 0:
            raise ValueError('Positive finite configuration required: ' + k)
    if c['reserve_seconds'] < 1800 or c['safety'] < 1.1:
        raise ValueError('Thirty-minute CPU reserve and inference safety required')
    return c


def _key(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode()).hexdigest()


def _digest(value):
    value = np.ascontiguousarray(value)
    return hashlib.sha256(str(value.dtype).encode() + str(value.shape).encode() + value.tobytes()).hexdigest()


def _strings(values):
    h = hashlib.sha256()
    for x in values:
        b = str(x).encode('utf-8'); h.update(len(b).to_bytes(8, 'big')); h.update(b)
    return h.hexdigest()


def _version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


def _versions():
    # Required bnb version remains explicit when a CPU replay has no bnb install.
    return {**{p: _version(p) for p in ('torch', 'transformers', 'numpy')}, 'bitsandbytes': '0.48.2'}


def _atomic_json(path, value):
    path = Path(path); tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(value, sort_keys=True, indent=2)); os.replace(tmp, path)


def _atomic_npz(path, **arrays):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('wb') as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)


def _available(budget, c):
    if hasattr(budget, 'remaining'):
        return max(0., float(budget.remaining()) - c['reserve_seconds'])
    if hasattr(budget, 'for_neural'):
        return max(0., float(budget.for_neural()))
    raise TypeError('Budget must expose remaining() or pre-reserved for_neural()')


def _check(budget, c, required=0.):
    if _available(budget, c) <= max(0., required):
        raise FocusBudget('Thirty-minute CPU reserve reached or next complete query does not fit')


def _sync(device):
    if str(device).startswith('cuda'):
        torch.cuda.synchronize(device)


def _clean(tr, te):
    cols = ['id', 'query_id', 'query_c', 'title_c', 'text_c']
    frames = []
    for frame in (tr, te):
        if not set(cols) <= set(frame.columns):
            raise ValueError('Canonical pair columns required')
        # Select first: no label or other metadata is ever read, copied, or hashed.
        f = frame.loc[:, cols].reset_index(drop=True).copy()
        if f.id.isna().any() or f.query_id.isna().any():
            raise ValueError('Missing pair/query ID')
        f['id'] = f.id.astype(str); f['query_id'] = f.query_id.astype(str)
        if not f.id.is_unique:
            raise ValueError('Duplicate pair ID')
        for col in cols[2:]:
            f[col] = f[col].fillna('').astype(str)
        if f.query_c.str.strip().eq('').any() or (f.groupby('query_id').query_c.nunique() > 1).any():
            raise ValueError('Empty or conflicting query text')
        frames.append(f)
    if set(frames[0].id) & set(frames[1].id) or set(frames[0].query_id) & set(frames[1].query_id):
        raise ValueError('Train/test pair or query overlap')
    return tuple(frames)


def _priority_inputs(tr, te, pairs, kinds):
    scores = {}
    for name in PRIORITY_SOURCES:
        if kinds.get(name) != 'zero_shot':
            raise ValueError('Priority requires verified zero_shot source: ' + name)
        pair = pairs[name]
        if not isinstance(pair, (tuple, list)) or len(pair) != 2:
            raise ValueError('Train/test source pair required: ' + name)
        values = tuple(np.asarray(a, dtype=np.float64) for a in pair)
        if any(a.shape != (len(f),) or not np.isfinite(a).all() for a, f in zip(values, (tr, te))):
            raise ValueError('Invalid label-free priority scores: ' + name)
        scores[name] = values
    return scores


def build_schedule(tr, te, scores):
    """Complete deterministic plan from three frozen sources, without labels."""
    result = []
    for side, frame in enumerate((tr, te)):
        entries = []
        for qid, ix in frame.groupby('query_id', sort=False).indices.items():
            rows = np.asarray(ix, dtype=np.int64)
            if len(rows) < 2:
                continue
            vectors = [scores[n][side][rows] for n in PRIORITY_SOURCES]
            rank = [pd.Series(x).rank(ascending=False, method='average').to_numpy(np.float64) / (len(rows) - 1) for x in vectors]
            # Subtraction cancels the rank origin; pair-order argmax breaks exact ties.
            difficulty = .5 * float(np.abs(rank[0] - rank[1]).mean()) + .3 * float(np.abs(rank[0] - rank[2]).mean()) + .2 * float(np.argmax(vectors[0]) != np.argmax(vectors[1]))
            entries.append(dict(split='train' if side == 0 else 'test', query_id=str(qid),
                rows=rows.tolist(), difficulty=difficulty))
        entries.sort(key=lambda e: (-e['difficulty'], e['query_id']))
        for rank, item in enumerate(entries):
            item['rank'] = rank; item['percentile'] = rank / len(entries)
        result.extend(entries)
    result.sort(key=lambda e: (e['percentile'], 0 if e['split'] == 'train' else 1, e['query_id']))
    return result


def _binding(tr, te, scores, c):
    binding = dict(model=MODEL_NAME, model_revision=MODEL_REVISION, quantization=QUANTIZATION,
        config=c, prompt=dict(task=TASK, prefix=PREFIX, suffix=SUFFIX), policy=POLICY,
        versions=_versions(), canonical={s: {col: _strings(f[col]) for col in f.columns}
            for s, f in (('train', tr), ('test', te))},
        priority={n: [_digest(a) for a in scores[n]] for n in PRIORITY_SOURCES})
    return _key(binding), binding


class PairTokens:
    """Official prompt with an exact total cap and untouched prefix/suffix."""
    def __init__(self, frame, tokenizer, c):
        self.frame = frame; self.tok = tokenizer; self.c = c; self.cache = {}
        tokenizer.padding_side = 'left'
        self.prefix = tokenizer.encode(PREFIX, add_special_tokens=False)
        self.suffix = tokenizer.encode(SUFFIX, add_special_tokens=False)

    def tokens(self, row):
        row = int(row)
        if row not in self.cache:
            f = self.frame.iloc[row]; query = f.query_c
            qids = self.tok.encode(query, add_special_tokens=False)
            if len(qids) > self.c['max_query_tokens']:
                query = self.tok.decode(qids[:self.c['max_query_tokens']], skip_special_tokens=False)
            # Ensure decode/re-encode boundaries cannot exceed the query cap.
            while len(self.tok.encode(query, add_special_tokens=False)) > self.c['max_query_tokens']:
                qids = self.tok.encode(query, add_special_tokens=False)[:self.c['max_query_tokens'] - 1]
                query = self.tok.decode(qids, skip_special_tokens=False)
            head = f'<Instruct>: {TASK}\n<Query>: {query}\n<Document>: '
            doc = f.title_c + '\n' + f.text_c
            cap = self.c['max_total_tokens'] - len(self.prefix) - len(self.suffix)
            body = self.tok.encode(head + doc, add_special_tokens=False)
            if len(body) > cap:
                d = self.tok.encode(doc, add_special_tokens=False)
                d = d[:max(0, cap - len(self.tok.encode(head, add_special_tokens=False)))]
                while True:
                    body = self.tok.encode(head + self.tok.decode(d, skip_special_tokens=False), add_special_tokens=False)
                    if len(body) <= cap:
                        break
                    if not d:
                        raise ValueError('Prompt/query exceeds fixed total token cap')
                    d = d[:max(0, len(d) - max(1, len(body) - cap))]
            ids = self.prefix + body + self.suffix
            if not ids or len(ids) > self.c['max_total_tokens']:
                raise ValueError('Invalid capped official prompt')
            self.cache[row] = np.asarray(ids, dtype=np.int32)
        return self.cache[row]

    def batch(self, rows, device):
        batch = self.tok.pad({'input_ids': [self.tokens(i).tolist() for i in rows]},
            padding=True, pad_to_multiple_of=8, return_tensors='pt', return_attention_mask=True)
        return {k: v.to(device) for k, v in batch.items() if k in ('input_ids', 'attention_mask')}


class FrozenPairEncoder(nn.Module):
    """FP32 pretrained yes-minus-no score and FP16 final hidden vector."""
    def __init__(self, backbone, yes_minus_no):
        super().__init__(); self.backbone = backbone; self.config = backbone.config
        if yes_minus_no.shape != (self.config.hidden_size,) or not torch.isfinite(yes_minus_no).all():
            raise ValueError('Invalid unquantized pretrained yes/no head rows')
        self.register_buffer('yes_minus_no', yes_minus_no.detach().float().clone())
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()

    def forward(self, input_ids, attention_mask):
        if not torch.all(attention_mask.sum(dim=1) > 0):
            raise ValueError('Empty padded input')
        with torch.inference_mode():
            h = self.backbone(input_ids=input_ids, attention_mask=attention_mask, use_cache=False, return_dict=True).last_hidden_state
            pos = (attention_mask.long() * torch.arange(h.shape[1], device=h.device)[None, :]).max(dim=1).values
            last = h[torch.arange(len(h), device=h.device), pos]
            with torch.autocast(device_type=last.device.type, enabled=False):
                score = F.linear(last.float(), self.yes_minus_no.float().unsqueeze(0)).squeeze(-1)
            return score, last.to(torch.float16)


def _load_model(device, c):
    if not str(device).startswith('cuda') or not torch.cuda.is_available():
        raise FocusBudget('CUDA required to load frozen NF4 Qwen8; cached groups remain replayable')
    if _version('bitsandbytes') != '0.48.2':
        raise RuntimeError('Pinned bitsandbytes==0.48.2 required for NF4 execution')
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION,
        padding_side='left', trust_remote_code=False)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules=['lm_head'])
    lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION,
        trust_remote_code=False, quantization_config=quant, torch_dtype=torch.float16,
        attn_implementation='sdpa', low_cpu_mem_usage=True, device_map={'': str(device)})
    expected = dict(model_type='qwen3', num_hidden_layers=36, hidden_size=4096,
        vocab_size=151669, num_attention_heads=32, num_key_value_heads=8,
        head_dim=128, intermediate_size=12288)
    if any(getattr(lm.config, k, None) != v for k, v in expected.items()):
        raise ValueError('Pinned Qwen8 architecture changed')
    weight = lm.lm_head.weight
    if (tuple(weight.shape) != (151669, 4096) or not isinstance(lm.lm_head, nn.Linear)
            or weight.dtype not in (torch.float16, torch.float32, torch.bfloat16)
            or hasattr(weight, 'quant_state') or 'bitsandbytes' in type(lm.lm_head).__module__):
        raise ValueError('lm_head must remain an unquantized full vocabulary matrix')
    yes = tok.convert_tokens_to_ids('yes'); no = tok.convert_tokens_to_ids('no')
    if yes == no or tok.encode('yes', add_special_tokens=False) != [yes] or tok.encode('no', add_special_tokens=False) != [no]:
        raise ValueError('Official lowercase yes/no token identities changed')
    row = weight[yes].detach().float() - weight[no].detach().float()
    if not bool(getattr(lm, 'is_loaded_in_4bit', False)):
        raise ValueError('NF4 quantization was not applied')
    backbone = lm.model; backbone.config.use_cache = False
    # Never call .to on a quantized module. Delete the 1.16 GiB vocabulary head.
    del weight; del lm
    model = FrozenPairEncoder(backbone, row); del row
    model.runtime_info = dict(quantization='NF4 double-quant; FP16 compute; unquantized lm_head rows extracted then lm_head freed',
        bitsandbytes=_version('bitsandbytes'), cuda_device=torch.cuda.get_device_name(device),
        all_weights_frozen=all(not p.requires_grad for p in model.parameters()),
        cuda_allocated_gb=torch.cuda.memory_allocated(device) / 2**30)
    gc.collect(); torch.cuda.empty_cache()
    return model, tok


def _query_identity(request, entry, frame):
    ids = frame.iloc[entry['rows']].id.to_numpy(dtype=str)
    signature = _key(dict(request=request, split=entry['split'], query_id=entry['query_id'], pair_ids=_strings(ids)))
    return signature, ids


def _query_path(out, entry):
    return Path(out) / 'queries' / (entry['split'] + '_' + _key(entry['query_id']) + '.npz')


def _query_sha(signature, ids, scores, hidden, done, complete):
    return _key(dict(signature=signature, pair_ids=_strings(ids), scores=_digest(scores),
        hidden=_digest(hidden), done=_digest(done), complete=bool(complete)))


def _save_query(path, signature, ids, scores, hidden, done):
    complete = bool(done.all())
    _atomic_npz(path, signature=np.asarray(signature), pair_ids=ids,
        scores=scores, hidden=hidden, done=done, complete=np.asarray(complete),
        sha256=np.asarray(_query_sha(signature, ids, scores, hidden, done, complete)))


def _read_query(path, signature, ids):
    if not Path(path).is_file():
        return None
    try:
        with np.load(path, allow_pickle=False) as z:
            got = str(z['signature'].item()); cached_ids = z['pair_ids'].copy()
            scores = z['scores'].copy(); hidden = z['hidden'].copy(); done = z['done'].copy()
            complete_array = z['complete']
            if complete_array.shape != () or complete_array.dtype != np.bool_:
                raise ValueError('Invalid complete flag')
            complete = bool(complete_array.item()); sha = str(z['sha256'].item())
    except Exception as exc:
        raise ValueError('Unreadable query cache: ' + str(path)) from exc
    if (got != signature or not np.array_equal(cached_ids, ids)
            or scores.shape != (len(ids),) or scores.dtype != np.float32
            or hidden.shape != (len(ids), HIDDEN_SIZE) or hidden.dtype != np.float16
            or done.shape != (len(ids),) or done.dtype != np.bool_
            or complete != bool(done.all()) or not np.isfinite(scores).all()
            or not np.isfinite(hidden).all() or np.any(scores[~done] != 0) or np.any(hidden[~done] != 0)
            or sha != _query_sha(signature, cached_ids, scores, hidden, done, complete)):
        raise ValueError('Query cache binding, payload SHA, or completeness failed: ' + str(path))
    return dict(scores=scores, hidden=hidden, done=done, complete=complete)


def _arrays(model, tokens, rows, device):
    scores, hidden = model(**tokens.batch(rows, device))
    a = scores.detach().float().cpu().numpy(); h = hidden.detach().to(torch.float16).cpu().numpy()
    if a.shape != (len(rows),) or h.shape != (len(rows), HIDDEN_SIZE) or not np.isfinite(a).all() or not np.isfinite(h).all():
        raise FloatingPointError('Nonfinite or incorrectly shaped frozen Qwen8 output')
    return a, h


def _clear(device):
    gc.collect()
    if str(device).startswith('cuda') and torch.cuda.is_available():
        torch.cuda.empty_cache()


def _benchmark(model, tokens, pending, c, budget, device, batch_size):
    # Actual upper-half through longest text pairs, using exactly the production
    # last-hidden-only forward. Cost governs the next query, never a full-run veto.
    by_length = sorted(pending, key=lambda i: len(tokens.frame.text_c.iloc[i]) + len(tokens.frame.title_c.iloc[i]))
    positions = np.unique(np.linspace(len(by_length) // 2, len(by_length) - 1,
        min(c['benchmark_items'], len(by_length))).astype(int))
    rows = np.asarray([by_length[i] for i in positions], dtype=np.int64)
    lengths = np.asarray([len(tokens.tokens(i)) for i in rows])
    rows = rows[np.argsort(lengths, kind='stable')]
    bs = batch_size
    while True:
        try:
            _check(budget, c); _arrays(model, tokens, rows[-bs:], device); _sync(device)
            start = time.monotonic()
            for j in range(0, len(rows), bs):
                _check(budget, c); _arrays(model, tokens, rows[j:j + bs], device)
            _sync(device); seconds = max(1e-6, time.monotonic() - start)
            return dict(seconds_per_pair=seconds / len(rows), items=len(rows),
                total_tokens=[int(len(tokens.tokens(i))) for i in rows], batch_size=bs,
                sampling='upper-half through maximum original text lengths; actual capped prompts',
                path='native last hidden, FP32 yes-minus-no, FP16 hidden; labels absent; frozen eval'), bs
        except torch.cuda.OutOfMemoryError:
            if bs <= 1:
                raise
            bs //= 2; _clear(device)


def _assemble(tr, te, schedule, out, request):
    records = {'train': [], 'test': []}; counts = {'train': 0, 'test': 0}
    frames = {'train': tr, 'test': te}
    masks = {side: np.zeros(len(f), dtype=bool) for side, f in frames.items()}
    frozen = {side: np.zeros(len(f), dtype=np.float32) for side, f in frames.items()}
    for e in schedule:
        side = e['split']; signature, ids = _query_identity(request, e, frames[side])
        state = _read_query(_query_path(out, e), signature, ids)
        if state is not None and state['complete']:
            rows = np.asarray(e['rows'], dtype=np.int64)
            masks[side][rows] = True; frozen[side][rows] = state['scores']
            records[side].append((rows, state['hidden'])); counts[side] += 1
    coverage = {side: dict(complete_queries=counts[side], eligible_queries=sum(e['split'] == side for e in schedule),
        rows=int(masks[side].sum()), total_rows=len(frames[side]),
        row_fraction=float(masks[side].mean()) if len(frames[side]) else 0.) for side in frames}
    if not counts['train'] or not counts['test']:
        return None, coverage
    bundle = {}
    for side, frame in frames.items():
        selected = np.flatnonzero(masks[side]); mapping = np.full(len(frame), -1, dtype=np.int64)
        mapping[selected] = np.arange(len(selected))
        hidden = np.empty((len(selected), HIDDEN_SIZE), dtype=np.float16)
        for rows, values in records[side]:
            hidden[mapping[rows]] = values
        bundle[side + '_mask'] = masks[side]; bundle[side + '_hidden'] = hidden
        bundle[side + '_frozen'] = frozen[side]
        bundle[side + '_ids'] = frame.id.to_numpy(dtype=str)
        bundle[side + '_rows'] = selected
    bundle['signature'] = _key(dict(request=request, revision=MODEL_REVISION,
        payload={k: _digest(v) for k, v in bundle.items()}))
    bundle['model_revision'] = MODEL_REVISION; bundle['request_signature'] = request
    return bundle, coverage


def run_focus_qwen8(tr, te, pairs, kinds, cache_dir, budget, config=None, device='cuda'):
    """Return (sparse frozen encoder bundle or None, coverage/report).

    Labels are never inspected. Priority uses only the three named zero-shot
    streams. The compact hidden matrices follow flatnonzero(full mask) order.
    Full vectors contain explicit zero sentinels outside complete query masks.
    Reusing a cache with more time grows the mask and changes bundle.signature.
    """
    started = time.monotonic(); c = _config(config); tr, te = _clean(tr, te)
    scores = _priority_inputs(tr, te, pairs, kinds)
    request, binding = _binding(tr, te, scores, c); schedule = build_schedule(tr, te, scores)
    out = Path(cache_dir); out.mkdir(parents=True, exist_ok=True)
    plan_path = out / 'plan.json'
    if plan_path.is_file():
        plan = json.loads(plan_path.read_text())
        if plan['request_signature'] != request or plan['schedule'] != schedule:
            raise ValueError('Frozen Qwen8 inputs/config/policy/versions changed; use a new cache directory')
    else:
        plan = dict(request_signature=request, binding=binding, schedule=schedule)
        _atomic_json(plan_path, plan)
    report = dict(status='initializing', model=MODEL_NAME, model_revision=MODEL_REVISION,
        request_signature=request, intentional_sparse_query_cascade=True, labels='never read or hashed',
        availability_scope=POLICY['scope'], priority_sources=list(PRIORITY_SOURCES),
        config=c, quantization=QUANTIZATION, versions=_versions(),
        observed_bitsandbytes=_version('bitsandbytes'), benchmark=plan.get('benchmark'),
        planned_queries=len(schedule), hidden_order='flatnonzero(mask); original canonical row order',
        missing_values='zero sentinel outside explicit complete-query mask; never a model prediction',
        cuda_available=bool(torch.cuda.is_available()))
    model = tok = tokens = None
    def finish(status):
        bundle, coverage = _assemble(tr, te, schedule, out, request)
        if model is not None and str(device).startswith('cuda') and torch.cuda.is_available():
            report['cuda_peak_allocated_gb'] = torch.cuda.max_memory_allocated(device) / 2**30
            report['cuda_allocated_gb_before_release'] = torch.cuda.memory_allocated(device) / 2**30
        report.update(status=status, coverage=coverage, seconds=time.monotonic() - started,
            available_gpu_seconds=_available(budget, c), signature=None if bundle is None else bundle['signature'])
        if bundle is not None:
            _atomic_npz(out / 'focused_bundle.npz', **{k: np.asarray(v) for k, v in bundle.items()})
        _atomic_json(out / 'focused_bundle.json', report)
        _atomic_json(out / 'report.json', report)
        return bundle, report
    try:
        pending = []
        frames = {'train': tr, 'test': te}
        offset = {'train': 0, 'test': len(tr)}
        for e in schedule:
            signature, ids = _query_identity(request, e, frames[e['split']])
            state = _read_query(_query_path(out, e), signature, ids)
            if state is None or not state['complete']:
                pending.append(e)
        if not pending:
            return finish('complete_cached')
        if str(device).startswith('cuda') and not torch.cuda.is_available():
            return finish('cached_only_no_cuda')
        _check(budget, c)
        if str(device).startswith('cuda') and torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(device)
        model, tok = _load_model(device, c)
        if any(p.requires_grad for p in model.parameters()):
            raise ValueError('Frozen Qwen8 encoder contains trainable weights')
        model.eval(); report['runtime'] = getattr(model, 'runtime_info', dict(all_weights_frozen=True, execution_device=str(device)))
        print('Qwen8 frozen NF4 runtime:', report['runtime'], flush=True)
        full = pd.concat([tr, te], ignore_index=True); tokens = PairTokens(full, tok, c)
        # Re-benchmark on each live GPU session; a resumed device may differ.
        probe_rows = [r + offset[e['split']] for e in pending for r in e['rows']]
        rates, bs = _benchmark(model, tokens, probe_rows, c, budget, device, c['eval_batch_size'])
        report['benchmark'] = rates; plan['benchmark'] = rates; _atomic_json(plan_path, plan)
        done_queries = len(schedule) - len(pending); last_progress = time.monotonic()
        done_rows = sum(len(e['rows']) for e in schedule) - sum(len(e['rows']) for e in pending)
        for entry in pending:
            side = entry['split']; rows = np.asarray(entry['rows'], dtype=np.int64)
            signature, ids = _query_identity(request, entry, frames[side]); path = _query_path(out, entry)
            state = _read_query(path, signature, ids)
            if state is None:
                state = dict(scores=np.zeros(len(rows), np.float32), hidden=np.zeros((len(rows), HIDDEN_SIZE), np.float16), done=np.zeros(len(rows), bool))
            missing = np.flatnonzero(~state['done'])
            # Conservative next-query guard, with complete groups already kept.
            eta = c['safety'] * rates['seconds_per_pair'] * len(missing) + 2.
            report['next_query_eta_seconds'] = eta; _check(budget, c, eta)
            j = 0
            while j < len(missing):
                _check(budget, c)
                ix = missing[j:j + bs]
                try:
                    a, h = _arrays(model, tokens, rows[ix] + offset[side], device)
                except torch.cuda.OutOfMemoryError:
                    if bs <= 1:
                        raise
                    bs //= 2; report['adaptive_batch_size'] = bs; _clear(device); continue
                state['scores'][ix] = a; state['hidden'][ix] = h; state['done'][ix] = True
                _save_query(path, signature, ids, state['scores'], state['hidden'], state['done'])
                j += len(ix)
                if time.monotonic() - last_progress >= c['progress_seconds']:
                    print(f'Qwen8 focus: {done_queries}/{len(schedule)} complete queries; {done_rows} published rows; '
                          f'coverage {done_rows/max(1,len(full)):.1%}; current query {int(state["done"].sum())}/{len(rows)}; GPU time left {_available(budget,c)/60:.1f}min; '
                          f'next query ETA {eta:.1f}s', flush=True)
                    last_progress = time.monotonic()
            done_queries += 1; done_rows += len(rows)
        report['adaptive_batch_size'] = bs
        return finish('complete')
    except FocusBudget as exc:
        report['reason'] = str(exc); return finish('sparse_budget')
    except torch.cuda.OutOfMemoryError as exc:
        report['reason'] = str(exc); return finish('sparse_oom')
    finally:
        del tokens, tok, model
        _clear(device)


In [ ]:
%%writefile meta_frozen_v4.py
import re
import numpy as np
import pandas as pd

def query_z(frame, values):
    """Order-preserving scale shared by logits, probabilities, and rankers."""
    v = np.asarray(values, dtype=float)
    s = pd.Series(v)
    q = frame["query_id"].reset_index(drop=True)
    g = s.groupby(q, sort=False)
    sd = g.transform("std").fillna(0).to_numpy()
    return np.clip((v - g.transform("mean").to_numpy()) / np.maximum(sd, 1e-6), -8, 8).astype("float32")

def _score_features(frame, score, prefix):
    """Scores plus query confidence, softmax, and within-article peers."""
    values = np.asarray(score, dtype=float)
    z = query_z(frame, values)
    s = pd.Series(values)
    q = frame["query_id"].reset_index(drop=True)
    g = s.groupby(q, sort=False)
    qmax, qmin = g.transform("max"), g.transform("min")
    e = pd.Series(np.exp(np.clip(values - qmax.to_numpy(), -60, 0)))
    softmax = e / e.groupby(q, sort=False).transform("sum").clip(lower=1e-12)
    f = pd.DataFrame({prefix + "_raw": values,
                      prefix + "_z": z,
                      prefix + "_rank": g.rank(ascending=False, method="average", pct=True).to_numpy(),
                      prefix + "_best_gap": values - qmax.to_numpy(),
                      prefix + "_qstd": g.transform("std").fillna(0).to_numpy(),
                      prefix + "_qrange": (qmax - qmin).to_numpy(),
                      prefix + "_softmax": softmax.to_numpy()})
    if "article_id" in frame:
        keys = [q, frame["article_id"].reset_index(drop=True)]
        ag = s.groupby(keys, sort=False)
        count = ag.transform("size")
        f[prefix + "_article_max"] = ag.transform("max").to_numpy()
        f[prefix + "_article_mean"] = ag.transform("mean").to_numpy()
        f[prefix + "_article_gap"] = (s - ag.transform("max")).to_numpy()
        peer = (ag.transform("sum") - s) / np.maximum(1, count - 1)
        f[prefix + "_article_peer"] = peer.where(count > 1, s).to_numpy()
        az = pd.Series(z).groupby(keys, sort=False)
        f[prefix + "_article_max_z"] = az.transform("max").to_numpy()
        f[prefix + "_article_mean_z"] = az.transform("mean").to_numpy()
    return f.replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")

def _coerce_base(Xbase, ntr, nte):
    if isinstance(Xbase, (tuple, list)) and len(Xbase) == 2:
        a, b = (pd.DataFrame(x).reset_index(drop=True) for x in Xbase)
    elif isinstance(Xbase, dict) and "tr" in Xbase and "te" in Xbase:
        a, b = (pd.DataFrame(Xbase[x]).reset_index(drop=True) for x in ("tr", "te"))
    else:
        full = pd.DataFrame(Xbase).reset_index(drop=True)
        if len(full) != ntr + nte:
            raise ValueError("Xbase must be (Xtr, Xte) or concatenated train then test")
        a, b = full.iloc[:ntr].copy(), full.iloc[ntr:].reset_index(drop=True)
    if len(a) != ntr or len(b) != nte or list(a.columns) != list(b.columns):
        raise ValueError("Base feature rows or columns do not match train/test")
    # Identifiers and targets never enter the estimator.
    excluded = {"id", "query_id", "document_id", "article_id", "relevance", "is_train", "fold", "grp"}
    cols = [c for c in a if str(c) not in excluded]
    a, b = a[cols].copy(), b[cols].copy()
    names = [f"x{i}_{re.sub('[^a-zA-Z0-9_]', '_', str(c))}" for i, c in enumerate(cols)]
    a.columns = b.columns = names
    for c in names:
        a[c] = pd.to_numeric(a[c], errors="coerce")
        b[c] = pd.to_numeric(b[c], errors="coerce")
    return (a.replace([np.inf, -np.inf], np.nan).astype("float32"),
            b.replace([np.inf, -np.inf], np.nan).astype("float32"))


In [ ]:
%%writefile refine_cpu.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _coerce_base,_score_features,query_z

def make_features(tr, te, xbase, pairs, order):
    a, b = _coerce_base(xbase,len(tr),len(te))
    af,bf=[a],[b]
    for j,name in enumerate(order):
        af.append(_score_features(tr,pairs[name][0],f's{j}'))
        bf.append(_score_features(te,pairs[name][1],f's{j}'))
    a,b=pd.concat(af,axis=1),pd.concat(bf,axis=1)
    trained=[n for n in order if n in ('bge','bge2','bge3')]
    for label,names in [('all',order),('trained',trained)]:
        if names:
            for frame,dest,side in [(tr,a,0),(te,b,1)]:
                z=np.column_stack([query_z(frame,pairs[n][side]) for n in names])
                dest[label+'_score_mean_z']=z.mean(1)
                dest[label+'_score_disagreement_z']=z.std(1)
    anchor_a=query_z(tr,np.mean([query_z(tr,pairs[n][0]) for n in trained],axis=0))
    anchor_b=query_z(te,np.mean([query_z(te,pairs[n][1]) for n in trained],axis=0))
    return a,b,anchor_a,anchor_b


In [ ]:
%%writefile joint_meta.py
import numpy as np
import pandas as pd

def _peer_features(frame,p,prefix):
    s=pd.Series(np.asarray(p,dtype=float));q=frame.query_id.reset_index(drop=True);a=frame.article_id.reset_index(drop=True)
    keys=[q,a];g=s.groupby(keys,sort=False);maximum=g.transform('max');count=g.transform('size')
    is_max=s.eq(maximum);nmax=is_max.groupby(keys,sort=False).transform('sum')
    lower=s.where(s<maximum).groupby(keys,sort=False).transform('max')
    peer=maximum.where(~(is_max & nmax.eq(1)),lower).fillna(s)
    f=pd.DataFrame({prefix+'_peer_max':peer,prefix+'_peer_margin':s-peer,
                    prefix+'_within_article_rank':g.rank(ascending=False,pct=True),
                    prefix+'_has_peer':count.gt(1).astype(float)})
    return f.astype(np.float32)


In [ ]:
%%writefile v8_fusion.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features

PARAMS=dict(learning_rate=.035,min_data_in_leaf=60,lambda_l1=.1,lambda_l2=15.,
            feature_fraction=.85,bagging_fraction=.85,bagging_freq=1,verbosity=-1,
            num_threads=2,seed=1701,deterministic=True,force_col_wise=True)

def build_features_v8(tr,te,X,Xt,new_pairs,new_kind):
    """Append completed raw streams, retaining every original V5 column.

    ``s4_raw`` is the original rr4 field in the verified V5 schema. The exact
    V5 loader establishes this order. Hash-order streams are not accepted.
    """
    if (len(X)!=len(tr) or len(Xt)!=len(te) or X.shape[1]!=300
            or list(X.columns)!=list(Xt.columns) or X.columns.duplicated().any()):
        raise ValueError('V8 requires the exact aligned 300-column V5 feature schema')
    if 's4_raw' not in X:
        raise ValueError('The verified V5 rr4 raw-score column s4_raw is missing')
    if set(new_pairs)!=set(new_kind):
        raise ValueError('Every completed score requires an explicit source kind')
    if any(k not in ('trained','zero_shot') for k in new_kind.values()):
        raise ValueError('Source kinds must be trained or zero_shot')
    for name in new_pairs:
        if not isinstance(name,str) or not (name=='jina35_lora' or name.startswith('gemma')):
            raise ValueError('Only existing jina35_lora and new Gemma streams are accepted: '+str(name))
    af,bf=[X.reset_index(drop=True).copy()],[Xt.reset_index(drop=True).copy()]
    for j,(name,pair) in enumerate(new_pairs.items()):
        if len(pair)!=2:raise ValueError('Train/test score pair required: '+name)
        for frame,p,base,dest in ((tr,pair[0],X,af),(te,pair[1],Xt,bf)):
            p=np.asarray(p)
            if p.shape!=(len(frame),) or not np.isfinite(p).all():
                raise ValueError('Incomplete completed score: '+name)
            prefix='v8_'+str(j)
            dest.append(_score_features(frame,p,prefix))
            dest.append(_peer_features(frame,p,prefix+'_peer'))
            delta=query_z(frame,p)-query_z(frame,base['s4_raw'].to_numpy())
            dest.append(pd.DataFrame({prefix+'_vs_rr4':delta,prefix+'_vs_rr4_abs':np.abs(delta)}))
    a,b=pd.concat(af,axis=1).astype(np.float32),pd.concat(bf,axis=1).astype(np.float32)
    if a.columns.duplicated().any() or list(a.columns)!=list(b.columns):
        raise ValueError('Appended V8 feature schemas differ')
    if not np.isfinite(a.to_numpy()).all() or not np.isfinite(b.to_numpy()).all():
        raise ValueError('V8 features must be finite')
    return a,b


In [ ]:
%%writefile giga_meta_v11.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features

SOURCE_ORDER=('giga_cosine','giga_linear','giga_rank')

def _names(scores,kinds):
    if (set(scores)!=set(kinds) or 'giga_cosine' not in scores or
            not set(scores)<=set(SOURCE_ORDER)):
        raise ValueError('Complete Giga source scores and their explicit kinds are required')
    for n in scores:
        expected='zero_shot' if n=='giga_cosine' else 'trained'
        if kinds[n]!=expected:raise ValueError('Incorrect Giga source kind: '+n)
    return tuple(n for n in SOURCE_ORDER if n in scores)

def build_source_features(frame,base338,scores,kinds):
    """Pure feature builder for one split; no test surrogate or target access."""
    if (len(base338)!=len(frame) or base338.shape[1]!=338 or base338.columns.duplicated().any()
            or 's4_raw' not in base338):raise ValueError('Expected aligned original V8 338-column inputs')
    if frame.id.isna().any() or not frame.id.is_unique or frame.query_id.isna().any():raise ValueError('Malformed pair/query IDs')
    if not {'id','query_id','article_id'}<=set(frame):raise ValueError('Pair/query/article metadata are required')
    names=_names(scores,kinds)
    parts=[base338.reset_index(drop=True).copy()]
    for j,name in enumerate(names):
        p=np.asarray(scores[name])
        if p.shape!=(len(frame),) or not np.issubdtype(p.dtype,np.number) or not np.isfinite(p).all():
            raise ValueError('A complete real score vector is required: '+name)
        prefix='giga'+str(j)
        parts.extend([_score_features(frame,p,prefix),_peer_features(frame,p,prefix+'_peer')])
        delta=query_z(frame,p)-query_z(frame,base338['s4_raw'].to_numpy())
        parts.append(pd.DataFrame({prefix+'_vs_rr4':delta,prefix+'_vs_rr4_abs':np.abs(delta)}))
    result=pd.concat(parts,axis=1).astype(np.float32)
    if result.shape[1]!=338+19*len(names) or result.columns.duplicated().any() or not np.isfinite(result.to_numpy()).all():raise ValueError('Invalid Giga feature matrix')
    if not np.array_equal(result.iloc[:,:338].to_numpy(),base338.to_numpy(np.float32)):raise ValueError('Original V8 feature prefix changed')
    return result


In [ ]:
%%writefile giga_fusion_v13.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features

def build_features_v13(tr, te, X, Xt, newpairs, kinds):
    """Append 13 score, four article-peer and two rr4-delta features."""
    if set(newpairs) != {'giga_lora'} or kinds != {'giga_lora': 'trained'}:
        raise ValueError('Exactly one completed trained giga_lora source is required')
    if (len(X) != len(tr) or len(Xt) != len(te) or X.shape[1] != 395 or Xt.shape[1] != 395
            or list(X) != list(Xt) or X.columns.duplicated().any() or 's4_raw' not in X):
        raise ValueError('Expected the exact aligned original 395-column V11 input schema')
    pair = newpairs['giga_lora']
    if not isinstance(pair, (tuple, list)) or len(pair) != 2:
        raise ValueError('Complete real train/test LoRA score pair is required')
    outputs = []
    for frame, base, score in [(tr, X, pair[0]), (te, Xt, pair[1])]:
        if not {'id', 'query_id', 'article_id'} <= set(frame):
            raise ValueError('Pair/query/article metadata are required')
        if frame.id.isna().any() or not frame.id.is_unique or frame.query_id.isna().any() or frame.article_id.isna().any():
            raise ValueError('Malformed pair/query/article IDs')
        p = np.asarray(score)
        if p.shape != (len(frame),) or not np.issubdtype(p.dtype, np.number) or not np.isfinite(p).all():
            raise ValueError('Incomplete real giga_lora scores')
        if not np.isfinite(base.to_numpy()).all():
            raise ValueError('Original V11 source features must be finite')
        prefix = 'giga_lora'
        delta = query_z(frame, p) - query_z(frame, base['s4_raw'].to_numpy())
        result = pd.concat([base.reset_index(drop=True).copy(), _score_features(frame, p, prefix),
                            _peer_features(frame, p, prefix + '_peer'),
                            pd.DataFrame({prefix + '_vs_rr4': delta, prefix + '_vs_rr4_abs': np.abs(delta)})], axis=1).astype(np.float32)
        if result.shape[1] != 414 or result.columns.duplicated().any() or not np.isfinite(result.to_numpy()).all():
            raise ValueError('Invalid 414-column Giga LoRA feature matrix')
        if not np.array_equal(result.iloc[:, :395].to_numpy(), base.to_numpy(np.float32)):
            raise ValueError('Original 395-column V11 feature prefix changed')
        outputs.append(result)
    if list(outputs[0]) != list(outputs[1]):
        raise ValueError('Train/test fusion feature schemas differ')
    return tuple(outputs)


In [ ]:
%%writefile giga_fusion_v15.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features

def build_features_v15(tr, te, X, Xt, newpairs, kinds):
    """Append 13 score, four article-peer and two rr4-delta features."""
    if set(newpairs) != {'giga_jina_lora'} or kinds != {'giga_jina_lora': 'trained'}:
        raise ValueError('Exactly one completed trained giga_jina_lora source is required')
    if (len(X) != len(tr) or len(Xt) != len(te) or X.shape[1] != 414 or Xt.shape[1] != 414
            or list(X) != list(Xt) or X.columns.duplicated().any() or 's4_raw' not in X):
        raise ValueError('Expected the exact aligned original 414-column V13 input schema')
    pair = newpairs['giga_jina_lora']
    if not isinstance(pair, (tuple, list)) or len(pair) != 2:
        raise ValueError('Complete real train/test LoRA score pair is required')
    outputs = []
    for frame, base, score in [(tr, X, pair[0]), (te, Xt, pair[1])]:
        if not {'id', 'query_id', 'article_id'} <= set(frame):
            raise ValueError('Pair/query/article metadata are required')
        if frame.id.isna().any() or not frame.id.is_unique or frame.query_id.isna().any() or frame.article_id.isna().any():
            raise ValueError('Malformed pair/query/article IDs')
        p = np.asarray(score)
        if p.shape != (len(frame),) or not np.issubdtype(p.dtype, np.number) or not np.isfinite(p).all():
            raise ValueError('Incomplete real giga_jina_lora scores')
        if not np.isfinite(base.to_numpy()).all():
            raise ValueError('Original V13 source features must be finite')
        prefix = 'giga_jina_lora'
        delta = query_z(frame, p) - query_z(frame, base['s4_raw'].to_numpy())
        result = pd.concat([base.reset_index(drop=True).copy(), _score_features(frame, p, prefix),
                            _peer_features(frame, p, prefix + '_peer'),
                            pd.DataFrame({prefix + '_vs_rr4': delta, prefix + '_vs_rr4_abs': np.abs(delta)})], axis=1).astype(np.float32)
        if result.shape[1] != 433 or result.columns.duplicated().any() or not np.isfinite(result.to_numpy()).all():
            raise ValueError('Invalid 433-column Giga/Jina LoRA feature matrix')
        if not np.array_equal(result.iloc[:, :414].to_numpy(), base.to_numpy(np.float32)):
            raise ValueError('Original 414-column V13 feature prefix changed')
        outputs.append(result)
    if list(outputs[0]) != list(outputs[1]):
        raise ValueError('Train/test fusion feature schemas differ')
    return tuple(outputs)


In [ ]:
%%writefile qwen8_focus_head_v17.py
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

MODEL_REVISION='77d193c791ed757ca307ee72715aa132723da912'
HIDDEN_SIZE=4096
HEAD_PARAMS=dict(alpha=100.,solver='lsqr',max_iter=200,tol=1e-4,fit_intercept=False)
RESIDUAL_ALPHA=.5
STD_FLOOR=1e-4

def validate_availability(tr, te, availability):
    if not isinstance(availability, (tuple, list)) or len(availability) != 2:
        raise ValueError('Explicit train/test availability masks are required')
    result = []
    for frame, mask in zip((tr, te), availability):
        mask = np.asarray(mask)
        if mask.dtype != np.dtype(bool) or mask.shape != (len(frame),):
            raise ValueError('Availability must be a full-length boolean mask')
        if not {'id', 'query_id', 'article_id'} <= set(frame):
            raise ValueError('Pair/query/article IDs are required')
        if (frame.id.isna().any() or not frame.id.is_unique or frame.query_id.isna().any()
                or frame.article_id.isna().any()):
            raise ValueError('Malformed pair/query/article IDs')
        if (pd.Series(mask).groupby(frame.query_id.reset_index(drop=True), sort=False).nunique() > 1).any():
            raise ValueError('Availability must select whole query groups, never partial queries')
        result.append(mask)
    return tuple(result)

def validate_score_pair(tr, te, pair, availability, name):
    if not isinstance(pair, (tuple, list)) or len(pair) != 2:
        raise ValueError('Complete train/test source pair required: ' + name)
    for frame, values, mask in zip((tr, te), pair, availability):
        p = np.asarray(values)
        if (p.shape != (len(frame),) or not np.issubdtype(p.dtype, np.number)
                or not np.isfinite(p).all()):
            raise ValueError('Finite full-length scores required: ' + name)
        if np.any(p[~mask] != 0):
            raise ValueError('Unavailable source rows must be exactly zero: ' + name)

def _center_hidden(frame, mask, hidden):
    centered = np.array(hidden, dtype=np.float32, copy=True, order='C')
    selected = frame.loc[mask, ['query_id']].reset_index(drop=True)
    for ix in selected.groupby('query_id', sort=False).indices.values():
        centered[ix] -= centered[ix].mean(axis=0, dtype=np.float32)
    return centered

def _fit_model(H, positions, target, weights):
    # One float32 training copy; both transforms mutate their temporary arrays.
    a = H[positions]
    scaler = StandardScaler(with_mean=False, copy=False)
    scaler.fit(a)
    scaler.scale_ = np.maximum(scaler.scale_, STD_FLOOR)
    a = scaler.transform(a, copy=False)
    ridge = Ridge(**HEAD_PARAMS)
    with threadpool_limits(limits=2):
        ridge.fit(a, target, sample_weight=weights)
    return dict(scaler=scaler, ridge=ridge)

def _predict_model(model, H, positions):
    out = np.empty(len(positions), np.float32)
    with threadpool_limits(limits=2):
        for start in range(0, len(positions), 512):
            ix = positions[start:start + 512]
            a = model['scaler'].transform(H[ix], copy=False)
            out[start:start + len(ix)] = model['ridge'].predict(a)
    if not np.isfinite(out).all():
        raise ValueError('Nonfinite residual head predictions')
    return out


In [ ]:
%%writefile qwen8_focus_meta_v17.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features
from qwen8_focus_head_v17 import validate_availability,validate_score_pair

SOURCE_ORDER=('q8_focus_frozen','q8_focus_head')
SOURCE_KINDS=dict(q8_focus_frozen='zero_shot',q8_focus_head='trained')

def _source_names(newpairs, kinds):
    if (set(newpairs) != set(kinds) or 'q8_focus_frozen' not in newpairs
            or not set(newpairs) <= set(SOURCE_ORDER)):
        raise ValueError('Frozen scores and optional complete OOF residual-head scores with explicit kinds are required')
    if any(kinds[name] != SOURCE_KINDS[name] for name in newpairs):
        raise ValueError('Incorrect Qwen8 source kind')
    return tuple(name for name in SOURCE_ORDER if name in newpairs)

def build_features_v17(tr, te, X, Xt, newpairs, kinds, availability):
    """Retain X433 and append nineteen frozen features plus availability.

    A learned head is validated for direct blending but excluded from all
    learned features. Zeroes outside coverage mean missing, never model score.
    """
    names = _source_names(newpairs, kinds)
    masks = validate_availability(tr, te, availability)
    if (len(X) != len(tr) or len(Xt) != len(te) or X.shape[1] != 433 or Xt.shape[1] != 433
            or list(X) != list(Xt) or X.columns.duplicated().any() or 's4_raw' not in X):
        raise ValueError('Expected the exact aligned original 433-column raw-score input schema')
    for name in names:
        validate_score_pair(tr, te, newpairs[name], masks, name)
    outputs = []
    name = 'q8_focus_frozen'
    for side, (frame, base, mask) in enumerate(zip((tr, te), (X, Xt), masks)):
        if not np.isfinite(base.to_numpy()).all():
            raise ValueError('Original raw-source features must be finite')
        p = np.asarray(newpairs[name][side])
        delta = query_z(frame, p) - query_z(frame, base['s4_raw'].to_numpy())
        new = pd.concat([_score_features(frame, p, name), _peer_features(frame, p, name + '_peer'),
                         pd.DataFrame({name + '_vs_rr4': delta, name + '_vs_rr4_abs': np.abs(delta)})], axis=1)
        new.loc[~mask, :] = 0.
        result = pd.concat([base.reset_index(drop=True), new,
                            pd.DataFrame({name + '_available': mask.astype(np.float32)})], axis=1).astype(np.float32)
        if result.shape[1] != 453 or result.columns.duplicated().any() or not np.isfinite(result.to_numpy()).all():
            raise ValueError('Invalid Qwen8 frozen-source feature matrix')
        if not np.array_equal(result.iloc[:, :433].to_numpy(), base.to_numpy(np.float32)):
            raise ValueError('Original 433-column raw-score feature prefix changed')
        outputs.append(result)
    if list(outputs[0]) != list(outputs[1]):
        raise ValueError('Train/test fusion feature schemas differ')
    return tuple(outputs)


In [ ]:
%%writefile v18_frozen_cpu.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features
from qwen8_focus_head_v17 import validate_availability
from qwen8_focus_meta_v17 import build_features_v17

CROSS_REFERENCES=('jina35_mean','jina35_lora','giga_lora','gemma_frozen')

def _ranks(frame, values):
    s = pd.Series(np.asarray(values), index=np.arange(len(frame)))
    q = frame.query_id.reset_index(drop=True)
    rank = s.groupby(q, sort=False).rank(method='average', ascending=False).to_numpy()
    count = s.groupby(q, sort=False).transform('size').to_numpy()
    return (rank - 1.) / np.maximum(count - 1., 1.)

def _top_ids(frame, values):
    ids = frame.id.astype(str).to_numpy()
    out = np.empty(len(frame), dtype=object)
    for ix in frame.groupby('query_id', sort=False).indices.values():
        best = ix[np.lexsort((ids[ix], -np.asarray(values)[ix]))[0]]
        out[ix] = ids[best]
    return out

def make_features(tr, te, reference, pairs, kinds, availability):
    """Pure, label-free builder preserving the exact V17 453-column prefix."""
    masks = validate_availability(tr, te, availability)
    qpair = {'q8_focus_frozen': pairs['q8_focus_frozen']}
    if kinds.get('q8_focus_frozen') != 'zero_shot':
        raise ValueError('Frozen Qwen8 logits must be declared zero_shot')
    A, B = build_features_v17(tr, te, reference['X'], reference['Xt'], qpair,
                              {'q8_focus_frozen': 'zero_shot'}, masks)
    rich = []
    for side, (frame, base, mask) in enumerate(zip((tr, te), (A, B), masks)):
        qraw = np.asarray(pairs['q8_focus_frozen'][side])
        qz, qr, qt = query_z(frame, qraw), _ranks(frame, qraw), _top_ids(frame, qraw)
        new = {}
        q = frame.query_id.reset_index(drop=True)
        for name in CROSS_REFERENCES:
            raw = np.asarray(pairs[name][side])
            if raw.shape != (len(frame),) or not np.isfinite(raw).all():
                raise ValueError('Incomplete fixed cross-reference stream: ' + name)
            rz, rr = query_z(frame, raw), _ranks(frame, raw)
            d, rd = qz - rz, qr - rr
            prefix = 'q8_cross_' + name + '_'
            new.update({prefix+'delta': d, prefix+'absdelta': np.abs(d),
                        prefix+'rank_delta': rd, prefix+'product': qz * rz,
                        prefix+'mean_absrankdelta': pd.Series(np.abs(rd)).groupby(q, sort=False).transform('mean').to_numpy(),
                        prefix+'top1_disagree': (qt != _top_ids(frame, raw)).astype(np.float32)})
        extra = pd.DataFrame(new, dtype=np.float32)
        extra.loc[~mask, :] = 0.
        result = pd.concat([base, extra], axis=1).astype(np.float32)
        if result.shape != (len(frame), 477) or result.columns.duplicated().any() or not np.isfinite(result.to_numpy()).all():
            raise ValueError('Invalid 477-column feature matrix')
        if not np.array_equal(result.iloc[:, :453].to_numpy(), base.to_numpy()):
            raise ValueError('V17 453-column prefix changed')
        rich.append(result)
    if list(rich[0]) != list(rich[1]):
        raise ValueError('Train/test cross-feature schemas differ')
    return A, B, rich[0], rich[1]

prepare_features=make_features


In [ ]:
%%writefile v18_nested_head.py
import numpy as np
import pandas as pd
from meta_frozen_v4 import _score_features,query_z
from joint_meta import _peer_features

FEATURE_COUNT=496

def append_head_features(frame, X477, scores, frozen, mask):
    if (len(X477) != len(frame) or X477.shape[1] != 477
            or X477.columns.duplicated().any() or not np.isfinite(X477.to_numpy()).all()):
        raise ValueError('Aligned finite rich 477-column frozen features are required')
    delta = query_z(frame, scores) - query_z(frame, frozen)
    new = pd.concat([_score_features(frame, scores, 'q8_nested_head'),
        _peer_features(frame, scores, 'q8_nested_head_peer'),
        pd.DataFrame(dict(q8_nested_head_vs_frozen=delta,
                          q8_nested_head_vs_frozen_abs=np.abs(delta)))], axis=1)
    if new.shape[1] != 19:
        raise ValueError('Expected exactly nineteen nested residual-head features')
    new.loc[~mask, :] = 0.
    result = pd.concat([X477.reset_index(drop=True), new], axis=1).astype(np.float32)
    if result.shape[1] != FEATURE_COUNT or result.columns.duplicated().any() or not np.isfinite(result.to_numpy()).all():
        raise ValueError('Invalid nested 496-column feature matrix')
    return result


In [ ]:
%%writefile final_utils.py
"""Utilities used by the fixed cold-start V19 pipeline."""
from pathlib import Path
import gc, hashlib, json, os
import numpy as np
import torch
from cache_core import dataset_signature

REVISIONS = {
 'BAAI/bge-reranker-v2-m3':'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e',
 'Qwen/Qwen3-Reranker-4B':'22e683669bc0f0bd69640a1354a6d0aebcfeede5',
 'Qwen/Qwen3-Embedding-0.6B':'97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3',
 'jinaai/jina-reranker-v3.5':'e8a93f33f0b22108f8c2364f8484ce3422552fbc',
 'BAAI/bge-reranker-v2-gemma':'1787044f8b6fb740a9de4557c3a12377f84d9e17',
 'ai-sage/Giga-Embeddings-instruct-3B-0826':'b71168088212f0a13688514e4fc288a86106c4e4',
 'Qwen/Qwen3-Reranker-8B':'77d193c791ed757ca307ee72715aa132723da912'}

class UnlimitedBudget:
    def remaining(self): return 1e12
    def for_neural(self): return 1e12
    def check(self): pass

def release():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def atomic_npz(path, **arrays):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    with tmp.open('wb') as f: np.savez_compressed(f,**arrays)
    os.replace(tmp,p)

class Sources:
    """New-run optional score caches, never an input requirement."""
    def __init__(self,tr,te,path,recipe):
        self.tr,self.te=tr,te
        self.root=Path(path);self.root.mkdir(parents=True,exist_ok=True)
        self.signature=hashlib.sha256((dataset_signature(tr,te)+json.dumps(
            dict(revisions=REVISIONS,recipe=recipe),sort_keys=True)).encode()).hexdigest()
        self.pairs={}
    def read(self,name):
        p=self.root/(name+'.npz')
        if not p.is_file(): return None
        with np.load(p,allow_pickle=False) as z:
            if str(z['signature'].item())!=self.signature: raise ValueError('Source cache recipe changed: '+name)
            for side,f in [('train',self.tr),('test',self.te)]:
                if not np.array_equal(z[side+'_ids'],f.id.to_numpy(dtype=str)):raise ValueError('Source IDs changed: '+name)
            pair=(z['train'].copy(),z['test'].copy())
            for a,key in zip(pair,['train','test']):
                if hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()!=str(z[key+'_sha'].item()):raise ValueError('Source cache corrupt: '+name)
        self.validate(name,pair);self.pairs[name]=pair;return pair
    def validate(self,name,pair):
        if len(pair)!=2:raise ValueError('Train/test pair required: '+name)
        for a,f in zip(pair,(self.tr,self.te)):
            if np.asarray(a).shape!=(len(f),) or not np.isfinite(a).all():raise ValueError('Incomplete source: '+name)
    def put(self,name,pair):
        self.validate(name,pair);a,b=(np.asarray(x,np.float32) for x in pair)
        atomic_npz(self.root/(name+'.npz'),signature=np.asarray(self.signature),
            train_ids=self.tr.id.to_numpy(dtype=str),test_ids=self.te.id.to_numpy(dtype=str),
            train=a,test=b,train_sha=np.asarray(hashlib.sha256(a.tobytes()).hexdigest()),
            test_sha=np.asarray(hashlib.sha256(b.tobytes()).hexdigest()))
        self.pairs[name]=(a,b)
    def stage(self,names,factory):
        if all(self.read(n) is not None for n in names):
            print('Complete cached sources:',', '.join(names),flush=True);return
        print('Computing sources:',', '.join(names),flush=True)
        result=factory()
        for name in names:
            if name not in result:raise RuntimeError('Required source was not produced: '+name)
            self.put(name,result[name])
        release()


In [ ]:
%%writefile final_data.py
"""CSV input, exact recorded split configuration and label-free features.

Embedded bytes contain only fold/availability membership, never targets,
weights, scores, hidden states or model predictions.
"""
from pathlib import Path
import base64,hashlib,json,zlib,zipfile,shutil
import numpy as np
import pandas as pd
from cache_core import load_competition,dataset_signature
from tabular_features import build_features
from final_utils import atomic_npz

SPLITS={'fold': 'eNqtnVuCIzkIBPW6/5n3a8fuWZQRorc+23Y9VAiSJKHn/PdYf479Oc6fY3yOU/718/v1+eMsj+qbn3N+f2/Up/rc4Pdfyzv8Pl/5s88XPx+Xa1H9uPp4l+vz+bhcsh8nKJfy69vVs816qcpTreIY1fH1ef3Q1ZP+deLyDuEK+ZVVC1HdUr2i5SurVmSV77EynMrw6vOcDZfXtvO9dLCwA95tbdrxteVd/d8T13dTrVb5m9vd1F85tS8plri8+cr4hj6OPOpfl09/SqvqbF3YBtV97Ormy8UW76s4fbVyYAPZJ1xiRn6M2hPU2/WHA6wetLqVu6+Z+RSLnEcdYqsrgxnGrXyLH9kDaw96KHzWTtAH3QmeIAfHOmiU7qE2xMuX6S0m5zTJGkq3kWPs9+8vLgisBB4ux8zglQkGDYhS1vGV3wOXT5G23JaTvLLeYzMHrMoKIJI3MPHPVfIba3U8BAG38vEWvNq88jnLoP1HgBzi4YErVZfMqwkJBnk+gYMImpXuB3K56r0DEqnfdY6T+yGlnPl9Zce78rEJC8dHn9nrDjLUDsTLGA2hJryrWa9sPmkMrJO/NgBJ1WkeeC0458irM8C/+dC0IAm0EfJAklAZ8qBgueIer31qRnmT0I9bgaExcTa+mReVsILCUjcTzYZf4uqcQCOenLQ/bucAOyJikXKmTEGW74iwG+yBDft7L3vYLO/7RmCr5tB4W6X4o2qJAiQAqohosYTO8vkQvcPuIqhEfGQZdA5EX01CwcXh3BAQibEGr1Zskvz6hk8BiD2j5HpoVgz5ZshrFkS81QF3mavPtIZeLuA6NWk5PL95yTAAeRG4jlf3TB0VEbZdd7JdjWg8M5yTtalzBeaDMyqEVwC+B+ygvEuwskzDDCJvL5GGQnTD3oH90Ggxm6HN9TTKB/JiwkaCzYnB/Zk6/14KuHW/5L62PDTEvSGbbck9qMdCRcWXvzTyG57upWWglCM7udqSMg9BfhHXq5FBDi192MSCmAXMWyg71xMzTvBoeWWOZ9JhczaigqoKQzyHtGHe+ENdpbugbVt7RSAQ721Rhm3LcUaW4ioZFz+qH25DpJ/wQoenH8ABFC+TICU4hW2ZM6IHIQPOTqy+DJalIH9rMGpACRyNpoESRp6YRBFXYGglMAB6gYPZ4CN8qZ3kA9PH/uJvJxf/4qo1IPYNFYANEOLTF2Utllef5HQBiplUmy4dT3bzQJ/7wOJrdyvEJOC7PEcx6QD/oGGurgxrBF9z2BmpARupxW1Q0ssL4G2fqj95TaG++kDDEb1GNA09JhJ5Wg1CrLvVMkORzHuGbDKUVEv904t4DJR3U4OqbDTkN3R6njcDCRy8WgEdXcY2lqPVlKfVCBOcJmUawEuAZ5bz16fRaRoJJBpVBFtgB+xEjKHOdTrZxCIHTizS0ZTO0ATJfujsecAUhyhpcnuE7rNzyQADNiOEc0t/DUiHNzQtTF97os8bONX7RYt+CHRQUf6d/9VNGrCoxypFD7BUwKpBhVrnJ5n5J8AmFuyCL0j9ajMtSmvJj+aim2WWhwXz2T+BDhhLHBCUG6LJPd7BXMa4xakfvJMv5zxXAgj3sz7NZiW2erB1Snut8h0dbbS1oss9tkgMiSHKQRaBx8w0alU92Eh2oe/thFQ+2Nv2wpIelJApUepeMGWbEPJmytFIcxa6UqRbYqi0rW2NVZ3UGKorThRidUvsXlbel20IdsDfdwFpEUQz2lrF2kG4r18NttySZ6tt48F7QUe1rrlRHqqbgrSmnhkE4EvFs3XmAehGaB0TscvXD9oYy6qsz9TZK6nXoeVBl/wopXowiEWtB1TER+Lcc/C+Q5QOKyFG5EYdVhZiNNvPDm5Bo1Tx+odGk2yroeuhXX/b1pJncjbaiI57uqIPCOyhsQ38EeX6eu9At8I7bjq2NKX79fx0Eb0SvgT6/005AR71odJkDULXhwZNtIDMPm9Q7/3ZZQBN7HPa3QA7W3oTrUawyZduXtDUbcZtvvpM3T22ZuVnAfh2SNBGeL2i1nd6UnoQhe9rF9lgdG/wLSuj8phV69iGItINEBvFPQS+jXE/pB05l7ATsEpYr2kQdmTeVnMqrU0hf8/PpPBtJ07JQPIjW7oi6aVmcLUk4Giqd9lXM3Rj+QKKxY+YBA5SNzdqZtFq0KCnBNWjVAhoje8i/gRVCpThdFrpqSMbfm5zZa/0mJqH8PGi0xX5fRvUK5G968N0uxWd10JtTL04jZEyxF8+F0/v4zNAcufPH53eQyNYtYr62roHzgZV5Lv0+9Gi+IdhSi8zJ6Vq5niUpolt60h0syFND9WqCt027qeo2ALQsXDXT0QzVuy5mtv9aWUlDsbwE2h1lvH9kn0rpIcxMi/zfDv1rWAByzYmtTo/dYGFRLjLztzEnivLofixUJrphLza9+fCyAjapJU521HithirA/ZDgr+srd4FaR3FzXynEzJv6ynE7ExIsotDsxJIfKhsaZHqsnc0YBdbxQDt3UXN3X62ErRE+0YsX5+zox1mkOL7DFLT3eDKhWMfmvpdT7RO4261KE3LzyK92FDxbsDhlN7YWAJwVcfqRk8nD/cERnnqPvYW43QzXJ2zIcGs/e6wQwcR1iDAb0z789OuYxjRY3geqlZ+crRWd+l5bJZkp4RJVKME5e0VUJr/hwcsrgiooIPZdJ1Oh12PDL06F2ocAMu07NPjyVwNJ6k4ZEy/aVpa/oCGNjvpG8CArkP9ekYh/Vea1nhwLUYhoGaV4h2Chb/7MJrUAzkr40VSFJQguoHQz/X2jUi3Zjrs5LUdRkyteqHfiYn11vzng4rpacxZbCImEUGjl4iZPlu/brQ1N9TFmdTXUwk6PoRmaRDWUxF+6yMHhg5d/qBybeT3D92Ql0ydEoJfjXBm2RrdlC/kg1RT0256cKAnjK6l/lZ/AhVg7Zjf4m/PP3jR++kegFZf1n6YzQ0hkFy6n6Fk5wRZRiG/C6r9sV12pt9rdeVDcUGSC8fy0Xp3tzRMvqeO0DqLkfQcEw1wvA3jrodMSGv1sOs7RxKGVzgreSyt2v18/A8Bn/ZT', 'source2': 'eNq9nMmW3DAMA+3//2jllkxe3GahwI6PmbSthQsIQrrvf5/rz3N+Pz/+/ucfz4//+viCGz9PX/rw0uuaPnB+PvjDdNjT58/7K+fPTFN9nNu4KFf1TPYBfmjM4v3XT593i/M+uUN34XE9zpmWafItvLBqQ4fFERYSL7AxA+L9T7+blrhb1wOf4IuDHd6Bv4pZDoOf3I0PzlgDt9enaUyh6fl/fDA76xzTnvBMZnayDMBDBFaeGDgJT7omUU9zd9nzhnO6unQZpZIgqnWpBpkLXeGfv+BhhjoO3+gAo5aLNuS4Iaya7XaA4PAXvA9vL/gF8XsqSmDCGsskvHE4ZWHEWmK5b9ULPEnAQm6clBnGhw/Q7cIDMqEeJ+nAyTjEK8HEzYHwaULyw1+zOHKGkGzw1WTnpU2v7dcQdwZjCUY5eCEepgCG3aqKkkiEJgO9392HRmy+x58Kmck/LOTtLBvv2xSuA8jFB0/nNtEN73+dZ57jyZz4VURjhGExEAksuC2yOJnVcTamLCrZpv9UcfKUYeqCa/AyFWqO2v/JkptCsOLhDQiSzNpXSURM/UyRzCAWvmd4xDzlDB/nRW8AoyqqBbMeZUmDAdWnMIF3OAjbotQUiIziRoGbuekI7iDoqb5PZ+rg4Gw19oKupYCwWv1hxImt7BtAQBWcBmubaSj5wBoIulQUDEFd3qunoQjnUEGPniFtC93KqALIkFLvsZiumsshmmUwwq1FAFQgs4gEg/4GRk6cXiiZqu7lHVN1BAPVtXMDbYbBSIeW2ibMTPFKtZVseTNkZC7wwUqcoB2QsK87qMa0jUxQGmzANNwlwcZr97yYkPxRjPfub+QIliixL5QyKqcM6zhSKvPr1DwGTAvKFO9JqbrZBHiM+nflsppmwAkZIl9oJw2Pt9Sqx6nTCRrXaBfzRlrWld1Vk5cNf10lkhODgiCCd9hX1PeGUG4EK1PFINgsURuW2rFLVBNfoQeV/KLtlfWi1KDd1ak0Og6vaZEpeSMvUwPFhdCIjBksz9tlKw6voWAERK/z8N1rj4kVJTzmSQOc+cFkEk1YJcy9LxFnl+yo62V2tS+vxo9Y8xzM5XrdMTqtd4qxP151fCqaIlHuSZixk8/o4nnjbIGBKzCwvH9rOvOViGZPHWaaW0q40fmgYCQFYVjE22A0BmCJvkdSjF1TKdwpn2uGwmtUXqClUskLnvnLYnxe3MxwNVHvxiW9aVusSSHKjHtrO6A5hPfYFZ/Ki61WXFU2Dbo6VqkiEisMAIu73AQre9vc/z6nJNYICdSnHWkPQg65AvXpOdHamaVrjFRcFYzaIp80snxRNroiR6nUqlRFXVicAin3QSx/gO0Uz/zqJK20gqtKCt1GI5/JFC0JaT/KzXhNu4TRxG0ZDQfPgWl+4G2tzcrxYQVkS8itzuIY6kmFt866DZkm9TS5K5S9pQAYqfSWnGbAsILegHXuJhlKlxJ6NcFntfxeGlIM5fPaDFFUstgmbuCGKjBQnJOfeE8C2LokK31HYjz0TTdxHbNpY0wKSnQObju0Hdg+Xh1BhgZOEOSLJU2EO8DcakuGeMfTnrr/VQRZ2+nDjamlUkbJ77c42qoXv6b/5ZvGidxpXQxlVQpE82tuuQV36tIqpk8rLSgO0Y0mQaUDjtmdB4pgN6A9kDLSMc0N07igbnnI4KWL13EyZMgJJA6Pee/LEHZluysvq4xk7BiTOSti1PeUayLGFWQhi51NDqBE0PuPTVPU1ASCClPlDG//d1aRs47uGiR+AULu5RkL22B/ljNVR4wCqU9ReZFRi14griHsrCouHmvpCYQFQgOH4ep6ywVYnokcCmlGvV4VxQyZMpiuIjG/0YzlPhxfNLXXS7XMfszfogWYZeiRfjhSoR3+5AvxHoMNn8FhihCLYW84ZucwpuO8ZY4ThkkEUaICVdXZXZ5flqLweb+KmHd0IzbDxD3eFqnUYByNGtJnLrMcr1NFf4M7gyYTXcWaxvsw5IsbaeBAJal4lhBbEKRLXUzgaqaMNtqAk2MVcZl9xYDxk9nvpvnXv1W3obAMb1jSgSwS5ul0vzxEOwE/18461NmGluowz1oJnxfmuwI7czJzq0jaUv6ZFsCiSm+P7Tmi2KdrJ2B/5+B/GRbuq24d8RGrYZyii//lJYurXOq+6NAQVo7oFIAxONDhCpCIZhum/wuPTbwK', 'train_mask': 'eNrdnEt25UAIQ/H+N92jnB70SwzSFVVpD+Nnuz4ghKBS9XU9n656uZ6Xp7T7f3/3zVCeySVN4aeHnzNXfwBvY22/Sdrp0RIVtDtbKzJahQ+v/PznusjMWhvTWznFYkdDdOfz3HZJ1pE17sSI+77TvW2i53fOaoYfYuX7DljtZR1g5ADbhPiBBZ2neAjnpvRuPf/e1giXggX1HquU5ZGdeM4INPYq2MHviCIi8nQxY0Rt+t4g8VvF8iuI2nVLsObf03Bjjilj3uWBWNYNJYBP+n2Fo0sYMQUAex1RecygAsSijkahKLD0/cR7xptPIOZg5Hg6XE+ACSkOE79yASBKpIrLVfgt6Y+ycCbdov8S+dMj0GQZTKsuAZ6qrJWxIqZmvIyga7pQGNbsd7pBiZJ/GDk4qjrHP7qmBwV2iiIlh0M/k2Bkd4W0RnOvt3fAykP5FUxoB/NXulAFFgUnUy+FLs+TKYV1ceHdNDUzGkU+mQUHNsIylSJBNjRtoZwrKQBIA5T06ZPakZQf0cnITKTSup68im4mo4DyMoirJSo8ETteShliHURFc8CHGx9Z8p8H9LGAitkkyFpsg7+jbPDGdsH1FgjCD+MI8De7F4XSETgjYz5ocbZ9YpNdK0nh0ELPoMJCxXLqM55hJ8vzCVGTo0oCy7N0Da2II1QGt3fEQzI3jRrb6n6JmK9+mZQxqqPYiXxCba5NtHErp0FmSk2X67xy8XOp9SOLKQygm+IeVR7HhCdrBYb33z61JZzy7zSOQUkqeIKh4WckoEGaASxa0J48nTxkkKo6Y90rW6XYbKy6LOXTMqSzRNwKr3OOFWQCadU6smFK3cUFnB3TA/S1QAvlPZ4VSabPzlKi2A0HoHBzTWGI5H9uXu7ZC9blaJ9nDlmvAsSgdBgIKFLQBpk+4n9YQ0cWJ8b8h2m8oNZkgLWYR8+byALbvd6qzonlB2DkZEHhwjQokp+uXw5XgNR45dAXpmVYfdVr6aeJY/01SvSTLXIjGpdiDXw7KvBjnoZY2wEl28Obn6iUE1fIqNTGo46J5l6M1ReGldAzUICK6HpLxfoLeB1f3Pl9AuJn81uSJCdhZ+KFB1ZMoUu3GAvPm++wY0/By2qmE3uH0jUexJbsTnJ3hpm1Hjvtx2Dt30oDyObgnQwJJDNJoJNQwUxeoekmpHmTHpjaai5Xgyqkr/+V80IppW8yWDelhYSRpcIO2kDR/2yqF+6p0ZUds51daNDJtDwA2po3EmEdk4LE1tNUKnIAoMM4gB40i+aBGRksyFZPaHUlUJuVZtzfJVTWGBIDoEb152R1hsB3lhoot3PkqJ16M5SYDnkyMfCQ1+buAWnqcAjfZrUv16coFdeSr/32LOG7r0Q40PWx/+qCK9DQqbQ2cryK3BwpICzQUsoOm2LaUbTGxECjLdWO839SfSOoKFupvOiwIV8iV1t8zQQIzjWoYgaPGKZgwBWm+5TOlfL73xT2IN3RgVUTF1DjDxtZU2o=', 'test_mask': 'eNrVWkmChEAIw/9/ek5z04JsaNfR7tpYQohW/Y/rPOo4xpOv0CB2aI4fPzNuR9Hg5brS3Tp1fjg0AROD03M5DUHkgBrMNR2J3QcbtNPufiaghs7yh59xA1caDVwR6T4usphteyKBHy7PwKctFvoZBLj5chpemrLQyxV3dnHFt7WUFLw9qwVEKRak8BGduVSzmUJoRnSyHDM1GjhZW9ivrbwGrGgLGsXttsIktW4PyxDonu0vAFeUanqj3+FmaEJv3KSAI0GduwDvvtBuWSQEcHbZoiHEjC5bYjOo0UK4ajNCP5hvHgS/n5HPfEVyjjQInuzrlF1uISslwX6djQ1i3ycSYIVLMVlr0T1gXywhUFzb60pss2wyEPRrpz+GhBGmfVS8HfUO0T/7ok7sVBUNgeKlZQ9bHWq3KI24KGel1PuJxZccY/0IOBNnDLKpDZJfrTOnsvUTPU0GM8YPKcQhD2oqG5ndqboyZ95J2pRRU0TqI9WnzBcMhNyqdT4S2616hb2bxPaXivmCdMpoGlSfkajHkmoRJvuut+tb7k/r9q3/2cOKqLoJVlrwVbgGHh0kM5Ys/oOAtSYvafTd9V6YvRpeBngWmLyLIqyU7HdREowyl48Pnxe+oqpnTif+M63uUUTtXIQIwuf5Ktdm/zJm7jYwiFGU+F41nIyILSMlJS4irHQEt5P+AF6GI8c='}
ID_HASHES={'train': 'f78c4a46a8b5f639da7097623c791b494b354101d79f68fac555e66ee8a422dc', 'test': '505331587536f79be0ab02b6108411ac391af5d1cfad8b337840e641486f548b'}

def _array(name,n):
    a=np.frombuffer(zlib.decompress(base64.b64decode(SPLITS[name])),np.uint8).copy()
    if a.shape!=(n,):raise ValueError('Split configuration length differs')
    return a

def check_ids(tr,te):
    for name,f in [('train',tr),('test',te)]:
        if hashlib.sha256('\n'.join(f.id.astype(str)).encode()).hexdigest()!=ID_HASHES[name]:
            raise ValueError('This fixed solution expects the original competition IDs/order: '+name)

def extract_data(path,destination):
    path=Path(path);dest=Path(destination);dest.mkdir(parents=True,exist_ok=True)
    names=['train.csv','test.csv','sample_submission.csv']
    if path.is_dir():
        if not all((path/n).is_file() for n in names):raise FileNotFoundError('CSV directory must contain train.csv, test.csv, sample_submission.csv')
        return path
    if not path.is_file() or not zipfile.is_zipfile(path):raise FileNotFoundError('Set INPUT_PATH to the original competition ZIP or CSV directory')
    with zipfile.ZipFile(path) as z:
        for n in names:
            members=[m for m in z.namelist() if Path(m).name==n and not m.endswith('/')]
            if len(members)!=1:raise ValueError('ZIP must contain exactly one '+n)
            with z.open(members[0]) as a,(dest/n).open('wb') as b:shutil.copyfileobj(a,b)
    return dest

def load_final_data(path,work):
    work=Path(work);work.mkdir(parents=True,exist_ok=True)
    directory=extract_data(path,work/'data')
    tr,te,_=load_competition(directory);check_ids(tr,te)
    assignment=_array('fold',len(tr))
    if not np.array_equal(np.unique(assignment),np.arange(5)):raise ValueError('Invalid saved partition')
    folds=[(np.flatnonzero(assignment!=k),np.flatnonzero(assignment==k)) for k in range(5)]
    for a,b in folds:
        for col in ['query_id','document_id','grp']:
            if set(tr.iloc[a][col])&set(tr.iloc[b][col]):raise ValueError('Saved split crosses '+col)
    sample=pd.read_csv(directory/'sample_submission.csv',dtype={'id':str})
    if len(sample)!=len(te) or not sample.id.is_unique or set(sample.id)!=set(te.id.astype(str)):raise ValueError('Invalid sample IDs')
    signature=dataset_signature(tr,te)
    manifest=work/'input_manifest.json'
    if manifest.is_file() and json.loads(manifest.read_text())['signature']!=signature:raise ValueError('Changed data: use a new WORK_DIR')
    manifest.write_text(json.dumps({'signature':signature,'train_rows':len(tr),'test_rows':len(te)}))
    cache=work/'tabular_features.npz'
    full=pd.concat([tr,te],ignore_index=True)
    if cache.is_file():
        with np.load(cache,allow_pickle=False) as z:
            if str(z['signature'].item())!=signature or not np.array_equal(z['ids'],full.id.to_numpy(str)):raise ValueError('Tabular cache differs')
            X=pd.DataFrame(z['values'],columns=z['columns'].astype(str))
    else:
        X=build_features(full)
        atomic_npz(cache,signature=np.asarray(signature),ids=full.id.to_numpy(str),values=X.to_numpy(),columns=np.asarray(list(X),dtype=str))
    if X.shape!=(len(full),106) or not np.isfinite(X.to_numpy()).all():raise ValueError('Expected finite106-column features')
    xbase=(X.iloc[:len(tr)].reset_index(drop=True),X.iloc[len(tr):].reset_index(drop=True))
    masks=(_array('train_mask',len(tr)).astype(bool),_array('test_mask',len(te)).astype(bool))
    for f,m in zip((tr,te),masks):
        if pd.Series(m).groupby(f.query_id).nunique().gt(1).any():raise ValueError('Partial Qwen8 mask splits a query')
    return directory,tr,te,folds,xbase,masks,sample

def source_folds(tr,dev):
    """The fixed two-fold DEV split used by the neural source models."""
    assignment=_array('source2',len(tr))
    if not np.array_equal(np.asarray(dev),np.flatnonzero(assignment<2)):raise ValueError('Unexpected source training partition')
    result=[]
    for k in range(2):
        a=dev[assignment[dev]!=k];b=dev[assignment[dev]==k]
        for col in ['query_id','document_id','grp']:
            if set(tr.iloc[a][col])&set(tr.iloc[b][col]):raise ValueError('Source split crosses '+col)
        result.append((a,b))
    return result


In [ ]:
%%writefile final_legacy_sources.py
"""Original supervised BGE and historical EOS embedding, built from CSVs."""
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer,AutoModel
import legacy_original as old
from final_utils import REVISIONS,release

def legacy_frames(data_dir,tr,te):
    f=old.load(str(data_dir));f['grp']=old.make_groups(f)
    a=f.loc[f.is_train.eq(1)].sort_values(['query_id','id']).reset_index(drop=True)
    b=f.loc[f.is_train.eq(0)].sort_values(['query_id','id']).reset_index(drop=True)
    for x,y in [(a,tr),(b,te)]:
        if not np.array_equal(x.id.astype(str),y.id.astype(str)):raise ValueError('Legacy ID order differs')
    return a,b

def train_bge(name,tr,te,folds,root,device='cuda'):
    cfg={**old.CFG[name],'revision':REVISIONS['BAAI/bge-reranker-v2-m3']}
    old.dev=device
    out=Path(root)/name;out.mkdir(parents=True,exist_ok=True)
    tok=AutoTokenizer.from_pretrained(cfg['name'],revision=cfg['revision'])
    a=np.zeros(len(tr),np.float32);b=np.zeros(len(te),np.float32)
    for k,(aa,bb) in enumerate(folds):
        p,t=old.train_fold(cfg,str(out),k,tr.iloc[aa].reset_index(drop=True),tr.iloc[bb].reset_index(drop=True),te,tok)
        if np.shape(p)!=(len(bb),) or np.shape(t)!=(len(te),):raise ValueError('Incomplete BGE fold')
        a[bb]=p;b+=t/len(folds)
    release();return a,b

@torch.inference_mode()
def embed_old(texts,tok,model,maxlen,device,bs=64):
    ids=tok(texts,add_special_tokens=False,truncation=True,max_length=maxlen-1)['input_ids']
    ids=[s+[tok.eos_token_id] for s in ids]
    order=np.argsort([len(s) for s in ids],kind='stable');out=[None]*len(ids)
    i=0
    while i<len(order):
        ix=order[i:i+bs];length=max(len(ids[j]) for j in ix)
        x=torch.full((len(ix),length),tok.eos_token_id,dtype=torch.long)
        m=torch.zeros_like(x)
        for r,j in enumerate(ix):x[r,:len(ids[j])]=torch.tensor(ids[j]);m[r,:len(ids[j])]=1
        try:
            with torch.autocast(device_type=torch.device(device).type,dtype=torch.float16,enabled=str(device).startswith('cuda')):
                h=model(input_ids=x.to(device),attention_mask=m.to(device)).last_hidden_state
            e=torch.nn.functional.normalize(h[torch.arange(len(ix),device=device),m.sum(1).to(device)-1].float(),dim=-1).cpu().numpy()
        except torch.cuda.OutOfMemoryError:
            if bs==1:raise
            bs=max(1,bs//2);release();continue
        for r,j in enumerate(ix):out[j]=e[r]
        i+=len(ix)
        if i%1024<bs:print('Legacy Qwen embeddings:',i,'/',len(ids),flush=True)
    return np.stack(out)

def old_embedding_pair(tr,te,device='cuda'):
    name='Qwen/Qwen3-Embedding-0.6B'
    tok=AutoTokenizer.from_pretrained(name,revision=REVISIONS[name])
    model=AutoModel.from_pretrained(name,revision=REVISIONS[name],torch_dtype=torch.float32).to(device).eval()
    task='Given a web search query, retrieve relevant passages that answer the query'
    result=[]
    for f in (tr,te):
        q=f.drop_duplicates('query_id');d=f.drop_duplicates('document_id')
        eq=embed_old([f'Instruct: {task}\nQuery:{x}' for x in q.query_c],tok,model,96,device)
        ed=embed_old([(t+'. '+x if t else x) for t,x in zip(d.title_c,d.text_c)],tok,model,320,device)
        qi={x:i for i,x in enumerate(q.query_id)};di={x:i for i,x in enumerate(d.document_id)}
        result.append(np.asarray([float(eq[qi[a]]@ed[di[b]]) for a,b in zip(f.query_id,f.document_id)],np.float32))
    del model;release();return tuple(result)


In [ ]:
%%writefile final_neural_sources.py
"""Generate only the nineteen scalar source streams used by the final V19."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
from final_utils import Sources,UnlimitedBudget,REVISIONS,release
from final_legacy_sources import legacy_frames,train_bge,old_embedding_pair

REQUIRED=('bge','bge2','bge3','q3emb_legacy','rr4','bge_pretrained','qwen_embed_correct',
          'jina35_id','jina35_reverse','jina35_mean','jina35_linear','jina35_adapter',
          'jina35_lora','gemma_frozen','giga_cosine','giga_linear','giga_rank',
          'giga_lora','giga_jina_lora')
TRAINED={'bge','bge2','bge3','jina35_linear','jina35_adapter','jina35_lora',
         'giga_linear','giga_rank','giga_lora','giga_jina_lora'}

def build_neural_sources(tr,te,folds,data_dir,outdir,device='cuda'):
    root=Path(outdir);root.mkdir(parents=True,exist_ok=True)
    cache=Sources(tr,te,root/'sources',recipe='fixed-v19-cold-v1')
    if all(cache.read(n) is not None for n in REQUIRED):
        return {n:cache.pairs[n] for n in REQUIRED},{n:'trained' if n in TRAINED else 'zero_shot' for n in REQUIRED}
    budget=UnlimitedBudget();full=pd.concat([tr,te],ignore_index=True)
    legacy_tr,legacy_te=legacy_frames(data_dir,tr,te)
    for name in ['bge','bge2','bge3']:
        cache.stage([name],lambda name=name:{name:train_bge(name,legacy_tr,legacy_te,folds,root/'legacy',device)})
    cache.stage(['q3emb_legacy'],lambda:{'q3emb_legacy':old_embedding_pair(legacy_tr,legacy_te,device)})

    # Revisions are supplied explicitly before the first model is downloaded.
    neural=root/'frozen_old';neural.mkdir(exist_ok=True)
    revpath=neural/'model_revisions.json'
    if revpath.is_file() and json.loads(revpath.read_text())!=REVISIONS:raise ValueError('Frozen revisions changed')
    revpath.write_text(json.dumps(REVISIONS,indent=2))
    from neural_features import qwen_reranker,bge_zeroshot,qwen_embeddings
    for name,fn in [('rr4',lambda:qwen_reranker(full,neural,budget,max_length=768)),
                    ('bge_pretrained',lambda:bge_zeroshot(full,neural,budget,max_length=512)),
                    ('qwen_embed_correct',lambda:qwen_embeddings(full,neural,budget,max_length=768))]:
        def generate(name=name,fn=fn):
            p=fn()
            if p is None:raise RuntimeError('Required frozen source unfinished: '+name)
            return {name:(p[:len(tr)],p[len(tr):])}
        cache.stage([name],generate)

    jina_names=['jina35_id','jina35_reverse','jina35_mean','jina35_linear','jina35_adapter']
    def jina_sources():
        from jina_joint import run_jina_joint
        from jina_probe import fit_jina_probe
        from jina_rank_adapter import fit_rank_adapter
        scores,v=run_jina_joint(full,root/'jina_frozen',budget,orders=('id','reverse'),return_embeddings=True,device=device)
        if set(scores)!={'jina35_id','jina35_reverse'} or not {'query','document'}<=set(v):raise RuntimeError('Incomplete Jina frozen states')
        out={n:(p[:len(tr)],p[len(tr):]) for n,p in scores.items()}
        out['jina35_mean']=tuple((.5*(out['jina35_id'][s]+out['jina35_reverse'][s])).astype(np.float32) for s in (0,1))
        q,d=v['query'],v['document'];args=(tr,te,folds,q[:len(tr)],d[:len(tr)],q[len(tr):],d[len(tr):])
        a,b,_=fit_jina_probe(*args,root/'jina_linear');out['jina35_linear']=(a,b)
        a,b,_=fit_rank_adapter(*args,root/'jina_adapter',max_seconds=1e10,device='cpu');out['jina35_adapter']=(a,b)
        return out
    cache.stage(jina_names,jina_sources)
    dev=np.setdiff1d(np.arange(len(tr)),folds[-1][1])
    def jina_lora():
        from jina_lora_v7 import fit_jina_lora
        pairs,report=fit_jina_lora(tr,te,{'id':cache.pairs['jina35_id'],'reverse':cache.pairs['jina35_reverse']},
            root/'jina_lora',budget,train_rows=dev,device=device,second_order=True)
        if 'jina35_lora' not in pairs:raise RuntimeError('Jina LoRA unfinished: '+str(report.get('status')))
        return pairs
    cache.stage(['jina35_lora'],jina_lora)
    def gemma():
        from gemma_v8 import run_frozen_gemma
        pairs,report=run_frozen_gemma(tr,te,root/'gemma',budget,device=device)
        if 'gemma_frozen' not in pairs:raise RuntimeError('Gemma frozen unfinished: '+str(report.get('status')))
        return pairs
    cache.stage(['gemma_frozen'],gemma)

    def giga_sources():
        from giga_encoder_v11 import encode_giga
        from giga_heads_v11 import fit_giga_heads_train,predict_giga_heads_test
        a,ar=encode_giga(tr,root/'giga_encoder/train',budget,phase='train',device=device)
        if not {'query','document'}<=set(a):raise RuntimeError('Incomplete Giga train vectors: '+str(ar.get('status')))
        b,br=encode_giga(te,root/'giga_encoder/test',budget,phase='test',device=device)
        if not {'query','document'}<=set(b):raise RuntimeError('Incomplete Giga test vectors: '+str(br.get('status')))
        scores,report=fit_giga_heads_train(tr,folds,a['query'],a['document'],root/'giga_heads',budget)
        if not {'giga_cosine','giga_linear','giga_rank'}<=set(scores):raise RuntimeError('Incomplete Giga supervised heads')
        test=predict_giga_heads_test(te,b['query'],b['document'],root/'giga_heads',report)
        # The test helper returns (scores, diagnostics).
        if isinstance(test,tuple):test=test[0]
        return {n:(scores[n],test[n]) for n in ['giga_cosine','giga_linear','giga_rank']}
    cache.stage(['giga_cosine','giga_linear','giga_rank'],giga_sources)
    def giga_lora():
        from giga_lora_v13 import run_giga_lora
        pairs,report=run_giga_lora(tr,te,folds,root/'giga_lora',budget,
            {'giga_cosine':cache.pairs['giga_cosine'][0]},device=device)
        if 'giga_lora' not in pairs:raise RuntimeError('Giga LoRA unfinished: '+str(report.get('status')))
        return pairs
    cache.stage(['giga_lora'],giga_lora)
    def giga_jina():
        from giga_lora_v15 import run_giga_jina_lora
        pairs,report=run_giga_jina_lora(tr,te,folds,root/'giga_jina_lora',budget,
            {'giga_cosine':cache.pairs['giga_cosine'][0],'jina35_mean':cache.pairs['jina35_mean'][0]},device=device)
        if 'giga_jina_lora' not in pairs:raise RuntimeError('Giga/Jina LoRA unfinished: '+str(report.get('status')))
        return pairs
    cache.stage(['giga_jina_lora'],giga_jina)
    release()
    return {n:cache.pairs[n] for n in REQUIRED},{n:'trained' if n in TRAINED else 'zero_shot' for n in REQUIRED}


In [ ]:
%%writefile final_fusion.py
"""Fixed V19 feature graph and final all-training fits; no model search."""
from pathlib import Path
import hashlib,json
import numpy as np
import pandas as pd
import lightgbm as lgb
from meta_frozen_v4 import _score_features,query_z
from refine_cpu import make_features
from joint_meta import _peer_features
from v8_fusion import build_features_v8,PARAMS
from giga_meta_v11 import build_source_features
from giga_fusion_v13 import build_features_v13
from giga_fusion_v15 import build_features_v15
from v18_frozen_cpu import prepare_features
from v18_nested_head import append_head_features
import qwen8_focus_head_v17 as head
from final_utils import atomic_npz

OLD=['bge','bge2','bge3','q3emb_legacy','rr4','bge_pretrained','qwen_embed_correct']
JINA=['jina35_id','jina35_reverse','jina35_mean','jina35_linear','jina35_adapter']

def build_raw_features(tr,te,xbase,pairs,kinds):
    X,Xt,_,_=make_features(tr,te,xbase,pairs,OLD)
    af,bf=[X],[Xt]
    for j,name in enumerate(JINA):
        for f,side,dest in [(tr,0,af),(te,1,bf)]:
            p=pairs[name][side];dest.append(_score_features(f,p,f'new{j}'))
            delta=query_z(f,p)-query_z(f,pairs['rr4'][side])
            dest.append(pd.DataFrame({f'new{j}_vs_rr4':delta,f'new{j}_vs_rr4_abs':np.abs(delta)}))
    for name in ['rr4']+JINA:
        af.append(_peer_features(tr,pairs[name][0],'peer_'+name))
        bf.append(_peer_features(te,pairs[name][1],'peer_'+name))
    A300,B300=(pd.concat(v,axis=1).astype(np.float32) for v in [af,bf])
    chosen=['jina35_lora','gemma_frozen']
    A338,B338=build_features_v8(tr,te,A300,B300,{n:pairs[n] for n in chosen},{n:kinds[n] for n in chosen})
    chosen=['giga_cosine','giga_linear','giga_rank'];kk={n:kinds[n] for n in chosen}
    A395=build_source_features(tr,A338,{n:pairs[n][0] for n in chosen},kk)
    B395=build_source_features(te,B338,{n:pairs[n][1] for n in chosen},kk)
    A414,B414=build_features_v13(tr,te,A395,B395,{'giga_lora':pairs['giga_lora']},{'giga_lora':'trained'})
    A433,B433=build_features_v15(tr,te,A414,B414,{'giga_jina_lora':pairs['giga_jina_lora']},{'giga_jina_lora':'trained'})
    return {300:(A300,B300),338:(A338,B338),395:(A395,B395),414:(A414,B414),433:(A433,B433)}

def fit_predict(tr,te,A,B,objective,leaves,rounds,path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    spec=dict(objective=objective,num_leaves=leaves,rounds=rounds)
    h=hashlib.sha256(json.dumps(dict(params=PARAMS,spec=spec,columns=list(A)),sort_keys=True).encode())
    for a in [A.to_numpy(np.float32),B.to_numpy(np.float32),tr.relevance.to_numpy(np.float32)]:h.update(np.ascontiguousarray(a).tobytes())
    key=h.hexdigest();metadata=path.with_suffix('.json')
    if path.is_file():
        if not metadata.is_file() or json.loads(metadata.read_text())['signature']!=key:raise ValueError('Final model inputs changed: '+str(path))
        record=json.loads(metadata.read_text())
        if record['model_sha']!=hashlib.sha256(path.read_bytes()).hexdigest():raise ValueError('Model cache corrupted')
        model=lgb.Booster(model_file=str(path))
    else:
        params={**PARAMS,'objective':objective,'num_leaves':leaves}
        rank=objective=='lambdarank'
        if rank:params.update(metric='ndcg',ndcg_eval_at=[10],lambdarank_truncation_level=12)
        else:params['metric']='binary_logloss'
        group=tr.groupby('query_id',sort=False).size().to_numpy() if rank else None
        model=lgb.train(params,lgb.Dataset(A,label=tr.relevance.to_numpy(),group=group),num_boost_round=rounds)
        model.save_model(str(path))
        metadata.write_text(json.dumps(dict(signature=key,model_sha=hashlib.sha256(path.read_bytes()).hexdigest(),spec=spec)))
    p=np.asarray(model.predict(B,raw_score=True,num_threads=2),np.float32)
    if p.shape!=(len(te),) or not np.isfinite(p).all():raise ValueError('Invalid fixed final prediction')
    return query_z(te,p)

def fit_v13_reference(tr,te,features,outdir):
    root=Path(outdir)
    A,B=features[338]
    r=fit_predict(tr,te,A,B,'lambdarank',7,260,root/'v8_rank7.txt')
    b=fit_predict(tr,te,A,B,'binary',15,300,root/'v8_bce15.txt')
    v8=query_z(te,.75*r+.25*b)
    A,B=features[395]
    b=fit_predict(tr,te,A,B,'binary',15,300,root/'v11_bce15.txt')
    v11=(.5*v8+.5*b).astype(np.float32)
    A,B=features[414]
    r=fit_predict(tr,te,A,B,'lambdarank',7,260,root/'v13_rank7.txt')
    b=fit_predict(tr,te,A,B,'binary',15,300,root/'v13_bce15.txt')
    return (.75*v11+.25*query_z(te,.75*r+.25*b)).astype(np.float32)

def ridge_features(tr,te,folds,bundle,masks,outdir):
    """Five-fold source OOF on all labeled train; full-train source for test."""
    root=Path(outdir);root.mkdir(parents=True,exist_ok=True)
    h=hashlib.sha256()
    for k in ['train_hidden','test_hidden','train_frozen','test_frozen']:
        a=np.ascontiguousarray(bundle[k]);h.update(str(a.shape).encode());h.update(a.tobytes())
    for a in [*masks,tr.relevance.to_numpy(np.float32)]:h.update(np.ascontiguousarray(a).tobytes())
    for a,b in folds:h.update(a.tobytes());h.update(b.tobytes())
    key=h.hexdigest();path=root/'ridge_scores.npz'
    if path.is_file():
        with np.load(path,allow_pickle=False) as z:
            if str(z['signature'].item())!=key:raise ValueError('Ridge cache changed')
            p,pt=z['train'].copy(),z['test'].copy()
        if not np.isfinite(p).all() or not np.isfinite(pt).all():raise ValueError('Incomplete Ridge cache')
        return p,pt
    H=head._center_hidden(tr,masks[0],bundle['train_hidden'])
    Ht=head._center_hidden(te,masks[1],bundle['test_hidden'])
    lookup=np.full(len(tr),-1,np.int64);lookup[masks[0]]=np.arange(masks[0].sum())
    base=query_z(tr,bundle['train_frozen']);baset=query_z(te,bundle['test_frozen'])
    weights=(1./tr.groupby('query_id',sort=False).query_id.transform('size').to_numpy()).astype(np.float32)
    p=np.zeros(len(tr),np.float32)
    def fit(rows):
        rows=np.sort(np.asarray(rows,np.int64));rows=rows[masks[0][rows]]
        target=query_z(tr.iloc[rows],tr.iloc[rows].relevance.to_numpy())-base[rows]
        return head._fit_model(H,lookup[rows],target,weights[rows])
    for k,(a,b) in enumerate(folds):
        b=b[masks[0][b]];model=fit(a)
        if len(b):p[b]=base[b]+head.RESIDUAL_ALPHA*head._predict_model(model,H,lookup[b])
        print('Final Ridge OOF',k+1,'/5',flush=True)
    model=fit(np.arange(len(tr)));pt=np.zeros(len(te),np.float32)
    positions=np.flatnonzero(masks[1])
    pt[positions]=baset[positions]+head.RESIDUAL_ALPHA*head._predict_model(model,Ht,np.arange(len(Ht)))
    atomic_npz(path,signature=np.asarray(key),train=p,test=pt)
    return p,pt

def subset_bundle(bundle,masks):
    result={}
    for side,m in zip(('train','test'),masks):
        previous=np.asarray(bundle[side+'_mask'],bool)
        if np.any(m&~previous):raise ValueError('Missing required Qwen8 rows')
        lookup=np.full(len(m),-1,np.int64);lookup[previous]=np.arange(previous.sum())
        result[side+'_hidden']=bundle[side+'_hidden'][lookup[np.flatnonzero(m)]]
        result[side+'_frozen']=np.where(m,bundle[side+'_frozen'],0.).astype(np.float32)
    return result

def final_fusion(tr,te,folds,xbase,pairs,kinds,bundle,partial_masks,outdir):
    root=Path(outdir);root.mkdir(parents=True,exist_ok=True)
    features=build_raw_features(tr,te,xbase,pairs,kinds)
    v13=fit_v13_reference(tr,te,features,root/'models')
    dev=np.setdiff1d(np.arange(len(tr)),folds[-1][1]);audit=np.sort(folds[-1][1])
    A433,B433=features[433]
    reference=dict(X=A433,Xt=B433,dev=dev,audit=audit,
                   dev_pred=np.zeros(len(dev)),audit_pred=np.zeros(len(audit)),test_pred=v13)
    outputs=[]
    for name,masks in [('partial',partial_masks),('full',(np.ones(len(tr),bool),np.ones(len(te),bool)))]:
        b=subset_bundle(bundle,masks)
        newpairs={**pairs,'q8_focus_frozen':(b['train_frozen'],b['test_frozen'])}
        newkinds={**kinds,'q8_focus_frozen':'zero_shot'}
        _,_,A477,B477=prepare_features(tr,te,reference,newpairs,newkinds,masks)
        p,pt=ridge_features(tr,te,folds,b,masks,root/name)
        A=append_head_features(tr,A477,p,b['train_frozen'],masks[0])
        B=append_head_features(te,B477,pt,b['test_frozen'],masks[1])
        z=fit_predict(tr,te,A,B,'lambdarank',7,260,root/'models'/(name+'_nested_rank7.txt'))
        if name=='partial':z=np.where(masks[1],z,v13).astype(np.float32)
        outputs.append(z)
        del A,B,A477,B477,b
    result=(.5*outputs[0]+.5*outputs[1]).astype(np.float32)
    atomic_npz(root/'final_scores.npz',test_ids=te.id.to_numpy(str),prediction=result,
               v13=v13,v18=outputs[0],v19_full_nested=outputs[1])
    return result,dict(version='V19',fixed_new_weight=.5,source_count=20,
                       final_feature_count=496,final_train_rows=len(tr),no_hyperparameter_search=True)


In [ ]:
# Не смешиваем кэши разных версий кода/библиотек.
import importlib
for p in RUNTIME_DIR.glob('*.py'):
    sys.modules.pop(p.stem,None)
importlib.invalidate_caches()
implementation=hashlib.sha256()
for p in sorted(RUNTIME_DIR.glob('*.py')):
    implementation.update(p.name.encode());implementation.update(p.read_bytes())
implementation.update(REQUIREMENTS_TEXT.encode())
recipe=implementation.hexdigest();plan=WORK_DIR/'recipe.json'
if plan.is_file() and json.loads(plan.read_text())['recipe']!=recipe:raise ValueError('Код изменился: выбери новый WORK_DIR')
plan.write_text(json.dumps({'recipe':recipe,'system':platform.platform(),'torch':torch.__version__}))
if not INPUT_PATH.exists():
    candidates=[WORK_DIR/'data',RUNTIME_DIR/'amml-ranking-competition-2026.zip',PROJECT_DIR/'data',Path('/content/amml-ranking-competition-2026.zip'),Path('/content/drive/MyDrive/amml-ranking-competition-2026.zip')]
    def ready_input(p):
        return p.is_file() or p.is_dir() and all((p/n).is_file() for n in ('train.csv','test.csv','sample_submission.csv'))
    INPUT_PATH=next((p for p in candidates if ready_input(p)),INPUT_PATH)
if not INPUT_PATH.exists():
    try:from google.colab import files
    except ImportError:raise FileNotFoundError('Укажи INPUT_PATH: исходный ZIP или каталог с train.csv/test.csv/sample_submission.csv')
    uploaded=files.upload()
    INPUT_PATH=next((RUNTIME_DIR/f for f in uploaded if f.lower().endswith('.zip')),INPUT_PATH)
from final_data import load_final_data
data_dir,tr,te,folds,xbase,partial_masks,sample=load_final_data(INPUT_PATH,WORK_DIR)
print('Данные:',len(tr),len(te),'| запросы:',tr.query_id.nunique(),te.query_id.nunique())
print('Фолды:',[len(b) for _,b in folds])


## Дообученные и замороженные источники

Обучаются три BGE cross-encoder, линейные/rank-головы и выбранные LoRA-адаптеры. Замороженные энкодеры получают только тексты. Разбиение удерживает связанные запросы и документы в одном фолде. Завершённые этапы сохраняются в новом рабочем каталоге и могут быть повторно использованы только при совпадении входов.

Перебор гиперпараметров и эксперименты, не входящие в конечную формулу, здесь не запускаются.


In [ ]:
from final_neural_sources import build_neural_sources
pairs,kinds=build_neural_sources(tr,te,folds,data_dir,WORK_DIR/'neural',device='cuda')
print('Все необходимые базовые источники завершены:',list(pairs))


## Полные 8B-представления

Qwen3-Reranker-8B заморожен, загружается в NF4 с FP16-вычислениями. Рассчитываются все кандидаты всех запросов

In [ ]:
from qwen8_focus_v17 import run_focus_qwen8
from final_utils import UnlimitedBudget
q8_bundle,q8_report=run_focus_qwen8(tr,te,pairs,kinds,WORK_DIR/'qwen8',UnlimitedBudget(),device='cuda')
if q8_bundle is None or not q8_bundle['train_mask'].all() or not q8_bundle['test_mask'].all():
    raise RuntimeError('Полный Qwen8-источник ещё не завершён; продолжи с тем же WORK_DIR')
print('Полное покрытие 8B проверено.')


## Фиксированный финальный ансамбль и отправка

Признаки BM25/TF-IDF, метаданные и оценки моделей поступают в зафиксированные LambdaRank/BCE-ранкеры

In [ ]:
from final_fusion import final_fusion
prediction,report=final_fusion(tr,te,folds,xbase,pairs,kinds,q8_bundle,partial_masks,WORK_DIR/'fusion')
lookup=pd.Series(np.asarray(prediction,np.float32),index=te.id.astype(str))
submission=pd.DataFrame({'id':sample.id,'prediction':sample.id.map(lookup)})
assert list(submission)==['id','prediction'] and len(submission)==13507
assert submission.id.is_unique and np.isfinite(submission.prediction).all()
OUTPUT_PATH.parent.mkdir(parents=True,exist_ok=True)
submission.to_csv(OUTPUT_PATH,index=False)
report.update(output=str(OUTPUT_PATH),sha256=hashlib.sha256(OUTPUT_PATH.read_bytes()).hexdigest(),
              system=platform.platform(),python=platform.python_version(),torch=torch.__version__)
(WORK_DIR/'final_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print('Готово:',OUTPUT_PATH,'| строк:',len(submission),'| SHA256:',report['sha256'])


как то так